<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/04_Statistical_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
# ==================================================================================================
# NOTEBOOK 04 — STATISTICAL BASELINES
# SPP-GAN: A Privacy-Preserving Statistical–Machine Learning Framework for
# High-Fidelity Synthetic Tabular Data Generation
# ==================================================================================================
#
# PURPOSE
# -------
# Establish reproducible statistical baseline generators using the authoritative
# native TRAINING data persisted by Notebook 02.
#
# BASELINES
# ---------
# 1. Independent Marginal Sampling
# 2. Gaussian Copula
#
# DATA POLICY
# -----------
# - TRAIN split only is used for fitting.
# - VALIDATION and TEST are never used for fitting.
# - Notebook 02 preprocessing is NOT refitted.
# - Raw data are NOT reloaded.
#
# SCHEMA POLICY
# -------------
# Native generative schema:
#     preprocessing features + target
#
# Statistical model input:
#     generative columns
#
# Target:
#     generated as part of the synthetic generative table
#     NEVER used as a predictor
#
# Identifier / provenance:
#     excluded from synthetic generation
#
# RAM POLICY
# ----------
# - One dataset at a time.
# - One baseline at a time.
# - No dataset concatenation.
# - Persist artifacts immediately.
# - Explicit garbage collection.
#
# ==================================================================================================

print("=" * 100)
print("NOTEBOOK 04 — STATISTICAL BASELINES")
print("=" * 100)

NOTEBOOK_ID = "04"
NOTEBOOK_NAME = "Statistical Baselines"
NOTEBOOK_VERSION = "2.0"

print(f"Notebook       : {NOTEBOOK_ID} — {NOTEBOOK_NAME}")
print(f"Version        : {NOTEBOOK_VERSION}")
print("Purpose        : Reproducible statistical synthetic-data baselines")
print("Data policy    : TRAIN ONLY")
print("Schema policy  : Notebook 02 native generative schema")
print("RAM policy     : One dataset / one baseline at a time")

NOTEBOOK 04 — STATISTICAL BASELINES
Notebook       : 04 — Statistical Baselines
Version        : 2.0
Purpose        : Reproducible statistical synthetic-data baselines
Data policy    : TRAIN ONLY
Schema policy  : Notebook 02 native generative schema
RAM policy     : One dataset / one baseline at a time


In [69]:
# ==================================================================================================
# NOTEBOOK 04 — STATISTICAL BASELINES
# SECTION 02 — LOAD CONFIGURATION
# ==================================================================================================
#
# PURPOSE
# -------
# Load and validate the authoritative configuration persisted by Notebook 00.
#
# IMPORTANT RESEARCH POLICY
# -------------------------
# • Notebook 00 is the authoritative configuration source.
# • Notebook 00 artifacts are JSON, not YAML.
# • Notebook 00 is FROZEN and must not be modified by Notebook 04.
# • Notebook 04 must not silently create or replace configuration values.
# • Statistical baseline definitions are specified separately in Section 05.
# • Only TRAIN data may be used for fitting statistical baselines.
# • Validation and TEST data remain reserved for downstream evaluation.
#
# NOTEBOOK 00 MANIFEST POLICY
# ---------------------------
# • Authoritative manifest version is 1.0.
# • Project identity is stored under:
#       project.project_name
#       project.project_version
#       project.notebook_id
#       project.notebook_name
# • Notebook 00 does NOT contain a top-level "integrity" object.
#
# EXPERIMENTAL DESIGN
# -------------------
# • Master seed             : 2025
# • Repetitions             : 5
# • Repetition seed offset  : 1000
# • Repetition seeds        : 3026–3030
# • Primary experimental unit:
#       dataset × repetition × method
#
# ==================================================================================================

print("=" * 100)
print("SECTION 02 — LOAD CONFIGURATION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

from pathlib import Path
import json
import hashlib
import gc


# --------------------------------------------------------------------------------------------------
# 2. Verify Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive"
)

MYDRIVE_ROOT = (
    DRIVE_ROOT
    / "MyDrive"
)


if not DRIVE_ROOT.exists():

    raise RuntimeError(
        "Google Drive is not mounted.\n"
        "Mount Google Drive before executing Notebook 04."
    )


if not MYDRIVE_ROOT.exists():

    raise RuntimeError(
        f"MyDrive directory not found:\n"
        f"{MYDRIVE_ROOT}"
    )


print(
    f"✓ Google Drive verified : {DRIVE_ROOT}"
)

print(
    f"✓ MyDrive verified      : {MYDRIVE_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 3. Define Canonical Project Root
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    MYDRIVE_ROOT
    / "SPP_GAN_Research"
)


if not PROJECT_ROOT.exists():

    raise RuntimeError(
        f"Canonical project root does not exist:\n"
        f"{PROJECT_ROOT}"
    )


if not PROJECT_ROOT.is_dir():

    raise RuntimeError(
        f"Canonical project root is not a directory:\n"
        f"{PROJECT_ROOT}"
    )


print(
    f"✓ Project root verified : {PROJECT_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 4. Define Canonical Notebook 00 Artifact Root
# --------------------------------------------------------------------------------------------------

NB00_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_00"
)


NB00_CONFIG_ROOT = (
    NB00_ROOT
    / "config"
)

NB00_ENV_ROOT = (
    NB00_ROOT
    / "environment"
)

NB00_MANIFEST_ROOT = (
    NB00_ROOT
    / "manifest"
)


print()
print("-" * 100)
print("NOTEBOOK 00 ARTIFACT ROOTS")
print("-" * 100)

print(
    f"✓ Notebook 00 root       : {NB00_ROOT}"
)

print(
    f"✓ Configuration root     : {NB00_CONFIG_ROOT}"
)

print(
    f"✓ Environment root       : {NB00_ENV_ROOT}"
)

print(
    f"✓ Manifest root          : {NB00_MANIFEST_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 5. Verify Notebook 00 Directory Structure
# --------------------------------------------------------------------------------------------------

REQUIRED_NB00_DIRECTORIES = {

    "Notebook 00 root":
        NB00_ROOT,

    "Configuration directory":
        NB00_CONFIG_ROOT,

    "Environment directory":
        NB00_ENV_ROOT,

    "Manifest directory":
        NB00_MANIFEST_ROOT,
}


for label, path in REQUIRED_NB00_DIRECTORIES.items():

    if not path.exists():

        raise RuntimeError(
            f"{label} is missing:\n"
            f"{path}"
        )


    if not path.is_dir():

        raise RuntimeError(
            f"{label} exists but is not a directory:\n"
            f"{path}"
        )


    print(
        f"✓ {label:<30}: {path}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Define Required Notebook 00 JSON Artifacts
# --------------------------------------------------------------------------------------------------

NB00_ARTIFACT_PATHS = {

    # Configuration artifacts
    "experiment_config":
        NB00_CONFIG_ROOT
        / "experiment_config.json",

    "dataset_registry":
        NB00_CONFIG_ROOT
        / "dataset_registry.json",

    "model_registry":
        NB00_CONFIG_ROOT
        / "model_registry.json",

    "evaluation_config":
        NB00_CONFIG_ROOT
        / "evaluation_config.json",

    "privacy_config":
        NB00_CONFIG_ROOT
        / "privacy_config.json",

    "sppgan_config":
        NB00_CONFIG_ROOT
        / "sppgan_config.json",

    # Environment artifact
    "environment":
        NB00_ENV_ROOT
        / "environment.json",

    # Manifest artifacts
    "manifest":
        NB00_MANIFEST_ROOT
        / "notebook_00_manifest.json",

    "configuration_fingerprint":
        NB00_MANIFEST_ROOT
        / "configuration_fingerprint.json",
}


# --------------------------------------------------------------------------------------------------
# 7. Verify Required Artifact Files
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 00 ARTIFACT DISCOVERY")
print("-" * 100)


for artifact_name, artifact_path in NB00_ARTIFACT_PATHS.items():

    if not artifact_path.exists():

        raise FileNotFoundError(
            f"Required Notebook 00 artifact is missing:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {artifact_path}"
        )


    if not artifact_path.is_file():

        raise RuntimeError(
            f"Notebook 00 artifact exists but is not a file:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {artifact_path}"
        )


    file_size = artifact_path.stat().st_size


    if file_size <= 0:

        raise RuntimeError(
            f"Notebook 00 artifact is empty:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {artifact_path}"
        )


    print(
        f"✓ {artifact_name:<30} : "
        f"{artifact_path} "
        f"({file_size:,} bytes)"
    )


# --------------------------------------------------------------------------------------------------
# 8. JSON Loading Helper
# --------------------------------------------------------------------------------------------------

def load_json_artifact(
    path: Path,
    artifact_name: str,
):

    try:

        with path.open(
            "r",
            encoding="utf-8",
        ) as file:

            data = json.load(
                file
            )


    except json.JSONDecodeError as exc:

        raise RuntimeError(
            f"Invalid JSON in Notebook 00 artifact:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {path}\n"
            f"  Error    : {exc}"
        ) from exc


    except Exception as exc:

        raise RuntimeError(
            f"Unable to read Notebook 00 artifact:\n"
            f"  Artifact : {artifact_name}\n"
            f"  Path     : {path}\n"
            f"  Error    : {exc}"
        ) from exc


    return data


# --------------------------------------------------------------------------------------------------
# 9. Load All Notebook 00 Artifacts
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("LOADING NOTEBOOK 00 JSON ARTIFACTS")
print("-" * 100)


EXPERIMENT_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["experiment_config"],
    "experiment_config",
)


DATASET_REGISTRY = load_json_artifact(
    NB00_ARTIFACT_PATHS["dataset_registry"],
    "dataset_registry",
)


MODEL_REGISTRY = load_json_artifact(
    NB00_ARTIFACT_PATHS["model_registry"],
    "model_registry",
)


EVALUATION_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["evaluation_config"],
    "evaluation_config",
)


PRIVACY_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["privacy_config"],
    "privacy_config",
)


SPPGAN_CONFIG = load_json_artifact(
    NB00_ARTIFACT_PATHS["sppgan_config"],
    "sppgan_config",
)


ENVIRONMENT = load_json_artifact(
    NB00_ARTIFACT_PATHS["environment"],
    "environment",
)


NOTEBOOK_00_MANIFEST = load_json_artifact(
    NB00_ARTIFACT_PATHS["manifest"],
    "manifest",
)


CONFIGURATION_FINGERPRINT = load_json_artifact(
    NB00_ARTIFACT_PATHS["configuration_fingerprint"],
    "configuration_fingerprint",
)


LOADED_ARTIFACTS = {

    "experiment_config":
        EXPERIMENT_CONFIG,

    "dataset_registry":
        DATASET_REGISTRY,

    "model_registry":
        MODEL_REGISTRY,

    "evaluation_config":
        EVALUATION_CONFIG,

    "privacy_config":
        PRIVACY_CONFIG,

    "sppgan_config":
        SPPGAN_CONFIG,

    "environment":
        ENVIRONMENT,

    "manifest":
        NOTEBOOK_00_MANIFEST,

    "configuration_fingerprint":
        CONFIGURATION_FINGERPRINT,
}


for artifact_name in LOADED_ARTIFACTS:

    print(
        f"✓ Loaded : {artifact_name}"
    )


# --------------------------------------------------------------------------------------------------
# 10. Validate Experiment Configuration Structure
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("VALIDATING EXPERIMENT CONFIGURATION STRUCTURE")
print("-" * 100)


if not isinstance(
    EXPERIMENT_CONFIG,
    dict,
):

    raise RuntimeError(
        "Notebook 00 experiment_config.json "
        "must contain a dictionary/object."
    )


REQUIRED_EXPERIMENT_KEYS = {

    "advanced_analysis",
    "advanced_analysis_registry",
    "comparison_model_registry",
    "comparison_models",
    "data_split",
    "datasets",
    "evaluation",
    "evaluation_registry",
    "experimental_unit",
    "repetitions",
    "seed_policy",
}


MISSING_EXPERIMENT_KEYS = (
    REQUIRED_EXPERIMENT_KEYS
    -
    set(EXPERIMENT_CONFIG.keys())
)


if MISSING_EXPERIMENT_KEYS:

    raise RuntimeError(
        "Notebook 00 experiment configuration "
        "is missing required keys:\n"
        f"{sorted(MISSING_EXPERIMENT_KEYS)}"
    )


print(
    "✓ Experiment configuration structure validated"
)


# --------------------------------------------------------------------------------------------------
# 11. Extract Authoritative Seed Policy
# --------------------------------------------------------------------------------------------------

SEED_POLICY = (
    EXPERIMENT_CONFIG.get(
        "seed_policy"
    )
)


if not isinstance(
    SEED_POLICY,
    dict,
):

    raise RuntimeError(
        "Notebook 00 experiment_config.json contains "
        "an invalid 'seed_policy' structure."
    )


MASTER_SEED = (
    SEED_POLICY.get(
        "master_seed"
    )
)


if MASTER_SEED is None:

    raise RuntimeError(
        "Notebook 00 experiment configuration does not contain "
        "the authoritative master seed at "
        "'seed_policy.master_seed'."
    )


MASTER_SEED = int(
    MASTER_SEED
)


DETERMINISTIC = bool(
    SEED_POLICY.get(
        "deterministic",
        True,
    )
)


REPETITIONS = int(
    SEED_POLICY.get(
        "repetitions",
        EXPERIMENT_CONFIG.get(
            "repetitions",
            5,
        ),
    )
)


REPETITION_SEED_OFFSET = int(
    SEED_POLICY.get(
        "repetition_seed_offset",
        1000,
    )
)


REPETITION_SEEDS = (
    SEED_POLICY.get(
        "repetition_seeds",
        {},
    )
)


if not isinstance(
    REPETITION_SEEDS,
    dict,
):

    raise RuntimeError(
        "Notebook 00 'seed_policy.repetition_seeds' "
        "must be a dictionary."
    )


print(
    "✓ Authoritative seed policy loaded"
)

print(
    f"  Master seed            : {MASTER_SEED}"
)

print(
    f"  Deterministic          : {DETERMINISTIC}"
)

print(
    f"  Repetitions            : {REPETITIONS}"
)

print(
    f"  Repetition seed offset : {REPETITION_SEED_OFFSET}"
)

print(
    f"  Repetition seeds       : {REPETITION_SEEDS}"
)


# --------------------------------------------------------------------------------------------------
# 12. Validate Repetition Seed Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_REPETITION_KEYS = {
    str(i)
    for i in range(
        1,
        REPETITIONS + 1,
    )
}


ACTUAL_REPETITION_KEYS = {
    str(key)
    for key in REPETITION_SEEDS.keys()
}


if ACTUAL_REPETITION_KEYS != EXPECTED_REPETITION_KEYS:

    raise RuntimeError(
        "Notebook 00 repetition seed registry is inconsistent.\n"
        f"Expected : {sorted(EXPECTED_REPETITION_KEYS)}\n"
        f"Found    : {sorted(ACTUAL_REPETITION_KEYS)}"
    )


for repetition in range(
    1,
    REPETITIONS + 1,
):

    repetition_key = str(
        repetition
    )


    repetition_seed = int(
        REPETITION_SEEDS[
            repetition_key
        ]
    )


    expected_seed = (
        MASTER_SEED
        +
        REPETITION_SEED_OFFSET
        +
        repetition
    )


    if repetition_seed != expected_seed:

        raise RuntimeError(
            "Notebook 00 repetition seed does not match "
            "the authoritative seed policy.\n"
            f"  Repetition : {repetition}\n"
            f"  Expected   : {expected_seed}\n"
            f"  Found      : {repetition_seed}"
        )


print(
    f"✓ Repetition seed registry validated "
    f"({REPETITIONS} repetitions)"
)


# --------------------------------------------------------------------------------------------------
# 13. Validate Dataset Registry
# --------------------------------------------------------------------------------------------------

if not isinstance(
    DATASET_REGISTRY,
    dict,
):

    raise RuntimeError(
        "Notebook 00 dataset_registry.json "
        "must contain a dictionary/object."
    )


EXPECTED_DATASETS = {

    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted",
}


def extract_dataset_entries(
    registry,
):

    """
    Resolve the dataset registry into a dictionary keyed by dataset ID.

    Notebook 00 is authoritative.
    This helper only resolves the persisted structure
    and does not create or modify configuration.
    """

    if all(
        dataset_id in registry
        for dataset_id in EXPECTED_DATASETS
    ):

        return registry


    for key in (
        "datasets",
        "registry",
        "dataset_registry",
    ):

        candidate = registry.get(
            key
        )


        if isinstance(
            candidate,
            dict,
        ):

            if all(
                dataset_id in candidate
                for dataset_id in EXPECTED_DATASETS
            ):

                return candidate


    raise RuntimeError(
        "Unable to resolve the authoritative dataset registry "
        "structure from Notebook 00."
    )


DATASET_ENTRIES = extract_dataset_entries(
    DATASET_REGISTRY
)


for dataset_id, expected_target in (
    EXPECTED_DATASETS.items()
):

    if dataset_id not in DATASET_ENTRIES:

        raise RuntimeError(
            f"Required dataset '{dataset_id}' is missing "
            "from Notebook 00 dataset registry."
        )


    dataset_entry = (
        DATASET_ENTRIES[
            dataset_id
        ]
    )


    if not isinstance(
        dataset_entry,
        dict,
    ):

        raise RuntimeError(
            f"Dataset registry entry for '{dataset_id}' "
            "must be a dictionary."
        )


    target_candidates = [

        dataset_entry.get(
            "target"
        ),

        dataset_entry.get(
            "target_column"
        ),

        dataset_entry.get(
            "label"
        ),
    ]


    target_value = next(
        (
            value
            for value in target_candidates
            if value is not None
        ),
        None,
    )


    if target_value != expected_target:

        raise RuntimeError(
            f"Dataset target mismatch for '{dataset_id}'.\n"
            f"Expected : {expected_target}\n"
            f"Found    : {target_value}"
        )


print(
    "✓ Dataset registry validated"
)

print(
    f"  Registered datasets : "
    f"{sorted(EXPECTED_DATASETS.keys())}"
)

print(
    f"  Target mapping      : "
    f"{EXPECTED_DATASETS}"
)


# --------------------------------------------------------------------------------------------------
# 14. Validate Experiment Data-Split Policy
# --------------------------------------------------------------------------------------------------

DATA_SPLIT_CONFIG = (
    EXPERIMENT_CONFIG.get(
        "data_split",
        {},
    )
)


if not isinstance(
    DATA_SPLIT_CONFIG,
    dict,
):

    raise RuntimeError(
        "Notebook 00 'data_split' must be a dictionary."
    )


EXPECTED_FIT_SPLIT = "train"


if DATA_SPLIT_CONFIG.get(
    "fit_split"
) != EXPECTED_FIT_SPLIT:

    raise RuntimeError(
        "Notebook 04 requires TRAIN-only fitting.\n"
        f"Expected fit_split : {EXPECTED_FIT_SPLIT}\n"
        f"Found              : "
        f"{DATA_SPLIT_CONFIG.get('fit_split')}"
    )


EXPECTED_SPLITS = {
    "train",
    "validation",
    "test",
}


if not DATA_SPLIT_CONFIG.get(
    "enabled",
    False,
):

    raise RuntimeError(
        "Notebook 00 data splitting is disabled. "
        "Notebook 04 requires the authoritative "
        "train/validation/test split."
    )


print(
    "✓ Data-split policy validated"
)

print(
    f"  Fit split       : "
    f"{DATA_SPLIT_CONFIG.get('fit_split')}"
)

print(
    f"  Train fraction  : "
    f"{DATA_SPLIT_CONFIG.get('train_fraction')}"
)

print(
    f"  Validation      : "
    f"{DATA_SPLIT_CONFIG.get('validation_fraction')}"
)

print(
    f"  Test fraction   : "
    f"{DATA_SPLIT_CONFIG.get('test_fraction')}"
)

print(
    f"  Required splits : "
    f"{sorted(EXPECTED_SPLITS)}"
)


# --------------------------------------------------------------------------------------------------
# 15. Validate Comparison Model Registry
# --------------------------------------------------------------------------------------------------

if not isinstance(
    MODEL_REGISTRY,
    dict,
):

    raise RuntimeError(
        "Notebook 00 model_registry.json "
        "must contain a dictionary/object."
    )


EXPECTED_COMPARISON_MODELS = {

    "statistical",
    "tvae",
    "ctgan",
    "dp_ctgan",
    "spp_gan",
}


MODEL_REGISTRY_KEYS = set(
    MODEL_REGISTRY.keys()
)


MODEL_REGISTRY_CANDIDATE = (
    MODEL_REGISTRY
)


if not EXPECTED_COMPARISON_MODELS.issubset(
    MODEL_REGISTRY_KEYS
):

    for key in (
        "models",
        "comparison_models",
        "comparison_model_registry",
        "registry",
    ):

        candidate = MODEL_REGISTRY.get(
            key
        )


        if isinstance(
            candidate,
            dict,
        ):

            if EXPECTED_COMPARISON_MODELS.issubset(
                set(candidate.keys())
            ):

                MODEL_REGISTRY_CANDIDATE = (
                    candidate
                )

                break


MODEL_REGISTRY_ENTRIES = (
    MODEL_REGISTRY_CANDIDATE
)


MISSING_MODELS = (
    EXPECTED_COMPARISON_MODELS
    -
    set(
        MODEL_REGISTRY_ENTRIES.keys()
    )
)


if MISSING_MODELS:

    raise RuntimeError(
        "Notebook 00 model registry is missing expected "
        f"comparison models: {sorted(MISSING_MODELS)}"
    )


print(
    "✓ Comparison model registry validated"
)

print(
    f"  Registered comparison models : "
    f"{sorted(EXPECTED_COMPARISON_MODELS)}"
)


# --------------------------------------------------------------------------------------------------
# 16. Validate Notebook 04 Baseline Identifiers
# --------------------------------------------------------------------------------------------------
#
# These identifiers define the exact statistical baselines used by Notebook 04.
# Their mathematical definitions belong to Section 05.
# --------------------------------------------------------------------------------------------------

STATISTICAL_BASELINE_IDS = [

    "independent_marginal",

    "gaussian_copula",
]


if len(
    STATISTICAL_BASELINE_IDS
) != len(
    set(
        STATISTICAL_BASELINE_IDS
    )
):

    raise RuntimeError(
        "Notebook 04 statistical baseline identifiers "
        "contain duplicates."
    )


print(
    "✓ Statistical baseline identifiers validated"
)

print(
    f"  Baselines : "
    f"{STATISTICAL_BASELINE_IDS}"
)


# --------------------------------------------------------------------------------------------------
# 17. Validate Experimental Unit
# --------------------------------------------------------------------------------------------------

EXPERIMENTAL_UNIT = (
    EXPERIMENT_CONFIG.get(
        "experimental_unit",
        {},
    )
)


if not isinstance(
    EXPERIMENTAL_UNIT,
    dict,
):

    raise RuntimeError(
        "Notebook 00 'experimental_unit' "
        "must be a dictionary."
    )


PRIMARY_EXPERIMENTAL_UNIT = (
    EXPERIMENTAL_UNIT.get(
        "primary_unit"
    )
)


EXPECTED_PRIMARY_UNIT = (
    "dataset × repetition × method"
)


if PRIMARY_EXPERIMENTAL_UNIT != (
    EXPECTED_PRIMARY_UNIT
):

    raise RuntimeError(
        "Unexpected primary experimental unit.\n"
        f"Expected : {EXPECTED_PRIMARY_UNIT}\n"
        f"Found    : {PRIMARY_EXPERIMENTAL_UNIT}"
    )


print(
    "✓ Experimental unit validated"
)

print(
    f"  Primary unit : "
    f"{PRIMARY_EXPERIMENTAL_UNIT}"
)


# --------------------------------------------------------------------------------------------------
# 18. Validate Notebook 00 Manifest Identity
# --------------------------------------------------------------------------------------------------

if not isinstance(
    NOTEBOOK_00_MANIFEST,
    dict,
):

    raise RuntimeError(
        "Notebook 00 manifest must contain "
        "a dictionary/object."
    )


# ----------------------------------------------------------------------------------------------
# 18.1 Manifest Version
# ----------------------------------------------------------------------------------------------

MANIFEST_VERSION = (
    NOTEBOOK_00_MANIFEST.get(
        "manifest_version"
    )
)


EXPECTED_MANIFEST_VERSION = "1.0"


if MANIFEST_VERSION != (
    EXPECTED_MANIFEST_VERSION
):

    raise RuntimeError(
        "Unexpected Notebook 00 manifest version.\n"
        f"Expected : {EXPECTED_MANIFEST_VERSION}\n"
        f"Found    : {MANIFEST_VERSION}"
    )


# ----------------------------------------------------------------------------------------------
# 18.2 Project Identity
# ----------------------------------------------------------------------------------------------

PROJECT_INFO = (
    NOTEBOOK_00_MANIFEST.get(
        "project",
        {},
    )
)


if not isinstance(
    PROJECT_INFO,
    dict,
):

    raise RuntimeError(
        "Notebook 00 manifest 'project' "
        "must be a dictionary/object."
    )


EXPECTED_PROJECT_NAME = (
    "SPP-GAN Research Project"
)


EXPECTED_PROJECT_VERSION = "1.0"


EXPECTED_NOTEBOOK_ID = "00"


EXPECTED_NOTEBOOK_NAME = (
    "Environment, Configuration & Reproducibility"
)


MANIFEST_PROJECT_NAME = (
    PROJECT_INFO.get(
        "project_name"
    )
)


MANIFEST_PROJECT_VERSION = (
    PROJECT_INFO.get(
        "project_version"
    )
)


MANIFEST_NOTEBOOK_ID = (
    PROJECT_INFO.get(
        "notebook_id"
    )
)


MANIFEST_NOTEBOOK_NAME = (
    PROJECT_INFO.get(
        "notebook_name"
    )
)


if MANIFEST_PROJECT_NAME != (
    EXPECTED_PROJECT_NAME
):

    raise RuntimeError(
        "Notebook 00 manifest project name mismatch.\n"
        f"Expected : {EXPECTED_PROJECT_NAME}\n"
        f"Found    : {MANIFEST_PROJECT_NAME}"
    )


if MANIFEST_PROJECT_VERSION != (
    EXPECTED_PROJECT_VERSION
):

    raise RuntimeError(
        "Notebook 00 manifest project version mismatch.\n"
        f"Expected : {EXPECTED_PROJECT_VERSION}\n"
        f"Found    : {MANIFEST_PROJECT_VERSION}"
    )


if str(
    MANIFEST_NOTEBOOK_ID
) != EXPECTED_NOTEBOOK_ID:

    raise RuntimeError(
        "Notebook 00 manifest notebook ID mismatch.\n"
        f"Expected : {EXPECTED_NOTEBOOK_ID}\n"
        f"Found    : {MANIFEST_NOTEBOOK_ID}"
    )


if MANIFEST_NOTEBOOK_NAME != (
    EXPECTED_NOTEBOOK_NAME
):

    raise RuntimeError(
        "Notebook 00 manifest notebook name mismatch.\n"
        f"Expected : {EXPECTED_NOTEBOOK_NAME}\n"
        f"Found    : {MANIFEST_NOTEBOOK_NAME}"
    )


# ----------------------------------------------------------------------------------------------
# 18.3 Confirm No Obsolete Integrity Dependency
# ----------------------------------------------------------------------------------------------

if "integrity" in NOTEBOOK_00_MANIFEST:

    raise RuntimeError(
        "Notebook 00 manifest unexpectedly contains a top-level "
        "'integrity' object. Notebook 04 expects the frozen "
        "Notebook 00 manifest structure without this obsolete field."
    )


print(
    "✓ Notebook 00 manifest identity validated"
)

print(
    f"  Project  : "
    f"{MANIFEST_PROJECT_NAME} "
    f"v{MANIFEST_PROJECT_VERSION}"
)

print(
    f"  Notebook : "
    f"{MANIFEST_NOTEBOOK_ID} — "
    f"{MANIFEST_NOTEBOOK_NAME}"
)

print(
    f"  Manifest : "
    f"v{MANIFEST_VERSION}"
)

print(
    "  Integrity dependency : NOT USED"
)


# --------------------------------------------------------------------------------------------------
# 19. Validate Configuration Fingerprint Artifact
# --------------------------------------------------------------------------------------------------

if not isinstance(
    CONFIGURATION_FINGERPRINT,
    dict,
):

    raise RuntimeError(
        "Notebook 00 configuration_fingerprint.json "
        "must contain a dictionary/object."
    )


EXPECTED_CONFIG_HASH = (
    CONFIGURATION_FINGERPRINT.get(
        "hash"
    )
)


FINGERPRINT_ALGORITHM = (
    CONFIGURATION_FINGERPRINT.get(
        "algorithm"
    )
)


FINGERPRINT_VERSION = (
    CONFIGURATION_FINGERPRINT.get(
        "fingerprint_version"
    )
)


if not EXPECTED_CONFIG_HASH:

    raise RuntimeError(
        "Notebook 00 configuration fingerprint "
        "does not contain the authoritative 'hash' field."
    )


if FINGERPRINT_ALGORITHM != (
    "SHA256"
):

    raise RuntimeError(
        "Unexpected configuration fingerprint algorithm.\n"
        f"Expected : SHA256\n"
        f"Found    : {FINGERPRINT_ALGORITHM}"
    )


if FINGERPRINT_VERSION != (
    "1.0"
):

    raise RuntimeError(
        "Unexpected configuration fingerprint version.\n"
        f"Expected : 1.0\n"
        f"Found    : {FINGERPRINT_VERSION}"
    )


print(
    "✓ Configuration fingerprint metadata validated"
)

print(
    f"  Algorithm : {FINGERPRINT_ALGORITHM}"
)

print(
    f"  Version   : {FINGERPRINT_VERSION}"
)

print(
    f"  Hash      : {EXPECTED_CONFIG_HASH}"
)


# --------------------------------------------------------------------------------------------------
# 20. Validate Train-Only Statistical Baseline Policy
# --------------------------------------------------------------------------------------------------

BASELINE_POLICY = {

    "fit_split":
        "train",

    "validation_used_for_fitting":
        False,

    "test_used_for_fitting":
        False,

    "target_used_as_predictor":
        False,

    "identifier_features_used":
        False,

    "provenance_features_used":
        False,

    "encoded_feature_matrix_used":
        False,
}


if BASELINE_POLICY[
    "fit_split"
] != "train":

    raise RuntimeError(
        "Statistical baselines must be fitted on TRAIN only."
    )


if BASELINE_POLICY[
    "validation_used_for_fitting"
]:

    raise RuntimeError(
        "Validation data must never be used for baseline fitting."
    )


if BASELINE_POLICY[
    "test_used_for_fitting"
]:

    raise RuntimeError(
        "Test data must never be used for baseline fitting."
    )


if BASELINE_POLICY[
    "target_used_as_predictor"
]:

    raise RuntimeError(
        "The target must not be used as a predictor."
    )


if BASELINE_POLICY[
    "identifier_features_used"
]:

    raise RuntimeError(
        "Explicit identifiers must not be used as generative features."
    )


if BASELINE_POLICY[
    "provenance_features_used"
]:

    raise RuntimeError(
        "Provenance fields must not be used as generative features."
    )


if BASELINE_POLICY[
    "encoded_feature_matrix_used"
]:

    raise RuntimeError(
        "Notebook 04 statistical baselines must operate on the "
        "canonical native generative representation, not the "
        "Notebook 02 encoded feature matrix."
    )


print(
    "✓ Train-only baseline policy validated"
)

print(
    "  Fitting split       : TRAIN"
)

print(
    "  Validation fitting  : DISABLED"
)

print(
    "  Test fitting        : DISABLED"
)

print(
    "  Target as predictor : DISABLED"
)

print(
    "  Explicit IDs        : EXCLUDED"
)

print(
    "  Provenance          : EXCLUDED"
)

print(
    "  Encoded matrix      : EXCLUDED"
)


# --------------------------------------------------------------------------------------------------
# 21. Validate Notebook 04 Experimental Run Count
# --------------------------------------------------------------------------------------------------
#
# This is the authoritative expected run count for the subsequent sections.
#
# Experimental unit:
#     dataset × repetition × method
#
# Therefore:
#     3 datasets × 5 repetitions × 2 statistical baselines = 30 runs
# --------------------------------------------------------------------------------------------------

EXPECTED_EXPERIMENTAL_RUNS = (
    len(EXPECTED_DATASETS)
    *
    REPETITIONS
    *
    len(STATISTICAL_BASELINE_IDS)
)


if EXPECTED_EXPERIMENTAL_RUNS != 30:

    raise RuntimeError(
        "Unexpected Notebook 04 experimental run count.\n"
        f"Expected : 30\n"
        f"Computed : {EXPECTED_EXPERIMENTAL_RUNS}\n"
        f"Datasets : {len(EXPECTED_DATASETS)}\n"
        f"Repetitions: {REPETITIONS}\n"
        f"Baselines : {len(STATISTICAL_BASELINE_IDS)}"
    )


print(
    "✓ Experimental run-count policy validated"
)

print(
    f"  Datasets     : {len(EXPECTED_DATASETS)}"
)

print(
    f"  Repetitions  : {REPETITIONS}"
)

print(
    f"  Baselines    : {len(STATISTICAL_BASELINE_IDS)}"
)

print(
    f"  Expected runs: {EXPECTED_EXPERIMENTAL_RUNS}"
)


# --------------------------------------------------------------------------------------------------
# 22. Freeze Configuration Snapshot for Notebook 04
# --------------------------------------------------------------------------------------------------

NB04_CONFIGURATION = {

    "project_root":
        str(PROJECT_ROOT),

    "notebook_00_root":
        str(NB00_ROOT),

    "master_seed":
        MASTER_SEED,

    "deterministic":
        DETERMINISTIC,

    "repetitions":
        REPETITIONS,

    "repetition_seed_offset":
        REPETITION_SEED_OFFSET,

    "repetition_seeds": {
        str(key): int(value)
        for key, value in REPETITION_SEEDS.items()
    },

    "datasets":
        EXPECTED_DATASETS.copy(),

    "expected_splits":
        sorted(EXPECTED_SPLITS),

    "fit_split":
        EXPECTED_FIT_SPLIT,

    "statistical_baselines":
        STATISTICAL_BASELINE_IDS.copy(),

    "expected_experimental_runs":
        EXPECTED_EXPERIMENTAL_RUNS,

    "primary_experimental_unit":
        PRIMARY_EXPERIMENTAL_UNIT,

    "baseline_policy":
        BASELINE_POLICY.copy(),

    "notebook_00_manifest_version":
        MANIFEST_VERSION,

    "notebook_00_configuration_hash":
        EXPECTED_CONFIG_HASH,
}


# --------------------------------------------------------------------------------------------------
# 23. Create Deterministic Configuration Digest for Notebook 04
# --------------------------------------------------------------------------------------------------

NB04_CONFIGURATION_SERIALIZED = json.dumps(
    NB04_CONFIGURATION,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
)


NB04_CONFIGURATION_HASH = hashlib.sha256(
    NB04_CONFIGURATION_SERIALIZED.encode(
        "utf-8"
    )
).hexdigest()


print()
print("-" * 100)
print("NOTEBOOK 04 CONFIGURATION SNAPSHOT")
print("-" * 100)

print(
    f"✓ Master seed              : {MASTER_SEED}"
)

print(
    f"✓ Repetitions              : {REPETITIONS}"
)

print(
    f"✓ Fit split                : {EXPECTED_FIT_SPLIT}"
)

print(
    f"✓ Statistical baselines    : "
    f"{STATISTICAL_BASELINE_IDS}"
)

print(
    f"✓ Datasets                 : "
    f"{list(EXPECTED_DATASETS.keys())}"
)

print(
    f"✓ Experimental unit        : "
    f"{PRIMARY_EXPERIMENTAL_UNIT}"
)

print(
    f"✓ Expected experimental runs: "
    f"{EXPECTED_EXPERIMENTAL_RUNS}"
)

print(
    f"✓ Notebook 00 config hash  : "
    f"{EXPECTED_CONFIG_HASH}"
)

print(
    f"✓ Notebook 04 config hash  : "
    f"{NB04_CONFIGURATION_HASH}"
)


# --------------------------------------------------------------------------------------------------
# 24. Final Section 02 Verification
# --------------------------------------------------------------------------------------------------

SECTION_02_CHECKS = {

    "google_drive_verified":
        DRIVE_ROOT.exists()
        and DRIVE_ROOT.is_dir(),

    "mydrive_verified":
        MYDRIVE_ROOT.exists()
        and MYDRIVE_ROOT.is_dir(),

    "project_root_verified":
        PROJECT_ROOT.exists()
        and PROJECT_ROOT.is_dir(),

    "notebook_00_root_verified":
        NB00_ROOT.exists()
        and NB00_ROOT.is_dir(),

    "all_required_directories_present":
        all(
            path.exists()
            and path.is_dir()
            for path in REQUIRED_NB00_DIRECTORIES.values()
        ),

    "all_required_artifacts_present":
        all(
            path.exists()
            and path.is_file()
            and path.stat().st_size > 0
            for path in NB00_ARTIFACT_PATHS.values()
        ),

    "experiment_config_loaded":
        isinstance(
            EXPERIMENT_CONFIG,
            dict,
        ),

    "dataset_registry_loaded":
        isinstance(
            DATASET_REGISTRY,
            dict,
        ),

    "model_registry_loaded":
        isinstance(
            MODEL_REGISTRY,
            dict,
        ),

    "evaluation_config_loaded":
        isinstance(
            EVALUATION_CONFIG,
            dict,
        ),

    "privacy_config_loaded":
        isinstance(
            PRIVACY_CONFIG,
            dict,
        ),

    "sppgan_config_loaded":
        isinstance(
            SPPGAN_CONFIG,
            dict,
        ),

    "environment_loaded":
        isinstance(
            ENVIRONMENT,
            dict,
        ),

    "manifest_loaded":
        isinstance(
            NOTEBOOK_00_MANIFEST,
            dict,
        ),

    "configuration_fingerprint_loaded":
        isinstance(
            CONFIGURATION_FINGERPRINT,
            dict,
        ),

    "master_seed_verified":
        MASTER_SEED == 2025,

    "deterministic_policy_verified":
        DETERMINISTIC is True,

    "repetition_count_verified":
        REPETITIONS == 5,

    "repetition_seed_offset_verified":
        REPETITION_SEED_OFFSET == 1000,

    "repetition_seed_registry_verified":
        ACTUAL_REPETITION_KEYS
        ==
        EXPECTED_REPETITION_KEYS,

    "repetition_seed_values_verified":
        all(
            int(
                REPETITION_SEEDS[str(i)]
            )
            ==
            (
                MASTER_SEED
                +
                REPETITION_SEED_OFFSET
                +
                i
            )
            for i in range(
                1,
                REPETITIONS + 1,
            )
        ),

    "train_only_fit_policy_verified":
        EXPECTED_FIT_SPLIT == "train",

    "validation_excluded_from_fitting":
        BASELINE_POLICY[
            "validation_used_for_fitting"
        ]
        is False,

    "test_excluded_from_fitting":
        BASELINE_POLICY[
            "test_used_for_fitting"
        ]
        is False,

    "target_not_predictor":
        BASELINE_POLICY[
            "target_used_as_predictor"
        ]
        is False,

    "identifiers_excluded":
        BASELINE_POLICY[
            "identifier_features_used"
        ]
        is False,

    "provenance_excluded":
        BASELINE_POLICY[
            "provenance_features_used"
        ]
        is False,

    "encoded_matrix_excluded":
        BASELINE_POLICY[
            "encoded_feature_matrix_used"
        ]
        is False,

    "dataset_registry_verified":
        set(
            EXPECTED_DATASETS.keys()
        )
        ==
        set(
            DATASET_ENTRIES.keys()
        )
        or
        set(
            EXPECTED_DATASETS.keys()
        ).issubset(
            set(
                DATASET_ENTRIES.keys()
            )
        ),

    "experimental_unit_verified":
        PRIMARY_EXPERIMENTAL_UNIT
        ==
        "dataset × repetition × method",

    "experimental_run_count_verified":
        EXPECTED_EXPERIMENTAL_RUNS
        ==
        30,

    "manifest_version_verified":
        MANIFEST_VERSION
        ==
        "1.0",

    "manifest_project_name_verified":
        MANIFEST_PROJECT_NAME
        ==
        "SPP-GAN Research Project",

    "manifest_project_version_verified":
        MANIFEST_PROJECT_VERSION
        ==
        "1.0",

    "manifest_notebook_id_verified":
        str(
            MANIFEST_NOTEBOOK_ID
        )
        ==
        "00",

    "manifest_notebook_name_verified":
        MANIFEST_NOTEBOOK_NAME
        ==
        "Environment, Configuration & Reproducibility",

    "manifest_no_obsolete_integrity_dependency":
        "integrity"
        not in NOTEBOOK_00_MANIFEST,

    "configuration_fingerprint_verified":
        bool(
            EXPECTED_CONFIG_HASH
        ),

    "fingerprint_algorithm_verified":
        FINGERPRINT_ALGORITHM
        ==
        "SHA256",

    "fingerprint_version_verified":
        FINGERPRINT_VERSION
        ==
        "1.0",

    "baseline_registry_verified":
        len(
            STATISTICAL_BASELINE_IDS
        )
        ==
        2,

    "independent_marginal_registered":
        "independent_marginal"
        in
        STATISTICAL_BASELINE_IDS,

    "gaussian_copula_registered":
        "gaussian_copula"
        in
        STATISTICAL_BASELINE_IDS,
}


FAILED_SECTION_02_CHECKS = [

    check_name

    for check_name, result
    in SECTION_02_CHECKS.items()

    if not result
]


print()
print("=" * 100)
print("SECTION 02 FINAL VERIFICATION")
print("=" * 100)

print(
    f"Total checks : "
    f"{len(SECTION_02_CHECKS)}"
)

print(
    f"Passed       : "
    f"{sum(SECTION_02_CHECKS.values())}"
)

print(
    f"Failed       : "
    f"{len(FAILED_SECTION_02_CHECKS)}"
)


if FAILED_SECTION_02_CHECKS:

    print()
    print("FAILED CHECKS")
    print("-" * 100)


    for check_name in FAILED_SECTION_02_CHECKS:

        print(
            f"✗ {check_name}"
        )


    raise RuntimeError(
        "Notebook 04 Section 02 verification FAILED."
    )


print()
print("✓ ALL SECTION 02 CHECKS PASSED")
print("✓ Notebook 00 authoritative configuration loaded successfully")
print("✓ Authoritative master seed : 2025")
print("✓ Repetition policy         : 5 repetitions")
print("✓ Repetition seeds         : 3026–3030")
print("✓ Expected experimental runs: 30")
print("✓ Primary experimental unit : dataset × repetition × method")
print("✓ Train-only baseline policy verified")
print("✓ Notebook 00 manifest identity verified")
print("✓ Notebook 00 configuration fingerprint verified")
print("✓ Statistical baseline registry verified")
print()
print("SECTION 02 STATUS : PASS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 25. Memory Cleanup
# --------------------------------------------------------------------------------------------------

gc.collect()

SECTION 02 — LOAD CONFIGURATION
✓ Google Drive verified : /content/drive
✓ MyDrive verified      : /content/drive/MyDrive
✓ Project root verified : /content/drive/MyDrive/SPP_GAN_Research

----------------------------------------------------------------------------------------------------
NOTEBOOK 00 ARTIFACT ROOTS
----------------------------------------------------------------------------------------------------
✓ Notebook 00 root       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
✓ Configuration root     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/config
✓ Environment root       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/environment
✓ Manifest root          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00/manifest
✓ Notebook 00 root              : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_00
✓ Configuration directory       : /content/drive/MyDrive/SPP_G

0

In [71]:
# ==================================================================================================
# SECTION 03 — LOAD PROCESSED TRAINING DATA
# ==================================================================================================

print("=" * 100)
print("SECTION 03 — LOAD PROCESSED TRAINING DATA")
print("=" * 100)

from pathlib import Path
import ast
import gc
import json
import hashlib
import pandas as pd
import numpy as np


# ==================================================================================================
# 1. VERIFY GOOGLE DRIVE AND PROJECT ROOT
# ==================================================================================================

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

if not DRIVE_ROOT.exists():
    raise RuntimeError(
        f"Google Drive not mounted: {DRIVE_ROOT}"
    )

if not MYDRIVE_ROOT.exists():
    raise RuntimeError(
        f"MyDrive not found: {MYDRIVE_ROOT}"
    )

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Project root not found: {PROJECT_ROOT}"
    )

print(f"✓ Google Drive     : {DRIVE_ROOT}")
print(f"✓ MyDrive          : {MYDRIVE_ROOT}")
print(f"✓ Project root     : {PROJECT_ROOT}")


# ==================================================================================================
# 2. CANONICAL NOTEBOOK 02 PATHS
# ==================================================================================================

NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

NB02_NATIVE_ROOT = (
    NB02_ROOT
    / "native"
)

NATIVE_MANIFEST_PATH = (
    NB02_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

if not NB02_ROOT.exists():
    raise RuntimeError(
        f"Notebook 02 root not found: {NB02_ROOT}"
    )

if not NB02_NATIVE_ROOT.exists():
    raise RuntimeError(
        f"Notebook 02 native directory not found: "
        f"{NB02_NATIVE_ROOT}"
    )

if not NATIVE_MANIFEST_PATH.exists():
    raise RuntimeError(
        f"Notebook 02 native manifest not found: "
        f"{NATIVE_MANIFEST_PATH}"
    )

print()
print(f"✓ Notebook 02 root : {NB02_ROOT}")
print(f"✓ Native data root : {NB02_NATIVE_ROOT}")
print(f"✓ Native manifest  : {NATIVE_MANIFEST_PATH}")


# ==================================================================================================
# 3. EXPECTED DATASET REGISTRY
# ==================================================================================================

EXPECTED_DATASETS = {
    "adult_income": {
        "target": "income",
        "identifiers": [],
    },
    "bank_marketing": {
        "target": "y",
        "identifiers": [],
    },
    "diabetes_130us": {
        "target": "readmitted",
        "identifiers": [
            "encounter_id",
            "patient_nbr",
        ],
    },
}

DATASET_IDS = list(
    EXPECTED_DATASETS.keys()
)

TARGET_COLUMNS = {
    dataset_id: EXPECTED_DATASETS[dataset_id]["target"]
    for dataset_id in DATASET_IDS
}

print()
print("Expected datasets:")

for dataset_id in DATASET_IDS:
    print(
        f"  • {dataset_id:<20} "
        f"target = "
        f"{EXPECTED_DATASETS[dataset_id]['target']}"
    )


# ==================================================================================================
# 4. HELPER FUNCTIONS
# ==================================================================================================

def normalize_scalar(value):
    """
    Normalize pandas / NumPy scalar values.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, np.generic):
        return value.item()

    return value


def parse_manifest_count(value, field_name):
    """
    Parse an integer count persisted in the Notebook 02 manifest.
    """

    value = normalize_scalar(value)

    if value is None:
        raise RuntimeError(
            f"{field_name} cannot be null."
        )

    if isinstance(
        value,
        bool,
    ):
        raise RuntimeError(
            f"{field_name} cannot be boolean."
        )

    try:
        return int(value)

    except Exception as exc:

        raise RuntimeError(
            f"Unable to parse {field_name}: "
            f"{value!r}"
        ) from exc


def parse_boolean(value):
    """
    Parse boolean-like manifest values.
    """

    value = normalize_scalar(value)

    if isinstance(value, bool):
        return value

    if isinstance(
        value,
        (int, np.integer),
    ):

        if value in (0, 1):
            return bool(value)

    if isinstance(value, str):

        normalized = (
            value
            .strip()
            .lower()
        )

        if normalized in {
            "true",
            "1",
            "yes",
            "y",
            "pass",
        }:
            return True

        if normalized in {
            "false",
            "0",
            "no",
            "n",
            "fail",
        }:
            return False

    raise RuntimeError(
        f"Unable to parse boolean manifest value: "
        f"{value!r}"
    )


def parse_identifier_columns(value):
    """
    Parse Notebook 02 identifier_columns.
    """

    value = normalize_scalar(value)

    if value is None:
        return []

    if isinstance(value, list):
        return [
            str(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return [
            str(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return [
            str(item)
            for item in value.tolist()
        ]

    if isinstance(value, str):

        text = value.strip()

        if not text:
            return []

        try:

            parsed = json.loads(text)

            if isinstance(
                parsed,
                list,
            ):

                return [
                    str(item)
                    for item in parsed
                ]

        except Exception:
            pass

        try:

            parsed = ast.literal_eval(text)

            if isinstance(
                parsed,
                (list, tuple),
            ):

                return [
                    str(item)
                    for item in parsed
                ]

        except Exception:
            pass

        if "," in text:

            return [
                item.strip()
                for item in text.split(",")
                if item.strip()
            ]

        return [text]

    raise RuntimeError(
        f"Unable to parse identifier_columns: "
        f"{value!r}"
    )


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    """
    Calculate SHA-256 without loading the complete file into RAM.
    """

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def resolve_manifest_path(
    relative_path,
    absolute_path,
):
    """
    Resolve canonical Notebook 02 native TRAIN file.
    """

    relative_path = normalize_scalar(
        relative_path
    )

    absolute_path = normalize_scalar(
        absolute_path
    )

    candidates = []

    if absolute_path:

        candidates.append(
            Path(
                str(absolute_path)
            )
        )

    if relative_path:

        relative = Path(
            str(relative_path)
        )

        candidates.extend([
            PROJECT_ROOT / relative,
            NB02_ROOT / relative,
            NB02_NATIVE_ROOT / relative,
        ])

    for candidate in candidates:

        if (
            candidate.exists()
            and candidate.is_file()
        ):

            return candidate

    raise FileNotFoundError(
        "Unable to resolve Notebook 02 native data file.\n"
        f"relative_path={relative_path!r}\n"
        f"absolute_path={absolute_path!r}"
    )


# ==================================================================================================
# 5. LOAD CANONICAL NOTEBOOK 02 NATIVE MANIFEST
# ==================================================================================================

print()
print("-" * 100)
print("LOAD NOTEBOOK 02 NATIVE MANIFEST")
print("-" * 100)

native_manifest = pd.read_csv(
    NATIVE_MANIFEST_PATH
)

print(
    "✓ Native manifest loaded"
)

print(
    f"  Rows    : {len(native_manifest):,}"
)

print(
    f"  Columns : {len(native_manifest.columns):,}"
)


# ==================================================================================================
# 6. VALIDATE EXACT MANIFEST SCHEMA
# ==================================================================================================

EXPECTED_MANIFEST_COLUMNS = [
    "dataset_id",
    "split",
    "relative_path",
    "absolute_path",
    "rows",
    "columns",
    "preprocessing_feature_columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "file_size_bytes",
    "sha256",
    "reload_validation",
    "status",
]

actual_manifest_columns = list(
    native_manifest.columns
)

missing_manifest_columns = [
    column
    for column in EXPECTED_MANIFEST_COLUMNS
    if column not in actual_manifest_columns
]

unexpected_manifest_columns = [
    column
    for column in actual_manifest_columns
    if column not in EXPECTED_MANIFEST_COLUMNS
]

if missing_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest is missing required columns:\n"
        + "\n".join(
            f"- {column}"
            for column in missing_manifest_columns
        )
    )

if unexpected_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest contains unexpected columns:\n"
        + "\n".join(
            f"- {column}"
            for column in unexpected_manifest_columns
        )
    )

if actual_manifest_columns != (
    EXPECTED_MANIFEST_COLUMNS
):

    raise RuntimeError(
        "Notebook 02 native manifest column order "
        "does not match the canonical schema."
    )

print(
    "✓ Exact required manifest schema verified"
)


# ==================================================================================================
# 7. VALIDATE MANIFEST ROW COUNT
# ==================================================================================================

EXPECTED_MANIFEST_ROWS = (
    len(DATASET_IDS) * 3
)

if len(native_manifest) != (
    EXPECTED_MANIFEST_ROWS
):

    raise RuntimeError(
        f"Expected {EXPECTED_MANIFEST_ROWS} "
        f"Notebook 02 native manifest records, "
        f"found {len(native_manifest)}."
    )

print(
    f"✓ Manifest row count verified: "
    f"{len(native_manifest)}"
)


# ==================================================================================================
# 8. VALIDATE DATASET COVERAGE
# ==================================================================================================

manifest_dataset_ids = sorted(
    native_manifest[
        "dataset_id"
    ]
    .astype(str)
    .unique()
)

if manifest_dataset_ids != (
    sorted(DATASET_IDS)
):

    raise RuntimeError(
        "Dataset coverage mismatch.\n"
        f"Expected: {sorted(DATASET_IDS)}\n"
        f"Found   : {manifest_dataset_ids}"
    )

print(
    "✓ Dataset coverage verified: "
    + ", ".join(DATASET_IDS)
)


# ==================================================================================================
# 9. VALIDATE SPLIT COVERAGE
# ==================================================================================================

EXPECTED_SPLITS = {
    "train",
    "validation",
    "test",
}

for dataset_id in DATASET_IDS:

    dataset_manifest = native_manifest[
        native_manifest[
            "dataset_id"
        ].astype(str)
        == dataset_id
    ]

    observed_splits = set(
        dataset_manifest[
            "split"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    if observed_splits != (
        EXPECTED_SPLITS
    ):

        raise RuntimeError(
            f"{dataset_id}: expected "
            f"TRAIN/VALIDATION/TEST coverage, "
            f"found {sorted(observed_splits)}."
        )

print(
    "✓ TRAIN / VALIDATION / TEST coverage "
    "verified for every dataset"
)


# ==================================================================================================
# 10. VALIDATE MANIFEST STATUS
# ==================================================================================================

status_values = (
    native_manifest[
        "status"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)

non_pass_status = native_manifest[
    status_values != "PASS"
]

if not non_pass_status.empty:

    raise RuntimeError(
        "Notebook 02 native manifest contains "
        "non-PASS records:\n"
        + non_pass_status[
            [
                "dataset_id",
                "split",
                "status",
            ]
        ].to_string(index=False)
    )

print(
    "✓ All Notebook 02 native manifest "
    "records have status=PASS"
)


# ==================================================================================================
# 11. VALIDATE PERSISTED NOTEBOOK 02 FLAGS
# ==================================================================================================

BOOLEAN_FLAG_COLUMNS = [
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "reload_validation",
]

for column in BOOLEAN_FLAG_COLUMNS:

    for index, value in native_manifest[
        column
    ].items():

        parsed = parse_boolean(
            value
        )

        if not parsed:

            dataset_id = native_manifest.loc[
                index,
                "dataset_id"
            ]

            split = native_manifest.loc[
                index,
                "split"
            ]

            raise RuntimeError(
                f"Notebook 02 integrity flag failed: "
                f"{column}={value!r} | "
                f"dataset={dataset_id} | "
                f"split={split}"
            )

print(
    "✓ Notebook 02 persisted integrity flags verified"
)


# ==================================================================================================
# 12. SELECT TRAIN RECORDS
# ==================================================================================================

train_manifest = native_manifest[
    native_manifest[
        "split"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    == "train"
].copy()

if len(train_manifest) != (
    len(DATASET_IDS)
):

    raise RuntimeError(
        f"Expected {len(DATASET_IDS)} TRAIN "
        f"manifest records, "
        f"found {len(train_manifest)}."
    )

print()
print(
    f"✓ TRAIN records selected : "
    f"{len(train_manifest)}"
)


# ==================================================================================================
# 13. VALIDATE TRAIN MANIFEST UNIQUENESS
# ==================================================================================================

train_duplicates = train_manifest[
    train_manifest[
        "dataset_id"
    ].duplicated(keep=False)
]

if not train_duplicates.empty:

    raise RuntimeError(
        "Duplicate TRAIN manifest records detected:\n"
        + train_duplicates[
            [
                "dataset_id",
                "split",
                "relative_path",
            ]
        ].to_string(index=False)
    )

print(
    "✓ No duplicate TRAIN manifest records"
)


# ==================================================================================================
# 14. INITIALIZE RUNTIME CONTAINERS
# ==================================================================================================

TRAINING_DATA = {}

TRAINING_METADATA = {}

TRAINING_FEATURE_COLUMNS = {}

TRAINING_GENERATIVE_COLUMNS = {}

TRAINING_TARGET_COLUMNS = {}

TRAINING_PROVENANCE_COLUMNS = {}

TRAINING_IDENTIFIER_COLUMNS = {}

print()
print(
    "Runtime containers initialized:"
)

print(
    "  ✓ TRAINING_DATA"
)

print(
    "  ✓ TRAINING_METADATA"
)

print(
    "  ✓ TRAINING_FEATURE_COLUMNS"
)

print(
    "  ✓ TRAINING_GENERATIVE_COLUMNS"
)

print(
    "  ✓ TRAINING_TARGET_COLUMNS"
)

print(
    "  ✓ TRAINING_PROVENANCE_COLUMNS"
)

print(
    "  ✓ TRAINING_IDENTIFIER_COLUMNS"
)


# ==================================================================================================
# 15. LOAD AND VALIDATE CANONICAL TRAINING DATA
# ==================================================================================================

print()
print("=" * 100)
print("LOADING CANONICAL NOTEBOOK 02 TRAIN DATA")
print("=" * 100)

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(
        f"DATASET : {dataset_id}"
    )
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Locate TRAIN record
    # ----------------------------------------------------------------------------------------------

    train_record = train_manifest[
        train_manifest[
            "dataset_id"
        ].astype(str)
        == dataset_id
    ]

    if len(train_record) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one "
            f"TRAIN manifest record, "
            f"found {len(train_record)}."
        )

    record = train_record.iloc[0]

    # ----------------------------------------------------------------------------------------------
    # Manifest metadata
    # ----------------------------------------------------------------------------------------------

    split = (
        str(record["split"])
        .strip()
        .lower()
    )

    if split != "train":

        raise RuntimeError(
            f"{dataset_id}: selected manifest "
            f"record is not TRAIN."
        )

    target_column = (
        str(record["target_column"])
        .strip()
    )

    provenance_column = (
        str(record["provenance_column"])
        .strip()
    )

    manifest_identifiers = (
        parse_identifier_columns(
            record[
                "identifier_columns"
            ]
        )
    )

    manifest_feature_count = (
        parse_manifest_count(
            record[
                "preprocessing_feature_columns"
            ],
            f"{dataset_id}: "
            f"preprocessing_feature_columns",
        )
    )

    manifest_generative_count = (
        parse_manifest_count(
            record[
                "generative_columns"
            ],
            f"{dataset_id}: "
            f"generative_columns",
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Expected registry
    # ----------------------------------------------------------------------------------------------

    expected_target = (
        EXPECTED_DATASETS[
            dataset_id
        ]["target"]
    )

    expected_identifiers = (
        EXPECTED_DATASETS[
            dataset_id
        ]["identifiers"]
    )

    if target_column != expected_target:

        raise RuntimeError(
            f"{dataset_id}: target mismatch.\n"
            f"Expected: {expected_target}\n"
            f"Found   : {target_column}"
        )

    if manifest_identifiers != (
        expected_identifiers
    ):

        raise RuntimeError(
            f"{dataset_id}: identifier registry mismatch.\n"
            f"Expected: {expected_identifiers}\n"
            f"Found   : {manifest_identifiers}"
        )

    # ----------------------------------------------------------------------------------------------
    # Resolve canonical TRAIN file
    # ----------------------------------------------------------------------------------------------

    train_path = resolve_manifest_path(
        record["relative_path"],
        record["absolute_path"],
    )

    print(
        f"  ✓ Split               : "
        f"{split}"
    )

    print(
        f"  ✓ Target column       : "
        f"{target_column}"
    )

    print(
        f"  ✓ Native TRAIN file  : "
        f"{train_path}"
    )

    # ----------------------------------------------------------------------------------------------
    # File-size integrity
    # ----------------------------------------------------------------------------------------------

    actual_file_size = (
        train_path.stat().st_size
    )

    expected_file_size = (
        parse_manifest_count(
            record["file_size_bytes"],
            f"{dataset_id}: file_size_bytes",
        )
    )

    if actual_file_size != (
        expected_file_size
    ):

        raise RuntimeError(
            f"{dataset_id}: file size mismatch.\n"
            f"Expected: {expected_file_size}\n"
            f"Found   : {actual_file_size}"
        )

    print(
        f"  ✓ File size           : "
        f"{actual_file_size:,} bytes"
    )

    # ----------------------------------------------------------------------------------------------
    # SHA-256 integrity
    # ----------------------------------------------------------------------------------------------

    expected_sha256 = (
        str(
            record["sha256"]
        )
        .strip()
        .lower()
    )

    actual_sha256 = (
        sha256_file(
            train_path
        )
        .lower()
    )

    if actual_sha256 != (
        expected_sha256
    ):

        raise RuntimeError(
            f"{dataset_id}: SHA-256 mismatch.\n"
            f"Expected: {expected_sha256}\n"
            f"Found   : {actual_sha256}"
        )

    print(
        f"  ✓ SHA-256             : "
        f"{actual_sha256}"
    )

    # ----------------------------------------------------------------------------------------------
    # Load TRAIN data
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(
        train_path
    )

    expected_rows = (
        parse_manifest_count(
            record["rows"],
            f"{dataset_id}: rows",
        )
    )

    expected_native_columns = (
        parse_manifest_count(
            record["columns"],
            f"{dataset_id}: columns",
        )
    )

    if len(df) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: row count mismatch.\n"
            f"Expected: {expected_rows}\n"
            f"Found   : {len(df)}"
        )

    if len(df.columns) != (
        expected_native_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: column count mismatch.\n"
            f"Expected: {expected_native_columns}\n"
            f"Found   : {len(df.columns)}"
        )

    print(
        f"  ✓ Loaded TRAIN data  : "
        f"{len(df):,} rows × "
        f"{len(df.columns):,} columns"
    )

    print(
        f"  ✓ Row count           : "
        f"{len(df):,}"
    )

    print(
        f"  ✓ Column count        : "
        f"{len(df.columns):,}"
    )

    # ----------------------------------------------------------------------------------------------
    # Column uniqueness
    # ----------------------------------------------------------------------------------------------

    if not df.columns.is_unique:

        duplicated_columns = (
            df.columns[
                df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate column "
            f"names found:\n"
            f"{duplicated_columns}"
        )

    print(
        "  ✓ Column names unique"
    )

    # ----------------------------------------------------------------------------------------------
    # Target presence
    # ----------------------------------------------------------------------------------------------

    if target_column not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column missing."
        )

    print(
        f"  ✓ Target present      : "
        f"{target_column}"
    )

    # ----------------------------------------------------------------------------------------------
    # Provenance presence
    # ----------------------------------------------------------------------------------------------

    if provenance_column not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column missing."
        )

    print(
        f"  ✓ Provenance present   : "
        f"{provenance_column}"
    )

    # ----------------------------------------------------------------------------------------------
    # Explicit identifier exclusion
    # ----------------------------------------------------------------------------------------------

    present_identifiers = [
        column
        for column in expected_identifiers
        if column in df.columns
    ]

    if present_identifiers:

        raise RuntimeError(
            f"{dataset_id}: explicit identifiers "
            f"are present in native TRAIN data:\n"
            f"{present_identifiers}"
        )

    print(
        "  ✓ Identifier exclusion verified: "
        + (
            "none"
            if not expected_identifiers
            else str(expected_identifiers)
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Derive generative columns from ACTUAL native schema
    # ----------------------------------------------------------------------------------------------

    actual_columns = list(
        df.columns
    )

    actual_generative_columns = [
        column
        for column in actual_columns
        if column != provenance_column
        and column not in expected_identifiers
    ]

    actual_generative_count = len(
        actual_generative_columns
    )

    if actual_generative_count != (
        manifest_generative_count
    ):

        raise RuntimeError(
            f"{dataset_id}: generative column count mismatch.\n"
            f"Manifest count: {manifest_generative_count}\n"
            f"Native count  : {actual_generative_count}"
        )

    generative_columns = (
        actual_generative_columns.copy()
    )

    print(
        f"  ✓ Generative columns : "
        f"{len(generative_columns)}"
    )

    # ----------------------------------------------------------------------------------------------
    # Target retained in generative schema
    # ----------------------------------------------------------------------------------------------

    if target_column not in (
        generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: target is missing "
            f"from generative schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Derive feature columns
    # ----------------------------------------------------------------------------------------------

    feature_columns = [
        column
        for column in generative_columns
        if column != target_column
    ]

    if len(feature_columns) != (
        manifest_feature_count
    ):

        raise RuntimeError(
            f"{dataset_id}: feature column count mismatch.\n"
            f"Manifest count: {manifest_feature_count}\n"
            f"Native count  : {len(feature_columns)}"
        )

    print(
        f"  ✓ Feature columns    : "
        f"{len(feature_columns)}"
    )

    # ----------------------------------------------------------------------------------------------
    # Schema count identity
    # ----------------------------------------------------------------------------------------------

    if (
        len(feature_columns)
        + 1
        != len(generative_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: feature + target "
            f"does not equal generative schema."
        )

    if (
        len(generative_columns)
        + 1
        != len(df.columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: generative + provenance "
            f"does not equal native schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Generative schema membership
    # ----------------------------------------------------------------------------------------------

    if not set(
        generative_columns
    ).issubset(
        set(actual_columns)
    ):

        raise RuntimeError(
            f"{dataset_id}: generative schema contains "
            f"columns missing from native TRAIN data."
        )

    print(
        "  ✓ Generative schema membership verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Native schema order
    # ----------------------------------------------------------------------------------------------

    provenance_positions = [
        index
        for index, column in enumerate(
            actual_columns
        )
        if column == provenance_column
    ]

    if len(provenance_positions) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one "
            f"provenance position, "
            f"found {provenance_positions}"
        )

    provenance_position = (
        provenance_positions[0]
    )

    print(
        f"  ✓ Native schema order verified "
        f"(provenance position={provenance_position})"
    )

    # ----------------------------------------------------------------------------------------------
    # Target uniqueness
    # ----------------------------------------------------------------------------------------------

    target_occurrences = sum(
        column == target_column
        for column in actual_columns
    )

    if target_occurrences != 1:

        raise RuntimeError(
            f"{dataset_id}: target column occurs "
            f"{target_occurrences} times."
        )

    print(
        "  ✓ Target uniqueness verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Provenance uniqueness
    # ----------------------------------------------------------------------------------------------

    provenance_occurrences = sum(
        column == provenance_column
        for column in actual_columns
    )

    if provenance_occurrences != 1:

        raise RuntimeError(
            f"{dataset_id}: provenance column occurs "
            f"{provenance_occurrences} times."
        )

    print(
        "  ✓ Provenance uniqueness verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Provenance integrity
    # ----------------------------------------------------------------------------------------------

    provenance_series = df[
        provenance_column
    ]

    if provenance_series.isna().any():

        raise RuntimeError(
            f"{dataset_id}: provenance contains "
            f"missing values."
        )

    if not provenance_series.is_unique:

        raise RuntimeError(
            f"{dataset_id}: provenance values "
            f"are not unique."
        )

    print(
        "  ✓ Provenance integrity verified"
    )

    # ----------------------------------------------------------------------------------------------
    # Target non-empty
    # ----------------------------------------------------------------------------------------------

    if df[
        target_column
    ].dropna().empty:

        raise RuntimeError(
            f"{dataset_id}: target data is empty."
        )

    print(
        "  ✓ Target data is non-empty"
    )

    # ----------------------------------------------------------------------------------------------
    # Notebook 02 schema flags
    # ----------------------------------------------------------------------------------------------

    for flag_column in (
        BOOLEAN_FLAG_COLUMNS
    ):

        if not parse_boolean(
            record[flag_column]
        ):

            raise RuntimeError(
                f"{dataset_id}: Notebook 02 "
                f"flag {flag_column} is not TRUE."
            )

    print(
        "  ✓ Notebook 02 schema flags verified"
    )

    # ==============================================================================================
    # CRITICAL RUNTIME CONTAINER PERSISTENCE
    # ==============================================================================================

    TRAINING_DATA[
        dataset_id
    ] = df

    TRAINING_FEATURE_COLUMNS[
        dataset_id
    ] = feature_columns.copy()

    TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ] = generative_columns.copy()

    TRAINING_TARGET_COLUMNS[
        dataset_id
    ] = target_column

    TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ] = provenance_column

    TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ] = expected_identifiers.copy()

    TRAINING_METADATA[
        dataset_id
    ] = {
        "dataset_id": dataset_id,
        "split": split,
        "native_path": str(train_path),
        "rows": len(df),
        "native_columns": len(df.columns),
        "feature_columns": feature_columns.copy(),
        "generative_columns": generative_columns.copy(),
        "target_column": target_column,
        "provenance_column": provenance_column,
        "identifier_columns": expected_identifiers.copy(),
        "provenance_position": provenance_position,
        "file_size_bytes": actual_file_size,
        "sha256": actual_sha256,
        "manifest_feature_count": manifest_feature_count,
        "manifest_generative_count": manifest_generative_count,
    }

    print(
        f"  ✓ {dataset_id} successfully validated"
    )


# ==================================================================================================
# 16. FINAL TRAINING DATA COVERAGE
# ==================================================================================================

print()
print("=" * 100)
print("FINAL TRAINING DATA COVERAGE")
print("=" * 100)

if set(
    TRAINING_DATA.keys()
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "TRAINING_DATA dataset coverage mismatch.\n"
        f"Expected: {DATASET_IDS}\n"
        f"Found   : {list(TRAINING_DATA.keys())}"
    )

print(
    "✓ Loaded datasets: "
    + ", ".join(DATASET_IDS)
)


# ==================================================================================================
# 17. FINAL RUNTIME SCHEMA CONTAINER COMPLETENESS
# ==================================================================================================

print()
print("-" * 100)
print("FINAL RUNTIME SCHEMA CONTAINER COMPLETENESS")
print("-" * 100)

EXPECTED_SCHEMA_CONTAINERS = {
    "TRAINING_DATA": TRAINING_DATA,
    "TRAINING_METADATA": TRAINING_METADATA,
    "TRAINING_FEATURE_COLUMNS": TRAINING_FEATURE_COLUMNS,
    "TRAINING_GENERATIVE_COLUMNS": TRAINING_GENERATIVE_COLUMNS,
    "TRAINING_TARGET_COLUMNS": TRAINING_TARGET_COLUMNS,
    "TRAINING_PROVENANCE_COLUMNS": TRAINING_PROVENANCE_COLUMNS,
    "TRAINING_IDENTIFIER_COLUMNS": TRAINING_IDENTIFIER_COLUMNS,
}

for container_name, container in (
    EXPECTED_SCHEMA_CONTAINERS.items()
):

    missing_datasets = sorted(
        set(DATASET_IDS)
        - set(container.keys())
    )

    unexpected_datasets = sorted(
        set(container.keys())
        - set(DATASET_IDS)
    )

    if missing_datasets:

        raise RuntimeError(
            f"{container_name} is missing dataset entries: "
            f"{missing_datasets}"
        )

    if unexpected_datasets:

        raise RuntimeError(
            f"{container_name} contains unexpected dataset entries: "
            f"{unexpected_datasets}"
        )

    if len(container) != len(DATASET_IDS):

        raise RuntimeError(
            f"{container_name}: expected "
            f"{len(DATASET_IDS)} entries, "
            f"found {len(container)}."
        )

    print(
        f"✓ {container_name:<35} "
        f"{len(container)}/{len(DATASET_IDS)} datasets present"
    )


# ==================================================================================================
# 18. FINAL METADATA VERIFICATION
# ==================================================================================================

print()
print("-" * 100)
print("FINAL METADATA VERIFICATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[
        dataset_id
    ]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    identifiers = TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ]

    metadata = TRAINING_METADATA[
        dataset_id
    ]

    if metadata["rows"] != len(df):

        raise RuntimeError(
            f"{dataset_id}: metadata row count mismatch."
        )

    if metadata["native_columns"] != len(
        df.columns
    ):

        raise RuntimeError(
            f"{dataset_id}: metadata native column count mismatch."
        )

    if metadata["feature_columns"] != features:

        raise RuntimeError(
            f"{dataset_id}: metadata feature schema mismatch."
        )

    if metadata["generative_columns"] != (
        generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: metadata generative schema mismatch."
        )

    if metadata["target_column"] != target:

        raise RuntimeError(
            f"{dataset_id}: metadata target mismatch."
        )

    if metadata["provenance_column"] != provenance:

        raise RuntimeError(
            f"{dataset_id}: metadata provenance mismatch."
        )

    if metadata["identifier_columns"] != identifiers:

        raise RuntimeError(
            f"{dataset_id}: metadata identifier mismatch."
        )

    print(
        f"✓ {dataset_id:<20} "
        f"rows={len(df):,} | "
        f"native_cols={len(df.columns):>2} | "
        f"features={len(features):>2} | "
        f"generative={len(generative_columns):>2} | "
        f"target={target}"
    )


# ==================================================================================================
# 19. RUNTIME OBJECT VERIFICATION
# ==================================================================================================

print()
print("-" * 100)
print("RUNTIME OBJECT VERIFICATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    if not isinstance(
        TRAINING_DATA[dataset_id],
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_DATA is not "
            f"a pandas DataFrame."
        )

    if not isinstance(
        TRAINING_FEATURE_COLUMNS[dataset_id],
        list,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_FEATURE_COLUMNS "
            f"is not a list."
        )

    if not isinstance(
        TRAINING_GENERATIVE_COLUMNS[dataset_id],
        list,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_GENERATIVE_COLUMNS "
            f"is not a list."
        )

    if not isinstance(
        TRAINING_TARGET_COLUMNS[dataset_id],
        str,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_TARGET_COLUMNS "
            f"is not a string."
        )

    if not isinstance(
        TRAINING_PROVENANCE_COLUMNS[dataset_id],
        str,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_PROVENANCE_COLUMNS "
            f"is not a string."
        )

    if not isinstance(
        TRAINING_IDENTIFIER_COLUMNS[dataset_id],
        list,
    ):

        raise RuntimeError(
            f"{dataset_id}: TRAINING_IDENTIFIER_COLUMNS "
            f"is not a list."
        )

print(
    "✓ TRAINING_DATA verified"
)

print(
    "✓ TRAINING_METADATA verified"
)

print(
    "✓ All schema runtime containers verified"
)


# ==================================================================================================
# 20. FINAL RESEARCH-POLICY VERIFICATION
# ==================================================================================================

print()
print("-" * 100)
print("FINAL RESEARCH-POLICY VERIFICATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[
        dataset_id
    ]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    identifiers = TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ]

    if target in features:

        raise RuntimeError(
            f"{dataset_id}: target leakage detected."
        )

    if provenance in features:

        raise RuntimeError(
            f"{dataset_id}: provenance leakage detected."
        )

    if provenance in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance included "
            f"in generative schema."
        )

    if set(
        identifiers
    ).intersection(
        features
    ):

        raise RuntimeError(
            f"{dataset_id}: identifiers included "
            f"in feature schema."
        )

    if set(
        identifiers
    ).intersection(
        generative_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: identifiers included "
            f"in generative schema."
        )

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target missing "
            f"from generative schema."
        )

    if len(features) != (
        len(generative_columns) - 1
    ):

        raise RuntimeError(
            f"{dataset_id}: feature/generative "
            f"schema relationship invalid."
        )

print(
    "✓ Target excluded from feature predictor schema"
)

print(
    "✓ Target retained in generative schema"
)

print(
    "✓ Provenance retained for auditability"
)

print(
    "✓ Explicit identifiers excluded"
)

print(
    "✓ No target/provenance/identifier leakage"
)

print(
    "✓ TRAIN-only policy verified"
)


# ==================================================================================================
# 21. FINAL TRAINING DATA SUMMARY
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 03 — COMPLETION SUMMARY")
print("=" * 100)

total_train_rows = sum(
    len(
        TRAINING_DATA[dataset_id]
    )
    for dataset_id in DATASET_IDS
)

print(
    f"Project root                 : "
    f"{PROJECT_ROOT}"
)

print(
    f"Notebook 02 root             : "
    f"{NB02_ROOT}"
)

print(
    f"Native manifest              : "
    f"{NATIVE_MANIFEST_PATH}"
)

print(
    f"TRAIN datasets loaded        : "
    f"{len(TRAINING_DATA)}"
)

print(
    f"Total TRAIN rows loaded      : "
    f"{total_train_rows:,}"
)

print()
print("Dataset summary:")

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[
        dataset_id
    ]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    provenance_position = (
        list(df.columns).index(
            provenance
        )
    )

    print(
        f"  {dataset_id:<20} "
        f"rows={len(df):,} | "
        f"native={len(df.columns):>2} | "
        f"features={len(features):>2} | "
        f"generative={len(generative_columns):>2} | "
        f"target={target} | "
        f"provenance_pos={provenance_position}"
    )


# ==================================================================================================
# 22. FINAL RESEARCH-POLICY SUMMARY
# ==================================================================================================

print()
print("Research-policy verification:")

print(
    "  ✓ Canonical Notebook 02 native TRAIN data used"
)

print(
    "  ✓ Raw datasets not reloaded"
)

print(
    "  ✓ Notebook 02 preprocessing not refitted"
)

print(
    "  ✓ Encoded/scaled matrices not used"
)

print(
    "  ✓ Only TRAIN split selected"
)

print(
    "  ✓ Validation split excluded"
)

print(
    "  ✓ Test split excluded"
)

print(
    "  ✓ Target retained in generative schema"
)

print(
    "  ✓ Target excluded from feature predictor schema"
)

print(
    "  ✓ Provenance retained for auditability"
)

print(
    "  ✓ Explicit identifiers excluded"
)

print(
    "  ✓ File-size integrity verified"
)

print(
    "  ✓ SHA-256 integrity verified"
)

print(
    "  ✓ Native column order preserved"
)

print(
    "  ✓ TRAIN dataset coverage verified"
)

print(
    "  ✓ Runtime schema containers complete"
)

print(
    "  ✓ Runtime objects verified"
)


# ==================================================================================================
# 23. MEMORY CLEANUP
# ==================================================================================================

gc.collect()


# ==================================================================================================
# FINAL STATUS
# ==================================================================================================

print()
print("=" * 100)
print("SECTION 03 STATUS: PASS")
print("=" * 100)

SECTION 03 — LOAD PROCESSED TRAINING DATA
✓ Google Drive     : /content/drive
✓ MyDrive          : /content/drive/MyDrive
✓ Project root     : /content/drive/MyDrive/SPP_GAN_Research

✓ Notebook 02 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native data root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ Native manifest  : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv

Expected datasets:
  • adult_income         target = income
  • bank_marketing       target = y
  • diabetes_130us       target = readmitted

----------------------------------------------------------------------------------------------------
LOAD NOTEBOOK 02 NATIVE MANIFEST
----------------------------------------------------------------------------------------------------
✓ Native manifest loaded
  Rows    : 9
  Columns : 19
✓ Exact required manifest schema verified
✓ Manifest row count verified: 9
✓ D

In [72]:
# ==================================================================================================
# 4. VALIDATE INPUT SCHEMA
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — VALIDATE INPUT SCHEMA")
print("=" * 100)

INPUT_SCHEMA_SUMMARY = {}

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[dataset_id]

    features = TRAINING_FEATURE_COLUMNS[
        dataset_id
    ]

    generative_columns = TRAINING_GENERATIVE_COLUMNS[
        dataset_id
    ]

    target = TRAINING_TARGET_COLUMNS[
        dataset_id
    ]

    provenance = TRAINING_PROVENANCE_COLUMNS[
        dataset_id
    ]

    identifiers = TRAINING_IDENTIFIER_COLUMNS[
        dataset_id
    ]

    # ----------------------------------------------------------------------------------------------
    # Basic existence
    # ----------------------------------------------------------------------------------------------

    if not set(features).issubset(df.columns):

        raise RuntimeError(
            f"{dataset_id}: preprocessing feature columns missing."
        )

    if not set(generative_columns).issubset(df.columns):

        raise RuntimeError(
            f"{dataset_id}: generative columns missing."
        )

    if target not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column missing."
        )

    if provenance not in df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column missing."
        )

    # ----------------------------------------------------------------------------------------------
    # Frozen Notebook 02 policy
    # ----------------------------------------------------------------------------------------------

    if target in features:

        raise RuntimeError(
            f"{dataset_id}: TARGET LEAKAGE — target appears in features."
        )

    if provenance in features:

        raise RuntimeError(
            f"{dataset_id}: PROVENANCE LEAKAGE."
        )

    identifier_overlap = (
        set(features)
        .intersection(identifiers)
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: IDENTIFIER LEAKAGE:\n"
            f"{sorted(identifier_overlap)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Generative schema
    # ----------------------------------------------------------------------------------------------

    if target not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target is not retained in generative schema."
        )

    if provenance in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance appears in generative schema."
        )

    generative_identifier_overlap = (
        set(generative_columns)
        .intersection(identifiers)
    )

    if generative_identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: identifiers appear in generative schema:\n"
            f"{sorted(generative_identifier_overlap)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Target registry consistency
    # ----------------------------------------------------------------------------------------------
    # TRAINING_TARGET_COLUMNS is the canonical target mapping loaded from
    # Notebook 02 artifacts in Section 03.
    # No undefined TARGET_COLUMNS reference is used here.

    if target != TRAINING_TARGET_COLUMNS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: target registry mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # Store validated schema summary
    # ----------------------------------------------------------------------------------------------

    INPUT_SCHEMA_SUMMARY[dataset_id] = {
        "training_rows": len(df),
        "native_columns": len(df.columns),
        "feature_columns": len(features),
        "generative_columns": len(generative_columns),
        "target_column": target,
        "provenance_column": provenance,
        "identifier_columns": identifiers,
    }

    print(
        f"✓ {dataset_id:<20} | "
        f"features={len(features):>3} | "
        f"generative={len(generative_columns):>3} | "
        f"target={target} | "
        f"target/provenance/ID separation PASS"
    )

print()
print("✓ SECTION 4 — INPUT SCHEMA : PASS")

SECTION 4 — VALIDATE INPUT SCHEMA
✓ adult_income         | features= 14 | generative= 15 | target=income | target/provenance/ID separation PASS
✓ bank_marketing       | features= 16 | generative= 17 | target=y | target/provenance/ID separation PASS
✓ diabetes_130us       | features= 47 | generative= 48 | target=readmitted | target/provenance/ID separation PASS

✓ SECTION 4 — INPUT SCHEMA : PASS


In [73]:
# ==================================================================================================
# 5. DEFINE EXACT STATISTICAL BASELINE METHODS
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — DEFINE EXACT STATISTICAL BASELINE METHODS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Exact Baseline Execution Registry
# --------------------------------------------------------------------------------------------------

BASELINE_METHODS = [
    "independent_marginal",
    "gaussian_copula",
]

# --------------------------------------------------------------------------------------------------
# 2. Exact Baseline Definitions
# --------------------------------------------------------------------------------------------------

BASELINE_DEFINITIONS = {

    "independent_marginal": {

        "name": "Independent Marginal Sampling",

        "family": "Statistical",

        "principle": (
            "Estimate the empirical marginal distribution of each "
            "generative variable independently and sample each "
            "variable independently."
        ),

        "dependency_model": "None",

        "privacy_guarantee": False,
    },

    "gaussian_copula": {

        "name": "Gaussian Copula",

        "family": "Statistical",

        "principle": (
            "Model individual variable distributions together with "
            "their dependence structure through a Gaussian copula."
        ),

        "dependency_model": "Gaussian copula",

        "privacy_guarantee": False,
    },
}

# --------------------------------------------------------------------------------------------------
# 3. Registry Integrity Validation
# --------------------------------------------------------------------------------------------------

if not isinstance(BASELINE_METHODS, list):
    raise RuntimeError(
        "BASELINE_METHODS must be a list."
    )

if len(BASELINE_METHODS) == 0:
    raise RuntimeError(
        "BASELINE_METHODS is empty."
    )

if len(BASELINE_METHODS) != len(set(BASELINE_METHODS)):
    raise RuntimeError(
        "BASELINE_METHODS contains duplicate baseline identifiers."
    )

missing_definitions = [
    baseline_name
    for baseline_name in BASELINE_METHODS
    if baseline_name not in BASELINE_DEFINITIONS
]

if missing_definitions:
    raise RuntimeError(
        f"Missing baseline definitions: {missing_definitions}"
    )

extra_definitions = [
    baseline_name
    for baseline_name in BASELINE_DEFINITIONS
    if baseline_name not in BASELINE_METHODS
]

if extra_definitions:
    raise RuntimeError(
        f"Baseline definitions not present in execution registry: {extra_definitions}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Validate Required Definition Fields
# --------------------------------------------------------------------------------------------------

REQUIRED_BASELINE_FIELDS = {
    "name",
    "family",
    "principle",
    "dependency_model",
    "privacy_guarantee",
}

for baseline_name in BASELINE_METHODS:

    definition = BASELINE_DEFINITIONS[baseline_name]

    missing_fields = REQUIRED_BASELINE_FIELDS.difference(
        definition.keys()
    )

    if missing_fields:
        raise RuntimeError(
            f"{baseline_name}: missing definition fields: "
            f"{sorted(missing_fields)}"
        )

    if definition["family"] != "Statistical":
        raise RuntimeError(
            f"{baseline_name}: baseline family must be 'Statistical'."
        )

    if not isinstance(definition["privacy_guarantee"], bool):
        raise RuntimeError(
            f"{baseline_name}: privacy_guarantee must be boolean."
        )

# --------------------------------------------------------------------------------------------------
# 5. Display Frozen Baseline Registry
# --------------------------------------------------------------------------------------------------

print()
print("FROZEN STATISTICAL BASELINE REGISTRY")
print("-" * 100)

for baseline_index, baseline_name in enumerate(BASELINE_METHODS, start=1):

    definition = BASELINE_DEFINITIONS[baseline_name]

    print()
    print(
        f"{baseline_index}. {baseline_name}"
    )
    print(
        f"   Name         : {definition['name']}"
    )
    print(
        f"   Family       : {definition['family']}"
    )
    print(
        f"   Dependency   : {definition['dependency_model']}"
    )
    print(
        f"   DP guarantee : {definition['privacy_guarantee']}"
    )
    print(
        f"   Principle    : {definition['principle']}"
    )

# --------------------------------------------------------------------------------------------------
# 6. Final Section 5 Integrity Gate
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if BASELINE_METHODS != EXPECTED_BASELINES:
    raise RuntimeError(
        "Baseline registry does not match the frozen research configuration."
    )

for baseline_name in EXPECTED_BASELINES:

    if baseline_name not in BASELINE_DEFINITIONS:
        raise RuntimeError(
            f"Required baseline missing: {baseline_name}"
        )

print()
print("-" * 100)
print(f"✓ Statistical baselines defined : {len(BASELINE_METHODS)}")
print(f"✓ Baseline identifiers           : {BASELINE_METHODS}")
print("✓ Registry integrity             : PASS")
print("✓ Exact baseline registry frozen.")

SECTION 5 — DEFINE EXACT STATISTICAL BASELINE METHODS

FROZEN STATISTICAL BASELINE REGISTRY
----------------------------------------------------------------------------------------------------

1. independent_marginal
   Name         : Independent Marginal Sampling
   Family       : Statistical
   Dependency   : None
   DP guarantee : False
   Principle    : Estimate the empirical marginal distribution of each generative variable independently and sample each variable independently.

2. gaussian_copula
   Name         : Gaussian Copula
   Family       : Statistical
   Dependency   : Gaussian copula
   DP guarantee : False
   Principle    : Model individual variable distributions together with their dependence structure through a Gaussian copula.

----------------------------------------------------------------------------------------------------
✓ Statistical baselines defined : 2
✓ Baseline identifiers           : ['independent_marginal', 'gaussian_copula']
✓ Registry integrity       

In [74]:
# ==================================================================================================
# 6. CONFIGURE STATISTICAL BASELINES
# ==================================================================================================

print("=" * 100)
print("SECTION 6 — CONFIGURE STATISTICAL BASELINES")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Exact Baseline Configuration
# --------------------------------------------------------------------------------------------------

BASELINE_CONFIG = {

    # ==============================================================================================
    # 1A. Independent Marginal Sampling
    # ==============================================================================================

    "independent_marginal": {

        # Sampling mechanism
        "sampling": "empirical_distribution",
        "replacement": True,

        # Synthetic sample size
        "sample_size": "training_rows",

        # Leakage control
        "fit_data": "native_train_only",

        # Schema policy
        "input_schema": "generative_schema",

        # Target / provenance / identifier policy
        "target_policy": "target_retained_not_predictor",
        "provenance_policy": "excluded",
        "identifier_policy": "excluded",

        # Implementation description
        "implementation": "custom_empirical_marginal_sampler",
        "dependency_structure": "independent",
    },


    # ==============================================================================================
    # 1B. Gaussian Copula
    # ==============================================================================================

    "gaussian_copula": {

        # ------------------------------------------------------------------
        # Model family
        # ------------------------------------------------------------------

        "implementation": "sdv_gaussian_copula_synthesizer",
        "copula_family": "gaussian",

        # ------------------------------------------------------------------
        # Marginal / latent distribution policy
        # ------------------------------------------------------------------

        # SDV GaussianCopulaSynthesizer uses a Gaussian latent/dependence
        # representation with configurable marginal distributions.
        "latent_distribution": "standard_normal",
        "default_distribution": "norm",

        # Numerical variables are modeled through the Gaussian-copula
        # synthesizer rather than through an independently sampled
        # empirical marginal distribution.
        "marginal_estimation": "parametric",

        # Dependence structure
        "dependence_model": "gaussian_copula",

        # ------------------------------------------------------------------
        # Numerical policy
        # ------------------------------------------------------------------

        "numerical_policy": {

            # Continuous numerical variables use the configured
            # Gaussian-copula marginal distribution.
            "continuous": "configured_parametric_distribution",

            # Integer-valued numerical variables use the same modeled
            # distribution with SDV rounding enforcement.
            "integer": "modeled_distribution_then_round",

            # Generated numerical values are constrained using
            # training-derived minimum/maximum values where supported.
            "min_max_constraint": True,
        },

        # ------------------------------------------------------------------
        # Categorical policy
        # ------------------------------------------------------------------

        "categorical_policy": {

            # Categorical columns are represented using SDV's metadata-
            # driven categorical transformation.
            "representation": "sdv_categorical_transformer",

            # Category encoding / latent representation is delegated
            # to the SDV/RDT transformation pipeline.
            "latent_mapping": "sdv_rdt_managed",

            # Reconstruction of categorical values is delegated to
            # the SDV/RDT transformation pipeline.
            "decoding": "sdv_rdt_managed",
        },

        # ------------------------------------------------------------------
        # SDV numerical controls
        # ------------------------------------------------------------------

        "enforce_min_max_values": True,
        "enforce_rounding": True,

        "rounding_policy": "integer_numeric_variables_only",

        # ------------------------------------------------------------------
        # Synthetic sample size
        # ------------------------------------------------------------------

        "sample_size": "training_rows",

        # ------------------------------------------------------------------
        # Leakage control
        # ------------------------------------------------------------------

        "fit_data": "native_train_only",

        # ------------------------------------------------------------------
        # Schema policy
        # ------------------------------------------------------------------

        "input_schema": "generative_schema",

        # Target is retained as a generative variable but is not used
        # as a supervised predictor.
        "target_policy": "target_retained_not_predictor",

        # Audit/provenance information is never modeled.
        "provenance_policy": "excluded",

        # Explicit identifiers are never modeled.
        "identifier_policy": "excluded",
    },
}


# --------------------------------------------------------------------------------------------------
# 2. Configuration Registry Integrity
# --------------------------------------------------------------------------------------------------

if not isinstance(BASELINE_CONFIG, dict):
    raise RuntimeError(
        "BASELINE_CONFIG must be a dictionary."
    )

if not BASELINE_CONFIG:
    raise RuntimeError(
        "BASELINE_CONFIG is empty."
    )


missing_configurations = [
    baseline_name
    for baseline_name in BASELINE_METHODS
    if baseline_name not in BASELINE_CONFIG
]

if missing_configurations:
    raise RuntimeError(
        "Missing configurations for baseline(s): "
        f"{missing_configurations}"
    )


unexpected_configurations = [
    baseline_name
    for baseline_name in BASELINE_CONFIG
    if baseline_name not in BASELINE_METHODS
]

if unexpected_configurations:
    raise RuntimeError(
        "Unexpected baseline configuration(s): "
        f"{unexpected_configurations}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Required Configuration Fields
# --------------------------------------------------------------------------------------------------

REQUIRED_CONFIG_FIELDS = {

    "independent_marginal": {
        "sampling",
        "replacement",
        "sample_size",
        "fit_data",
        "input_schema",
        "target_policy",
        "provenance_policy",
        "identifier_policy",
        "implementation",
        "dependency_structure",
    },

    "gaussian_copula": {
        "implementation",
        "copula_family",
        "latent_distribution",
        "default_distribution",
        "marginal_estimation",
        "dependence_model",
        "numerical_policy",
        "categorical_policy",
        "enforce_min_max_values",
        "enforce_rounding",
        "rounding_policy",
        "sample_size",
        "fit_data",
        "input_schema",
        "target_policy",
        "provenance_policy",
        "identifier_policy",
    },
}


for baseline_name in BASELINE_METHODS:

    configuration = BASELINE_CONFIG[baseline_name]

    required_fields = REQUIRED_CONFIG_FIELDS[baseline_name]

    missing_fields = required_fields.difference(
        configuration.keys()
    )

    if missing_fields:
        raise RuntimeError(
            f"{baseline_name}: missing configuration field(s): "
            f"{sorted(missing_fields)}"
        )


# --------------------------------------------------------------------------------------------------
# 4. Independent Marginal Configuration Validation
# --------------------------------------------------------------------------------------------------

independent_config = BASELINE_CONFIG[
    "independent_marginal"
]


if independent_config["sampling"] != "empirical_distribution":
    raise RuntimeError(
        "Independent Marginal Sampling must use "
        "'empirical_distribution'."
    )


if independent_config["replacement"] is not True:
    raise RuntimeError(
        "Independent Marginal Sampling must use sampling with replacement."
    )


if independent_config["sample_size"] != "training_rows":
    raise RuntimeError(
        "Independent Marginal Sampling must generate "
        "the training-row sample size."
    )


if independent_config["fit_data"] != "native_train_only":
    raise RuntimeError(
        "Independent Marginal Sampling must use "
        "native training data only."
    )


if independent_config["input_schema"] != "generative_schema":
    raise RuntimeError(
        "Independent Marginal Sampling must operate on "
        "the generative schema."
    )


if independent_config["target_policy"] != (
    "target_retained_not_predictor"
):
    raise RuntimeError(
        "Independent Marginal target policy is invalid."
    )


if independent_config["provenance_policy"] != "excluded":
    raise RuntimeError(
        "Independent Marginal must exclude provenance."
    )


if independent_config["identifier_policy"] != "excluded":
    raise RuntimeError(
        "Independent Marginal must exclude explicit identifiers."
    )


if independent_config["implementation"] != (
    "custom_empirical_marginal_sampler"
):
    raise RuntimeError(
        "Independent Marginal implementation policy is invalid."
    )


if independent_config["dependency_structure"] != "independent":
    raise RuntimeError(
        "Independent Marginal baseline must have "
        "an independent dependency structure."
    )


# --------------------------------------------------------------------------------------------------
# 5. Gaussian Copula Configuration Validation
# --------------------------------------------------------------------------------------------------

gaussian_config = BASELINE_CONFIG[
    "gaussian_copula"
]


if gaussian_config["implementation"] != (
    "sdv_gaussian_copula_synthesizer"
):
    raise RuntimeError(
        "Gaussian Copula must use the declared "
        "SDV GaussianCopulaSynthesizer implementation."
    )


if gaussian_config["copula_family"] != "gaussian":
    raise RuntimeError(
        "Gaussian Copula family must be 'gaussian'."
    )


if gaussian_config["latent_distribution"] != "standard_normal":
    raise RuntimeError(
        "Gaussian Copula latent distribution must be "
        "'standard_normal'."
    )


if gaussian_config["default_distribution"] != "norm":
    raise RuntimeError(
        "Gaussian Copula default distribution must be 'norm'."
    )


if gaussian_config["marginal_estimation"] != "parametric":
    raise RuntimeError(
        "Gaussian Copula marginal estimation must be "
        "'parametric' for the declared SDV configuration."
    )


if gaussian_config["dependence_model"] != "gaussian_copula":
    raise RuntimeError(
        "Gaussian Copula dependence model must be "
        "'gaussian_copula'."
    )


if gaussian_config["enforce_min_max_values"] is not True:
    raise RuntimeError(
        "Gaussian Copula must enforce training-derived "
        "minimum/maximum constraints."
    )


if gaussian_config["enforce_rounding"] is not True:
    raise RuntimeError(
        "Gaussian Copula must enable rounding."
    )


if gaussian_config["rounding_policy"] != (
    "integer_numeric_variables_only"
):
    raise RuntimeError(
        "Gaussian Copula rounding policy must apply only "
        "to integer-valued numerical variables."
    )


# --------------------------------------------------------------------------------------------------
# 6. Numerical Policy Validation
# --------------------------------------------------------------------------------------------------

numerical_policy = gaussian_config[
    "numerical_policy"
]


required_numerical_policy = {
    "continuous",
    "integer",
    "min_max_constraint",
}


if set(numerical_policy.keys()) != required_numerical_policy:
    raise RuntimeError(
        "Gaussian Copula numerical policy schema mismatch."
    )


if numerical_policy["continuous"] != (
    "configured_parametric_distribution"
):
    raise RuntimeError(
        "Continuous numerical variables must use the "
        "configured parametric distribution policy."
    )


if numerical_policy["integer"] != (
    "modeled_distribution_then_round"
):
    raise RuntimeError(
        "Integer numerical variables must use the "
        "modeled-distribution-then-round policy."
    )


if numerical_policy["min_max_constraint"] is not True:
    raise RuntimeError(
        "Numerical min/max constraints must be enabled."
    )


# --------------------------------------------------------------------------------------------------
# 7. Categorical Policy Validation
# --------------------------------------------------------------------------------------------------

categorical_policy = gaussian_config[
    "categorical_policy"
]


required_categorical_policy = {
    "representation",
    "latent_mapping",
    "decoding",
}


if set(categorical_policy.keys()) != required_categorical_policy:
    raise RuntimeError(
        "Gaussian Copula categorical policy schema mismatch."
    )


if categorical_policy["representation"] != (
    "sdv_categorical_transformer"
):
    raise RuntimeError(
        "Categorical variables must use the declared "
        "SDV categorical transformation policy."
    )


if categorical_policy["latent_mapping"] != (
    "sdv_rdt_managed"
):
    raise RuntimeError(
        "Categorical latent mapping must be delegated "
        "to the SDV/RDT transformation pipeline."
    )


if categorical_policy["decoding"] != (
    "sdv_rdt_managed"
):
    raise RuntimeError(
        "Categorical decoding must be delegated "
        "to the SDV/RDT transformation pipeline."
    )


# --------------------------------------------------------------------------------------------------
# 8. Gaussian Schema / Research-Policy Validation
# --------------------------------------------------------------------------------------------------

if gaussian_config["input_schema"] != "generative_schema":
    raise RuntimeError(
        "Gaussian Copula must operate on the generative schema."
    )


if gaussian_config["target_policy"] != (
    "target_retained_not_predictor"
):
    raise RuntimeError(
        "Gaussian Copula target policy is invalid."
    )


if gaussian_config["provenance_policy"] != "excluded":
    raise RuntimeError(
        "Gaussian Copula must exclude provenance."
    )


if gaussian_config["identifier_policy"] != "excluded":
    raise RuntimeError(
        "Gaussian Copula must exclude explicit identifiers."
    )


if gaussian_config["sample_size"] != "training_rows":
    raise RuntimeError(
        "Gaussian Copula must generate the training-row sample size."
    )


if gaussian_config["fit_data"] != "native_train_only":
    raise RuntimeError(
        "Gaussian Copula must use native training data only."
    )


# --------------------------------------------------------------------------------------------------
# 9. Cross-Baseline Research Policy Validation
# --------------------------------------------------------------------------------------------------

for baseline_name in BASELINE_METHODS:

    configuration = BASELINE_CONFIG[
        baseline_name
    ]

    # Training-only fitting
    if configuration["fit_data"] != "native_train_only":
        raise RuntimeError(
            f"{baseline_name}: research leakage policy violated. "
            "Baseline fitting must use native training data only."
        )

    # Training-sized synthetic sample
    if configuration["sample_size"] != "training_rows":
        raise RuntimeError(
            f"{baseline_name}: sample-size policy violated. "
            "Synthetic sample size must equal training rows."
        )

    # Generative schema
    if configuration["input_schema"] != "generative_schema":
        raise RuntimeError(
            f"{baseline_name}: baseline must operate on "
            "the generative schema."
        )

    # Provenance exclusion
    if configuration["provenance_policy"] != "excluded":
        raise RuntimeError(
            f"{baseline_name}: provenance must be excluded."
        )

    # Identifier exclusion
    if configuration["identifier_policy"] != "excluded":
        raise RuntimeError(
            f"{baseline_name}: explicit identifiers must be excluded."
        )

    # Target policy
    if configuration["target_policy"] != (
        "target_retained_not_predictor"
    ):
        raise RuntimeError(
            f"{baseline_name}: target policy is invalid."
        )


# --------------------------------------------------------------------------------------------------
# 10. Cross-Check Baseline Registry and Configuration
# --------------------------------------------------------------------------------------------------

if set(BASELINE_CONFIG.keys()) != set(BASELINE_METHODS):
    raise RuntimeError(
        "BASELINE_CONFIG identifiers do not exactly match "
        "BASELINE_METHODS."
    )


for baseline_name in BASELINE_METHODS:

    if baseline_name not in BASELINE_DEFINITIONS:
        raise RuntimeError(
            f"{baseline_name}: missing corresponding baseline definition."
        )


# --------------------------------------------------------------------------------------------------
# 11. Research Configuration Snapshot
# --------------------------------------------------------------------------------------------------

BASELINE_CONFIGURATION_SUMMARY = {

    "baseline_count": len(BASELINE_CONFIG),

    "baseline_identifiers": list(BASELINE_METHODS),

    "fit_policy": "native_train_only",

    "sample_size_policy": "training_rows",

    "input_schema_policy": "generative_schema",

    "target_policy": "target_retained_not_predictor",

    "provenance_policy": "excluded",

    "identifier_policy": "excluded",

    "privacy_guarantee": {
        baseline_name: BASELINE_DEFINITIONS[baseline_name][
            "privacy_guarantee"
        ]
        for baseline_name in BASELINE_METHODS
    },

    "implementations": {
        baseline_name: BASELINE_CONFIG[baseline_name][
            "implementation"
        ]
        for baseline_name in BASELINE_METHODS
    },
}


# --------------------------------------------------------------------------------------------------
# 12. Display Frozen Configuration
# --------------------------------------------------------------------------------------------------

print()
print("FROZEN STATISTICAL BASELINE CONFIGURATION")
print("-" * 100)

print(
    json.dumps(
        BASELINE_CONFIG,
        indent=2,
        sort_keys=False,
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Final Configuration Integrity Gate
# --------------------------------------------------------------------------------------------------

if len(BASELINE_CONFIG) != 2:
    raise RuntimeError(
        "Expected exactly 2 statistical baseline configurations."
    )


if BASELINE_METHODS != [
    "independent_marginal",
    "gaussian_copula",
]:
    raise RuntimeError(
        "Baseline execution registry does not match "
        "the frozen research configuration."
    )


if set(BASELINE_CONFIG.keys()) != {
    "independent_marginal",
    "gaussian_copula",
}:
    raise RuntimeError(
        "Baseline configuration registry does not match "
        "the frozen research configuration."
    )


print()
print("-" * 100)
print(
    f"✓ Statistical baselines configured : "
    f"{len(BASELINE_CONFIG)}"
)

print(
    f"✓ Configured identifiers            : "
    f"{list(BASELINE_CONFIG.keys())}"
)

print(
    "✓ Configuration completeness        : PASS"
)

print(
    "✓ Method-specific validation        : PASS"
)

print(
    "✓ Training-only policy              : PASS"
)

print(
    "✓ Training-sized sampling policy    : PASS"
)

print(
    "✓ Mixed-type schema policy          : PASS"
)

print(
    "✓ Target/provenance/ID policy       : PASS"
)

print(
    "✓ Implementation policy             : PASS"
)

print(
    "✓ Configuration integrity            : PASS"
)

print(
    "✓ Baseline configuration frozen."
)

SECTION 6 — CONFIGURE STATISTICAL BASELINES

FROZEN STATISTICAL BASELINE CONFIGURATION
----------------------------------------------------------------------------------------------------
{
  "independent_marginal": {
    "sampling": "empirical_distribution",
    "replacement": true,
    "sample_size": "training_rows",
    "fit_data": "native_train_only",
    "input_schema": "generative_schema",
    "target_policy": "target_retained_not_predictor",
    "provenance_policy": "excluded",
    "identifier_policy": "excluded",
    "implementation": "custom_empirical_marginal_sampler",
    "dependency_structure": "independent"
  },
  "gaussian_copula": {
    "implementation": "sdv_gaussian_copula_synthesizer",
    "copula_family": "gaussian",
    "latent_distribution": "standard_normal",
    "default_distribution": "norm",
    "marginal_estimation": "parametric",
    "dependence_model": "gaussian_copula",
    "numerical_policy": {
      "continuous": "configured_parametric_distribution",
    

In [75]:
# ==================================================================================================
# 7. SET REPRODUCIBLE SEEDS
# ==================================================================================================

print("=" * 100)
print("SECTION 7 — SET REPRODUCIBLE SEEDS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Required Imports
# --------------------------------------------------------------------------------------------------

import os
import random
import numpy as np


# --------------------------------------------------------------------------------------------------
# 2. Validate Notebook 00 Master Seed
# --------------------------------------------------------------------------------------------------

if "MASTER_SEED" not in globals():
    raise RuntimeError(
        "MASTER_SEED is not available. "
        "Load the frozen Notebook 00 configuration before Section 7."
    )

if not isinstance(MASTER_SEED, (int, np.integer)):
    raise RuntimeError(
        "MASTER_SEED must be an integer."
    )

MASTER_SEED = int(MASTER_SEED)

if MASTER_SEED != 2025:
    raise RuntimeError(
        f"Unexpected MASTER_SEED={MASTER_SEED}. "
        "Frozen Notebook 00 master seed is 2025."
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate Notebook 00 Repetition Configuration
# --------------------------------------------------------------------------------------------------

if "REPETITIONS" not in globals():
    raise RuntimeError(
        "REPETITIONS is not available. "
        "Load the frozen Notebook 00 experiment configuration before Section 7."
    )

if not isinstance(REPETITIONS, (int, np.integer)):
    raise RuntimeError(
        "REPETITIONS must be an integer."
    )

REPETITIONS = int(REPETITIONS)

if REPETITIONS != 5:
    raise RuntimeError(
        f"Unexpected REPETITIONS={REPETITIONS}. "
        "Frozen Notebook 00 configuration requires 5 repetitions."
    )


# --------------------------------------------------------------------------------------------------
# 4. Validate Authoritative Notebook 00 Repetition Seed Registry
# --------------------------------------------------------------------------------------------------

if "REPETITION_SEEDS" not in globals():
    raise RuntimeError(
        "REPETITION_SEEDS is not available. "
        "Load the frozen Notebook 00 repetition seed registry before Section 7."
    )

if not isinstance(REPETITION_SEEDS, dict):
    raise RuntimeError(
        "REPETITION_SEEDS must be a dictionary matching the "
        "frozen Notebook 00 seed registry."
    )


# --------------------------------------------------------------------------------------------------
# 5. Validate Exact Notebook 00 Repetition Seed Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_REPETITION_SEEDS = {
    "1": 3026,
    "2": 3027,
    "3": 3028,
    "4": 3029,
    "5": 3030,
}

if REPETITION_SEEDS != EXPECTED_REPETITION_SEEDS:
    raise RuntimeError(
        "Notebook 00 repetition seed registry does not match "
        "the frozen research configuration. "
        f"Expected {EXPECTED_REPETITION_SEEDS}, "
        f"received {REPETITION_SEEDS}."
    )


if len(REPETITION_SEEDS) != REPETITIONS:
    raise RuntimeError(
        "Number of repetition seeds does not equal REPETITIONS."
    )


if len(set(REPETITION_SEEDS.values())) != REPETITIONS:
    raise RuntimeError(
        "Notebook 00 repetition seeds are not unique."
    )


# --------------------------------------------------------------------------------------------------
# 6. Normalize Repetition Seed Registry for Execution
# --------------------------------------------------------------------------------------------------

REPETITION_SEED_REGISTRY = {
    int(repetition): int(seed)
    for repetition, seed in REPETITION_SEEDS.items()
}


if sorted(REPETITION_SEED_REGISTRY.keys()) != list(
    range(1, REPETITIONS + 1)
):
    raise RuntimeError(
        "Repetition identifiers must be consecutive integers "
        "from 1 through REPETITIONS."
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate Dataset and Baseline Registries
# --------------------------------------------------------------------------------------------------

if "DATASET_IDS" not in globals():
    raise RuntimeError(
        "DATASET_IDS is not available."
    )

if "BASELINE_METHODS" not in globals():
    raise RuntimeError(
        "BASELINE_METHODS is not available."
    )

if not isinstance(DATASET_IDS, (list, tuple)):
    raise RuntimeError(
        "DATASET_IDS must be a list or tuple."
    )

if not isinstance(BASELINE_METHODS, (list, tuple)):
    raise RuntimeError(
        "BASELINE_METHODS must be a list or tuple."
    )

DATASET_IDS = list(DATASET_IDS)
BASELINE_METHODS = list(BASELINE_METHODS)

if len(DATASET_IDS) == 0:
    raise RuntimeError(
        "DATASET_IDS is empty."
    )

if len(BASELINE_METHODS) == 0:
    raise RuntimeError(
        "BASELINE_METHODS is empty."
    )

if len(set(DATASET_IDS)) != len(DATASET_IDS):
    raise RuntimeError(
        "DATASET_IDS contains duplicate dataset identifiers."
    )

if len(set(BASELINE_METHODS)) != len(BASELINE_METHODS):
    raise RuntimeError(
        "BASELINE_METHODS contains duplicate baseline identifiers."
    )


# --------------------------------------------------------------------------------------------------
# 8. Validate Frozen Dataset and Baseline Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

EXPECTED_BASELINE_METHODS = [
    "independent_marginal",
    "gaussian_copula",
]

if DATASET_IDS != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "DATASET_IDS does not match the frozen Notebook 00 configuration."
    )

if BASELINE_METHODS != EXPECTED_BASELINE_METHODS:
    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 "
        "statistical baseline configuration."
    )


# --------------------------------------------------------------------------------------------------
# 9. Global Reproducibility Function
# --------------------------------------------------------------------------------------------------

def seed_everything(seed):
    """
    Initialize Python and NumPy reproducibility controls.

    PYTHONHASHSEED is recorded at process level for reproducibility
    provenance. In an already-running Python process, changing the
    environment variable does not retroactively change hash randomization.
    """

    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)


# --------------------------------------------------------------------------------------------------
# 10. Deterministic Experimental Run Seed Policy
# --------------------------------------------------------------------------------------------------

def get_experiment_seed(
    repetition_seed,
    dataset_index,
    baseline_index,
):
    """
    Derive a deterministic seed for one experimental unit.

    Experimental unit:
        dataset × baseline × repetition

    The authoritative repetition seed comes directly from Notebook 00.
    Dataset and baseline offsets provide deterministic separation between
    experimental units within each repetition.
    """

    repetition_seed = int(repetition_seed)
    dataset_index = int(dataset_index)
    baseline_index = int(baseline_index)

    if dataset_index < 0:
        raise ValueError(
            "dataset_index must be non-negative."
        )

    if baseline_index < 0:
        raise ValueError(
            "baseline_index must be non-negative."
        )

    return (
        repetition_seed
        + ((dataset_index + 1) * 100)
        + ((baseline_index + 1) * 10)
    )


# --------------------------------------------------------------------------------------------------
# 11. Build Experimental Seed Registry
# --------------------------------------------------------------------------------------------------

EXPERIMENT_SEED_REGISTRY = {}

for repetition in range(1, REPETITIONS + 1):

    repetition_seed = REPETITION_SEED_REGISTRY[
        repetition
    ]

    EXPERIMENT_SEED_REGISTRY[
        repetition
    ] = {}

    for dataset_index, dataset_id in enumerate(
        DATASET_IDS
    ):

        EXPERIMENT_SEED_REGISTRY[
            repetition
        ][dataset_id] = {}

        for baseline_index, baseline_name in enumerate(
            BASELINE_METHODS
        ):

            experiment_seed = get_experiment_seed(
                repetition_seed=repetition_seed,
                dataset_index=dataset_index,
                baseline_index=baseline_index,
            )

            EXPERIMENT_SEED_REGISTRY[
                repetition
            ][dataset_id][baseline_name] = experiment_seed


# --------------------------------------------------------------------------------------------------
# 12. Flatten Experimental Seed Registry
# --------------------------------------------------------------------------------------------------

EXPERIMENT_SEED_RECORDS = []

for repetition in range(1, REPETITIONS + 1):

    repetition_seed = REPETITION_SEED_REGISTRY[
        repetition
    ]

    for dataset_index, dataset_id in enumerate(
        DATASET_IDS
    ):

        for baseline_index, baseline_name in enumerate(
            BASELINE_METHODS
        ):

            experiment_seed = EXPERIMENT_SEED_REGISTRY[
                repetition
            ][dataset_id][baseline_name]

            EXPERIMENT_SEED_RECORDS.append(
                {
                    "repetition": repetition,
                    "repetition_seed": repetition_seed,
                    "dataset_id": dataset_id,
                    "dataset_index": dataset_index,
                    "baseline_name": baseline_name,
                    "baseline_index": baseline_index,
                    "experiment_seed": experiment_seed,
                }
            )


# --------------------------------------------------------------------------------------------------
# 13. Validate Expected Experimental Unit Count
# --------------------------------------------------------------------------------------------------

EXPECTED_EXPERIMENT_COUNT = (
    REPETITIONS
    * len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

if EXPECTED_EXPERIMENT_COUNT != 30:
    raise RuntimeError(
        "Frozen Notebook 04 statistical baseline design must contain "
        "exactly 30 experimental units: "
        "5 repetitions × 3 datasets × 2 baselines."
    )

if len(EXPERIMENT_SEED_RECORDS) != EXPECTED_EXPERIMENT_COUNT:
    raise RuntimeError(
        "Unexpected number of experimental seed records. "
        f"Expected {EXPECTED_EXPERIMENT_COUNT}, "
        f"received {len(EXPERIMENT_SEED_RECORDS)}."
    )


# --------------------------------------------------------------------------------------------------
# 14. Verify Experimental Seed Uniqueness
# --------------------------------------------------------------------------------------------------

ALL_EXPERIMENT_SEEDS = [
    record["experiment_seed"]
    for record in EXPERIMENT_SEED_RECORDS
]

if len(ALL_EXPERIMENT_SEEDS) != len(
    set(ALL_EXPERIMENT_SEEDS)
):
    raise RuntimeError(
        "Experimental seed collision detected."
    )


# --------------------------------------------------------------------------------------------------
# 15. Verify Repetition Seed Traceability
# --------------------------------------------------------------------------------------------------

for record in EXPERIMENT_SEED_RECORDS:

    if (
        record["repetition_seed"]
        != REPETITION_SEED_REGISTRY[
            record["repetition"]
        ]
    ):
        raise RuntimeError(
            "Experimental seed record is not traceable to "
            "the authoritative Notebook 00 repetition seed."
        )


# --------------------------------------------------------------------------------------------------
# 16. Initialize Global Reproducibility
# --------------------------------------------------------------------------------------------------

seed_everything(
    MASTER_SEED
)


# --------------------------------------------------------------------------------------------------
# 17. Display Reproducibility Configuration
# --------------------------------------------------------------------------------------------------

print()
print("REPRODUCIBILITY CONFIGURATION")
print("-" * 100)

print(
    f"✓ MASTER_SEED                     : {MASTER_SEED}"
)

print(
    f"✓ REPETITIONS                     : {REPETITIONS}"
)

print(
    f"✓ REPETITION_SEEDS                : "
    f"{REPETITION_SEEDS}"
)

print(
    "✓ Python random seeded"
)

print(
    "✓ NumPy random seeded"
)

print(
    "✓ PYTHONHASHSEED configured"
)


# --------------------------------------------------------------------------------------------------
# 18. Display Experimental Seed Registry
# --------------------------------------------------------------------------------------------------

print()
print("EXPERIMENTAL SEED REGISTRY")
print("-" * 100)

for repetition in range(1, REPETITIONS + 1):

    print()
    print(
        f"Repetition {repetition}"
    )

    print(
        f"  Repetition seed : "
        f"{REPETITION_SEED_REGISTRY[repetition]}"
    )

    for dataset_id in DATASET_IDS:

        print(
            f"  {dataset_id}"
        )

        for baseline_name in BASELINE_METHODS:

            experiment_seed = (
                EXPERIMENT_SEED_REGISTRY[
                    repetition
                ][dataset_id][baseline_name]
            )

            print(
                f"    {baseline_name:<25} : "
                f"{experiment_seed}"
            )


# --------------------------------------------------------------------------------------------------
# 19. Final Integrity Gate
# --------------------------------------------------------------------------------------------------

if MASTER_SEED != 2025:
    raise RuntimeError(
        "Master seed integrity check failed."
    )

if REPETITIONS != 5:
    raise RuntimeError(
        "Repetition count integrity check failed."
    )

if REPETITION_SEEDS != EXPECTED_REPETITION_SEEDS:
    raise RuntimeError(
        "Notebook 00 repetition seed registry integrity check failed."
    )

if len(REPETITION_SEED_REGISTRY) != 5:
    raise RuntimeError(
        "Expected exactly 5 repetition seeds."
    )

if len(EXPERIMENT_SEED_RECORDS) != 30:
    raise RuntimeError(
        "Expected exactly 30 experimental seed records."
    )

if len(set(ALL_EXPERIMENT_SEEDS)) != 30:
    raise RuntimeError(
        "Expected exactly 30 unique experimental seeds."
    )


print()
print("-" * 100)

print(
    f"✓ Master seed validated             : {MASTER_SEED}"
)

print(
    f"✓ Repetitions covered              : {REPETITIONS}"
)

print(
    f"✓ Datasets covered                 : {len(DATASET_IDS)}"
)

print(
    f"✓ Baselines covered                : {len(BASELINE_METHODS)}"
)

print(
    f"✓ Expected experimental runs       : "
    f"{EXPECTED_EXPERIMENT_COUNT}"
)

print(
    f"✓ Unique experimental seeds        : "
    f"{len(ALL_EXPERIMENT_SEEDS)}"
)

print(
    "✓ Notebook 00 repetition seeds     : PASS"
)

print(
    "✓ Python/NumPy reproducibility     : PASS"
)

print(
    "✓ Dataset/baseline/repetition seed policy : PASS"
)

print(
    "✓ Seed collision check             : PASS"
)

print(
    "✓ Experimental seed traceability   : PASS"
)

print(
    "✓ Reproducibility configuration frozen."
)

SECTION 7 — SET REPRODUCIBLE SEEDS

REPRODUCIBILITY CONFIGURATION
----------------------------------------------------------------------------------------------------
✓ MASTER_SEED                     : 2025
✓ REPETITIONS                     : 5
✓ REPETITION_SEEDS                : {'1': 3026, '2': 3027, '3': 3028, '4': 3029, '5': 3030}
✓ Python random seeded
✓ NumPy random seeded
✓ PYTHONHASHSEED configured

EXPERIMENTAL SEED REGISTRY
----------------------------------------------------------------------------------------------------

Repetition 1
  Repetition seed : 3026
  adult_income
    independent_marginal      : 3136
    gaussian_copula           : 3146
  bank_marketing
    independent_marginal      : 3236
    gaussian_copula           : 3246
  diabetes_130us
    independent_marginal      : 3336
    gaussian_copula           : 3346

Repetition 2
  Repetition seed : 3027
  adult_income
    independent_marginal      : 3137
    gaussian_copula           : 3147
  bank_marketing
    i

In [76]:
# ==================================================================================================
# NOTEBOOK 04 — ENVIRONMENT SETUP FOR STATISTICAL BASELINES
# ==================================================================================================

import sys
import subprocess

print("=" * 100)
print("NOTEBOOK 04 — SDV ENVIRONMENT CHECK")
print("=" * 100)

print(f"Python version : {sys.version}")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sdv",
    ]
)

print()
print("✓ SDV installation completed.")
print("⚠ Restart the Colab runtime if pip reports dependency conflicts.")

NOTEBOOK 04 — SDV ENVIRONMENT CHECK
Python version : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

✓ SDV installation completed.
⚠ Restart the Colab runtime if pip reports dependency conflicts.


In [77]:
# ==================================================================================================
# 8. FIT BASELINE DISTRIBUTIONS
# ==================================================================================================

print("=" * 100)
print("SECTION 8 — FIT BASELINE DISTRIBUTIONS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Required Imports
# --------------------------------------------------------------------------------------------------

import gc
import time
import os
import sys
import hashlib
import pickle
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. SDV Imports
# --------------------------------------------------------------------------------------------------

try:
    import sdv
    from sdv.metadata import Metadata
    from sdv.single_table import GaussianCopulaSynthesizer

except ImportError as exc:
    raise RuntimeError(
        "SDV is required for Section 8 but could not be imported."
    ) from exc


print(
    f"✓ SDV version    : {getattr(sdv, '__version__', 'unknown')}"
)

print(
    f"✓ Python version : {sys.version.split()[0]}"
)


# --------------------------------------------------------------------------------------------------
# 3. Validate SDV Version
# --------------------------------------------------------------------------------------------------

EXPECTED_SDV_VERSION = "1.38.3"

if getattr(sdv, "__version__", None) != EXPECTED_SDV_VERSION:
    raise RuntimeError(
        f"Unexpected SDV version: {getattr(sdv, '__version__', 'unknown')}. "
        f"Expected SDV version: {EXPECTED_SDV_VERSION}."
    )


# --------------------------------------------------------------------------------------------------
# 4. Validate Required Section 07 Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_07_OBJECTS = [
    "REPETITIONS",
    "REPETITION_SEED_REGISTRY",
    "EXPERIMENT_SEED_REGISTRY",
    "EXPERIMENT_SEED_RECORDS",
]

for object_name in REQUIRED_SECTION_07_OBJECTS:

    if object_name not in globals():
        raise RuntimeError(
            f"{object_name} is not available. "
            "Run the final Section 07 before Section 08."
        )


# --------------------------------------------------------------------------------------------------
# 5. Validate Frozen Experiment Design
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

EXPECTED_BASELINE_METHODS = [
    "independent_marginal",
    "gaussian_copula",
]

EXPECTED_REPETITION_SEEDS = {
    1: 3026,
    2: 3027,
    3: 3028,
    4: 3029,
    5: 3030,
}

EXPECTED_EXPERIMENT_COUNT = 30


if REPETITIONS != 5:
    raise RuntimeError(
        f"Expected 5 repetitions, received {REPETITIONS}."
    )

if DATASET_IDS != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "DATASET_IDS does not match the frozen Notebook 04 configuration."
    )

if BASELINE_METHODS != EXPECTED_BASELINE_METHODS:
    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 configuration."
    )

if REPETITION_SEED_REGISTRY != EXPECTED_REPETITION_SEEDS:
    raise RuntimeError(
        "REPETITION_SEED_REGISTRY does not match the frozen "
        "Notebook 00 repetition seed configuration."
    )

if len(EXPERIMENT_SEED_RECORDS) != EXPECTED_EXPERIMENT_COUNT:
    raise RuntimeError(
        "Section 07 must contain exactly 30 experimental seed records."
    )


# --------------------------------------------------------------------------------------------------
# 6. Validate Section 03 Training Containers
# --------------------------------------------------------------------------------------------------

REQUIRED_TRAINING_CONTAINERS = [
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
]

for object_name in REQUIRED_TRAINING_CONTAINERS:

    if object_name not in globals():
        raise RuntimeError(
            f"{object_name} is not available. "
            "Run the preceding Notebook 04 sections first."
        )


# --------------------------------------------------------------------------------------------------
# 7. Validate Section 06 Configuration
# --------------------------------------------------------------------------------------------------

if "BASELINE_CONFIG" not in globals():
    raise RuntimeError(
        "BASELINE_CONFIG is not available. "
        "Run Section 06 before Section 08."
    )

if set(BASELINE_CONFIG.keys()) != set(BASELINE_METHODS):
    raise RuntimeError(
        "BASELINE_CONFIG does not exactly match BASELINE_METHODS."
    )


# --------------------------------------------------------------------------------------------------
# 8. Canonical Notebook 04 Artifact Roots
# --------------------------------------------------------------------------------------------------

NB04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

NB04_BASELINE_ROOT = (
    NB04_ROOT
    / "baselines"
)

NB04_MODEL_ROOT = (
    NB04_BASELINE_ROOT
    / "models"
)

NB04_METADATA_ROOT = (
    NB04_BASELINE_ROOT
    / "metadata"
)

NB04_MANIFEST_ROOT = (
    NB04_BASELINE_ROOT
    / "manifests"
)


NB04_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB04_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB04_MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    f"✓ Baseline artifact root : {NB04_BASELINE_ROOT}"
)

print(
    f"✓ Model artifact root    : {NB04_MODEL_ROOT}"
)

print(
    f"✓ Metadata artifact root : {NB04_METADATA_ROOT}"
)

print(
    f"✓ Manifest artifact root : {NB04_MANIFEST_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 9. SHA-256 Helper
# --------------------------------------------------------------------------------------------------

def sha256_file(path):

    path = Path(path)

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 10. Safe Model Persistence
# --------------------------------------------------------------------------------------------------

def persist_model_artifact(
    model,
    artifact_path,
):

    artifact_path = Path(
        artifact_path
    )

    artifact_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = artifact_path.with_suffix(
        artifact_path.suffix + ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "wb",
    ) as handle:

        pickle.dump(
            model,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    if not temporary_path.exists():
        raise RuntimeError(
            f"Temporary model artifact was not created: "
            f"{temporary_path}"
        )

    if temporary_path.stat().st_size <= 0:
        raise RuntimeError(
            f"Temporary model artifact is empty: "
            f"{temporary_path}"
        )

    os.replace(
        temporary_path,
        artifact_path,
    )

    if not artifact_path.exists():
        raise RuntimeError(
            f"Model artifact was not persisted: "
            f"{artifact_path}"
        )

    if artifact_path.stat().st_size <= 0:
        raise RuntimeError(
            f"Model artifact is empty: "
            f"{artifact_path}"
        )

    return {
        "artifact_path": str(
            artifact_path
        ),
        "artifact_size_bytes": int(
            artifact_path.stat().st_size
        ),
        "artifact_sha256": sha256_file(
            artifact_path
        ),
    }


# --------------------------------------------------------------------------------------------------
# 11. Independent Marginal Fitter
# --------------------------------------------------------------------------------------------------

def fit_independent_marginal(
    dataframe,
    columns,
):

    fitted = {}

    for column in columns:

        series = dataframe[
            column
        ]

        values = (
            series.astype(object)
            .where(
                series.notna(),
                "__NB04_MISSING__",
            )
        )

        frequencies = (
            values.value_counts(
                normalize=True,
                dropna=False,
            )
        )

        fitted[column] = {
            "values": frequencies.index.tolist(),
            "probabilities": (
                frequencies.values
                .astype(float)
                .tolist()
            ),
            "dtype": str(
                series.dtype
            ),
            "n_unique": int(
                series.nunique(
                    dropna=False
                )
            ),
            "missing_count": int(
                series.isna().sum()
            ),
            "missing_rate": float(
                series.isna().mean()
            ),
        }

    return fitted


# --------------------------------------------------------------------------------------------------
# 12. Canonical Gaussian Copula Metadata Handler
# --------------------------------------------------------------------------------------------------

GAUSSIAN_METADATA_CACHE = {}


def get_or_create_gaussian_metadata(
    dataset_id,
    model_input,
    generative_columns,
):

    metadata_path = (
        NB04_METADATA_ROOT
        / f"{dataset_id}_gaussian_copula_metadata.json"
    )


    # ----------------------------------------------------------------------------------------------
    # Reuse existing canonical metadata artifact.
    # ----------------------------------------------------------------------------------------------

    if metadata_path.exists():

        metadata = Metadata.load_from_json(
            filepath=str(
                metadata_path
            )
        )

        metadata.validate()

        metadata_dict = metadata.to_dict()

        tables = metadata_dict.get(
            "tables",
            {},
        )

        if dataset_id not in tables:
            raise RuntimeError(
                f"{dataset_id}: existing metadata does not contain "
                f"table '{dataset_id}'."
            )

        detected_columns = list(
            tables[
                dataset_id
            ].get(
                "columns",
                {},
            ).keys()
        )

        if detected_columns != list(
            generative_columns
        ):
            raise RuntimeError(
                f"{dataset_id}: existing metadata schema mismatch.\n"
                f"Expected: {list(generative_columns)}\n"
                f"Found   : {detected_columns}"
            )

        metadata_hash = sha256_file(
            metadata_path
        )

        record = {
            "dataset_id": dataset_id,
            "baseline": "gaussian_copula",
            "metadata_path": str(
                metadata_path
            ),
            "metadata_size_bytes": int(
                metadata_path.stat().st_size
            ),
            "metadata_sha256": metadata_hash,
            "generative_columns": list(
                generative_columns
            ),
            "target_column": TRAINING_TARGET_COLUMNS[
                dataset_id
            ],
            "fit_data": "native_train_only",
        }

        GAUSSIAN_METADATA_CACHE[
            dataset_id
        ] = record

        return (
            metadata,
            record,
        )


    # ----------------------------------------------------------------------------------------------
    # Create canonical metadata only when it does not already exist.
    # ----------------------------------------------------------------------------------------------

    metadata = Metadata.detect_from_dataframe(
        data=model_input,
        table_name=dataset_id,
        infer_keys=None,
    )

    metadata.validate()


    # ----------------------------------------------------------------------------------------------
    # Validate detected metadata schema.
    # ----------------------------------------------------------------------------------------------

    metadata_dict = metadata.to_dict()

    tables = metadata_dict.get(
        "tables",
        {},
    )

    if dataset_id not in tables:
        raise RuntimeError(
            f"{dataset_id}: detected metadata does not contain "
            f"table '{dataset_id}'."
        )

    detected_columns = list(
        tables[
            dataset_id
        ].get(
            "columns",
            {},
        ).keys()
    )

    if detected_columns != list(
        generative_columns
    ):
        raise RuntimeError(
            f"{dataset_id}: detected metadata schema mismatch.\n"
            f"Expected: {list(generative_columns)}\n"
            f"Found   : {detected_columns}"
        )


    # ----------------------------------------------------------------------------------------------
    # Save only because the canonical file does not exist.
    # ----------------------------------------------------------------------------------------------

    metadata.save_to_json(
        filepath=str(
            metadata_path
        )
    )

    if not metadata_path.exists():
        raise RuntimeError(
            f"{dataset_id}: metadata artifact was not created."
        )

    if metadata_path.stat().st_size <= 0:
        raise RuntimeError(
            f"{dataset_id}: metadata artifact is empty."
        )

    metadata_hash = sha256_file(
        metadata_path
    )

    record = {
        "dataset_id": dataset_id,
        "baseline": "gaussian_copula",
        "metadata_path": str(
            metadata_path
        ),
        "metadata_size_bytes": int(
            metadata_path.stat().st_size
        ),
        "metadata_sha256": metadata_hash,
        "generative_columns": list(
            generative_columns
        ),
        "target_column": TRAINING_TARGET_COLUMNS[
            dataset_id
        ],
        "fit_data": "native_train_only",
    }

    GAUSSIAN_METADATA_CACHE[
        dataset_id
    ] = record

    return (
        metadata,
        record,
    )


# --------------------------------------------------------------------------------------------------
# 13. Initialize Runtime Containers
# --------------------------------------------------------------------------------------------------

FITTED_BASELINE_MODELS = {}

FIT_RUNTIME_RECORDS = []

BASELINE_METADATA_ARTIFACTS = []

BASELINE_FIT_MANIFEST_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 14. Fit All 30 Experimental Runs
# --------------------------------------------------------------------------------------------------

for repetition in range(
    1,
    REPETITIONS + 1,
):

    repetition_seed = (
        REPETITION_SEED_REGISTRY[
            repetition
        ]
    )

    print()
    print("=" * 100)
    print(
        f"REPETITION {repetition} / {REPETITIONS}"
    )
    print(
        f"Authoritative repetition seed : "
        f"{repetition_seed}"
    )
    print("=" * 100)


    for dataset_index, dataset_id in enumerate(
        DATASET_IDS
    ):

        print()
        print("-" * 100)
        print(
            f"FITTING — {dataset_id} | REPETITION {repetition}"
        )
        print("-" * 100)


        # ------------------------------------------------------------------------------------------
        # Retrieve canonical training data.
        # ------------------------------------------------------------------------------------------

        if dataset_id not in TRAINING_DATA:
            raise RuntimeError(
                f"{dataset_id}: training data is unavailable."
            )

        train_df = TRAINING_DATA[
            dataset_id
        ]

        generative_columns = list(
            TRAINING_GENERATIVE_COLUMNS[
                dataset_id
            ]
        )

        target_column = (
            TRAINING_TARGET_COLUMNS[
                dataset_id
            ]
        )

        provenance_column = (
            TRAINING_PROVENANCE_COLUMNS[
                dataset_id
            ]
        )

        identifier_columns = list(
            TRAINING_IDENTIFIER_COLUMNS[
                dataset_id
            ]
        )


        # ------------------------------------------------------------------------------------------
        # Validate training dataset.
        # ------------------------------------------------------------------------------------------

        if not isinstance(
            train_df,
            pd.DataFrame,
        ):
            raise RuntimeError(
                f"{dataset_id}: TRAINING_DATA must be a pandas DataFrame."
            )

        if train_df.empty:
            raise RuntimeError(
                f"{dataset_id}: training dataset is empty."
            )

        if not set(
            generative_columns
        ).issubset(
            train_df.columns
        ):
            raise RuntimeError(
                f"{dataset_id}: generative columns are not fully "
                "present in training data."
            )

        if target_column not in generative_columns:
            raise RuntimeError(
                f"{dataset_id}: target column is not retained "
                "in the generative schema."
            )

        if provenance_column in generative_columns:
            raise RuntimeError(
                f"{dataset_id}: provenance column appears "
                "in the generative schema."
            )

        if set(
            identifier_columns
        ).intersection(
            generative_columns
        ):
            raise RuntimeError(
                f"{dataset_id}: identifier columns appear "
                "in the generative schema."
            )


        # ------------------------------------------------------------------------------------------
        # Construct exact generative-schema input.
        # ------------------------------------------------------------------------------------------

        model_input = (
            train_df[
                generative_columns
            ]
            .copy()
        )

        if list(
            model_input.columns
        ) != generative_columns:
            raise RuntimeError(
                f"{dataset_id}: generative column order mismatch."
            )


        for baseline_index, baseline_name in enumerate(
            BASELINE_METHODS
        ):

            # --------------------------------------------------------------------------------------
            # Run-local object initialization.
            # --------------------------------------------------------------------------------------

            model = None
            synthesizer = None
            metadata = None
            metadata_path = None
            metadata_hash = None


            # --------------------------------------------------------------------------------------
            # Retrieve authoritative experiment seed.
            # --------------------------------------------------------------------------------------

            experiment_seed = (
                EXPERIMENT_SEED_REGISTRY[
                    repetition
                ][
                    dataset_id
                ][
                    baseline_name
                ]
            )


            # --------------------------------------------------------------------------------------
            # Verify seed against Section 07 flattened registry.
            # --------------------------------------------------------------------------------------

            matching_records = [
                record
                for record in EXPERIMENT_SEED_RECORDS
                if (
                    record["repetition"] == repetition
                    and record["dataset_id"] == dataset_id
                    and record["baseline_name"] == baseline_name
                )
            ]

            if len(matching_records) != 1:
                raise RuntimeError(
                    f"Expected exactly one seed record for "
                    f"{repetition}/{dataset_id}/{baseline_name}; "
                    f"found {len(matching_records)}."
                )

            if matching_records[0][
                "experiment_seed"
            ] != experiment_seed:
                raise RuntimeError(
                    f"Experiment seed mismatch for "
                    f"{repetition}/{dataset_id}/{baseline_name}."
                )


            # --------------------------------------------------------------------------------------
            # Initialize deterministic random state.
            # --------------------------------------------------------------------------------------

            random.seed(
                experiment_seed
            )

            np.random.seed(
                experiment_seed
            )

            os.environ[
                "PYTHONHASHSEED"
            ] = str(
                experiment_seed
            )

            start_time = time.perf_counter()


            # ======================================================================================
            # INDEPENDENT MARGINAL
            # ======================================================================================

            if baseline_name == (
                "independent_marginal"
            ):

                model = fit_independent_marginal(
                    dataframe=model_input,
                    columns=generative_columns,
                )

                model_path = (
                    NB04_MODEL_ROOT
                    / f"repetition_{repetition}"
                    / dataset_id
                    / "independent_marginal.pkl"
                )

                artifact_info = persist_model_artifact(
                    model=model,
                    artifact_path=model_path,
                )


                FITTED_BASELINE_MODELS.setdefault(
                    repetition,
                    {},
                )

                FITTED_BASELINE_MODELS[
                    repetition
                ].setdefault(
                    dataset_id,
                    {},
                )

                FITTED_BASELINE_MODELS[
                    repetition
                ][
                    dataset_id
                ][
                    baseline_name
                ] = {
                    "model_artifact_path": artifact_info[
                        "artifact_path"
                    ],
                    "model_artifact_size_bytes": artifact_info[
                        "artifact_size_bytes"
                    ],
                    "model_artifact_sha256": artifact_info[
                        "artifact_sha256"
                    ],
                    "experiment_seed": int(
                        experiment_seed
                    ),
                    "repetition_seed": int(
                        repetition_seed
                    ),
                }


            # ======================================================================================
            # GAUSSIAN COPULA
            # ======================================================================================

            elif baseline_name == (
                "gaussian_copula"
            ):

                gaussian_config = BASELINE_CONFIG[
                    "gaussian_copula"
                ]

                if gaussian_config[
                    "implementation"
                ] != (
                    "sdv_gaussian_copula_synthesizer"
                ):
                    raise RuntimeError(
                        "Gaussian Copula implementation does not match "
                        "the frozen Section 06 configuration."
                    )

                if gaussian_config[
                    "copula_family"
                ] != "gaussian":
                    raise RuntimeError(
                        "Gaussian Copula family does not match "
                        "the frozen Section 06 configuration."
                    )


                # ----------------------------------------------------------------------------------
                # Obtain existing or newly created canonical metadata.
                # ----------------------------------------------------------------------------------

                (
                    metadata,
                    metadata_record,
                ) = get_or_create_gaussian_metadata(
                    dataset_id=dataset_id,
                    model_input=model_input,
                    generative_columns=generative_columns,
                )

                metadata_path = metadata_record[
                    "metadata_path"
                ]

                metadata_hash = metadata_record[
                    "metadata_sha256"
                ]


                # ----------------------------------------------------------------------------------
                # Register canonical metadata exactly once per dataset.
                # ----------------------------------------------------------------------------------

                if not any(
                    artifact["dataset_id"] == dataset_id
                    for artifact in BASELINE_METADATA_ARTIFACTS
                ):
                    BASELINE_METADATA_ARTIFACTS.append(
                        metadata_record
                    )


                # ----------------------------------------------------------------------------------
                # Validate persisted metadata one final time.
                # ----------------------------------------------------------------------------------

                metadata.validate()

                metadata_dict = metadata.to_dict()

                tables = metadata_dict.get(
                    "tables",
                    {},
                )

                if dataset_id not in tables:
                    raise RuntimeError(
                        f"{dataset_id}: persisted metadata table missing."
                    )

                detected_columns = list(
                    tables[
                        dataset_id
                    ].get(
                        "columns",
                        {},
                    ).keys()
                )

                if detected_columns != generative_columns:
                    raise RuntimeError(
                        f"{dataset_id}: persisted metadata schema mismatch."
                    )


                # ----------------------------------------------------------------------------------
                # Construct SDV Gaussian Copula synthesizer.
                # ----------------------------------------------------------------------------------

                synthesizer = (
                    GaussianCopulaSynthesizer(
                        metadata,
                        enforce_min_max_values=(
                            gaussian_config[
                                "enforce_min_max_values"
                            ]
                        ),
                        enforce_rounding=(
                            gaussian_config[
                                "enforce_rounding"
                            ]
                        ),
                        default_distribution=(
                            gaussian_config[
                                "default_distribution"
                            ]
                        ),
                    )
                )


                # ----------------------------------------------------------------------------------
                # Fit exclusively on native TRAIN generative data.
                # ----------------------------------------------------------------------------------

                synthesizer.fit(
                    model_input
                )


                # ----------------------------------------------------------------------------------
                # Persist repetition-specific fitted model.
                # ----------------------------------------------------------------------------------

                model_path = (
                    NB04_MODEL_ROOT
                    / f"repetition_{repetition}"
                    / dataset_id
                    / "gaussian_copula.pkl"
                )

                artifact_info = persist_model_artifact(
                    model=synthesizer,
                    artifact_path=model_path,
                )


                FITTED_BASELINE_MODELS.setdefault(
                    repetition,
                    {},
                )

                FITTED_BASELINE_MODELS[
                    repetition
                ].setdefault(
                    dataset_id,
                    {},
                )

                FITTED_BASELINE_MODELS[
                    repetition
                ][
                    dataset_id
                ][
                    baseline_name
                ] = {
                    "model_artifact_path": artifact_info[
                        "artifact_path"
                    ],
                    "model_artifact_size_bytes": artifact_info[
                        "artifact_size_bytes"
                    ],
                    "model_artifact_sha256": artifact_info[
                        "artifact_sha256"
                    ],
                    "metadata_artifact_path": metadata_path,
                    "metadata_artifact_sha256": metadata_hash,
                    "experiment_seed": int(
                        experiment_seed
                    ),
                    "repetition_seed": int(
                        repetition_seed
                    ),
                }


            else:

                raise RuntimeError(
                    f"Unknown baseline method: {baseline_name}"
                )


            # --------------------------------------------------------------------------------------
            # Runtime record.
            # --------------------------------------------------------------------------------------

            elapsed = (
                time.perf_counter()
                - start_time
            )

            fit_record = {
                "repetition": int(
                    repetition
                ),
                "repetition_seed": int(
                    repetition_seed
                ),
                "experiment_seed": int(
                    experiment_seed
                ),
                "dataset_id": dataset_id,
                "dataset_index": int(
                    dataset_index
                ),
                "baseline": baseline_name,
                "baseline_index": int(
                    baseline_index
                ),
                "training_rows": int(
                    len(train_df)
                ),
                "training_columns": int(
                    len(generative_columns)
                ),
                "generative_columns": int(
                    len(generative_columns)
                ),
                "target_column": target_column,
                "fit_data": "native_train_only",
                "input_schema": "generative_schema",
                "provenance_policy": "excluded",
                "identifier_policy": "excluded",
                "fit_runtime_seconds": float(
                    elapsed
                ),
                "model_artifact_path": artifact_info[
                    "artifact_path"
                ],
                "model_artifact_size_bytes": artifact_info[
                    "artifact_size_bytes"
                ],
                "model_artifact_sha256": artifact_info[
                    "artifact_sha256"
                ],
                "metadata_artifact_path": metadata_path,
                "metadata_artifact_sha256": metadata_hash,
                "status": "PASS",
            }

            FIT_RUNTIME_RECORDS.append(
                fit_record
            )

            BASELINE_FIT_MANIFEST_RECORDS.append(
                fit_record.copy()
            )


            print(
                f"✓ repetition={repetition} | "
                f"{dataset_id:<18} | "
                f"{baseline_name:<24} | "
                f"seed={experiment_seed} | "
                f"rows={len(train_df):,} | "
                f"columns={len(generative_columns)} | "
                f"fit_runtime={elapsed:.3f}s"
            )

            if baseline_name == (
                "gaussian_copula"
            ):

                print(
                    f"  Metadata : {metadata_path}"
                )

                print(
                    f"  Metadata SHA256 : {metadata_hash}"
                )

            print(
                f"  Model    : {artifact_info['artifact_path']}"
            )

            print(
                f"  Model SHA256 : {artifact_info['artifact_sha256']}"
            )


            # --------------------------------------------------------------------------------------
            # Release run-local fitted objects.
            # --------------------------------------------------------------------------------------

            if baseline_name == "independent_marginal":

                model = None

            elif baseline_name == "gaussian_copula":

                synthesizer = None
                metadata = None

            gc.collect()


        # ------------------------------------------------------------------------------------------
        # Release dataset-level input copy.
        # ------------------------------------------------------------------------------------------

        del model_input

        gc.collect()


# --------------------------------------------------------------------------------------------------
# 15. Validate Total Fitted Runs
# --------------------------------------------------------------------------------------------------

if len(
    FIT_RUNTIME_RECORDS
) != EXPECTED_EXPERIMENT_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_EXPERIMENT_COUNT} fitted runs, "
        f"found {len(FIT_RUNTIME_RECORDS)}."
    )

if len(
    BASELINE_FIT_MANIFEST_RECORDS
) != EXPECTED_EXPERIMENT_COUNT:
    raise RuntimeError(
        "Baseline fit manifest must contain exactly 30 records."
    )


# --------------------------------------------------------------------------------------------------
# 16. Validate Run Uniqueness
# --------------------------------------------------------------------------------------------------

RUN_KEYS = [
    (
        record["repetition"],
        record["dataset_id"],
        record["baseline"],
    )
    for record in FIT_RUNTIME_RECORDS
]

if len(RUN_KEYS) != len(
    set(RUN_KEYS)
):
    raise RuntimeError(
        "Duplicate dataset × baseline × repetition run detected."
    )


# --------------------------------------------------------------------------------------------------
# 17. Validate Exact Repetition Coverage
# --------------------------------------------------------------------------------------------------

observed_repetitions = sorted(
    set(
        record["repetition"]
        for record in FIT_RUNTIME_RECORDS
    )
)

if observed_repetitions != [
    1,
    2,
    3,
    4,
    5,
]:
    raise RuntimeError(
        "All five repetitions must be represented exactly."
    )


# --------------------------------------------------------------------------------------------------
# 18. Validate Exact Dataset × Baseline Coverage
# --------------------------------------------------------------------------------------------------

for repetition in range(
    1,
    6,
):

    repetition_records = [
        record
        for record in FIT_RUNTIME_RECORDS
        if record["repetition"] == repetition
    ]

    if len(repetition_records) != 6:
        raise RuntimeError(
            f"Repetition {repetition} must contain exactly 6 runs."
        )

    observed_pairs = {
        (
            record["dataset_id"],
            record["baseline"],
        )
        for record in repetition_records
    }

    expected_pairs = {
        (
            dataset_id,
            baseline_name,
        )
        for dataset_id in DATASET_IDS
        for baseline_name in BASELINE_METHODS
    }

    if observed_pairs != expected_pairs:
        raise RuntimeError(
            f"Repetition {repetition}: dataset/baseline coverage mismatch."
        )


# --------------------------------------------------------------------------------------------------
# 19. Validate Experiment Seed Uniqueness and Traceability
# --------------------------------------------------------------------------------------------------

experiment_seeds = [
    record["experiment_seed"]
    for record in FIT_RUNTIME_RECORDS
]

if len(experiment_seeds) != len(
    set(experiment_seeds)
):
    raise RuntimeError(
        "Experiment seed collision detected."
    )

if len(experiment_seeds) != 30:
    raise RuntimeError(
        "Exactly 30 unique experiment seeds are required."
    )


for record in FIT_RUNTIME_RECORDS:

    expected_seed = (
        EXPERIMENT_SEED_REGISTRY[
            record["repetition"]
        ][
            record["dataset_id"]
        ][
            record["baseline"]
        ]
    )

    if record["experiment_seed"] != expected_seed:
        raise RuntimeError(
            f"Experiment seed traceability failure for "
            f"{record['repetition']}/"
            f"{record['dataset_id']}/"
            f"{record['baseline']}."
        )


# --------------------------------------------------------------------------------------------------
# 20. Validate Training-Only Policy
# --------------------------------------------------------------------------------------------------

for record in FIT_RUNTIME_RECORDS:

    if record["fit_data"] != "native_train_only":
        raise RuntimeError(
            "Training-only policy violation detected."
        )

    if record["input_schema"] != "generative_schema":
        raise RuntimeError(
            "Generative schema policy violation detected."
        )

    if record["provenance_policy"] != "excluded":
        raise RuntimeError(
            "Provenance exclusion policy violation detected."
        )

    if record["identifier_policy"] != "excluded":
        raise RuntimeError(
            "Identifier exclusion policy violation detected."
        )


# --------------------------------------------------------------------------------------------------
# 21. Validate Model Artifacts
# --------------------------------------------------------------------------------------------------

for record in FIT_RUNTIME_RECORDS:

    model_path = Path(
        record["model_artifact_path"]
    )

    if not model_path.exists():
        raise RuntimeError(
            f"Missing model artifact: {model_path}"
        )

    if model_path.stat().st_size <= 0:
        raise RuntimeError(
            f"Empty model artifact: {model_path}"
        )

    actual_hash = sha256_file(
        model_path
    )

    if actual_hash != record[
        "model_artifact_sha256"
    ]:
        raise RuntimeError(
            f"Model SHA-256 mismatch: {model_path}"
        )


# --------------------------------------------------------------------------------------------------
# 22. Validate Canonical Metadata Artifacts
# --------------------------------------------------------------------------------------------------

if len(
    BASELINE_METADATA_ARTIFACTS
) != len(DATASET_IDS):
    raise RuntimeError(
        f"Expected {len(DATASET_IDS)} canonical metadata artifacts, "
        f"found {len(BASELINE_METADATA_ARTIFACTS)}."
    )


for metadata_record in BASELINE_METADATA_ARTIFACTS:

    metadata_path = Path(
        metadata_record["metadata_path"]
    )

    if not metadata_path.exists():
        raise RuntimeError(
            f"Missing metadata artifact: {metadata_path}"
        )

    if metadata_path.stat().st_size <= 0:
        raise RuntimeError(
            f"Empty metadata artifact: {metadata_path}"
        )

    actual_hash = sha256_file(
        metadata_path
    )

    if actual_hash != metadata_record[
        "metadata_sha256"
    ]:
        raise RuntimeError(
            f"Metadata SHA-256 mismatch: {metadata_path}"
        )

    metadata = Metadata.load_from_json(
        filepath=str(
            metadata_path
        )
    )

    metadata.validate()

    metadata_dict = metadata.to_dict()

    dataset_id = metadata_record[
        "dataset_id"
    ]

    tables = metadata_dict.get(
        "tables",
        {},
    )

    if dataset_id not in tables:
        raise RuntimeError(
            f"{dataset_id}: metadata table missing."
        )

    detected_columns = list(
        tables[
            dataset_id
        ].get(
            "columns",
            {},
        ).keys()
    )

    expected_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    if detected_columns != expected_columns:
        raise RuntimeError(
            f"{dataset_id}: metadata schema integrity failure."
        )

    del metadata
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 23. Persist Baseline Fit Manifest
# --------------------------------------------------------------------------------------------------

BASELINE_FIT_MANIFEST_DF = pd.DataFrame(
    BASELINE_FIT_MANIFEST_RECORDS
)

BASELINE_FIT_MANIFEST_PATH = (
    NB04_MANIFEST_ROOT
    / "baseline_fit_manifest.csv"
)

BASELINE_FIT_MANIFEST_DF.to_csv(
    BASELINE_FIT_MANIFEST_PATH,
    index=False,
)

if not BASELINE_FIT_MANIFEST_PATH.exists():
    raise RuntimeError(
        "Baseline fit manifest was not persisted."
    )

BASELINE_FIT_MANIFEST_SHA256 = sha256_file(
    BASELINE_FIT_MANIFEST_PATH
)


# --------------------------------------------------------------------------------------------------
# 24. Persist Runtime Manifest
# --------------------------------------------------------------------------------------------------

FIT_RUNTIME_DF = pd.DataFrame(
    FIT_RUNTIME_RECORDS
)

FIT_RUNTIME_PATH = (
    NB04_MANIFEST_ROOT
    / "baseline_fit_runtime.csv"
)

FIT_RUNTIME_DF.to_csv(
    FIT_RUNTIME_PATH,
    index=False,
)

if not FIT_RUNTIME_PATH.exists():
    raise RuntimeError(
        "Baseline runtime manifest was not persisted."
    )

FIT_RUNTIME_SHA256 = sha256_file(
    FIT_RUNTIME_PATH
)


# --------------------------------------------------------------------------------------------------
# 25. Persist Canonical Metadata Registry
# --------------------------------------------------------------------------------------------------

BASELINE_METADATA_DF = pd.DataFrame(
    BASELINE_METADATA_ARTIFACTS
)

BASELINE_METADATA_REGISTRY_PATH = (
    NB04_MANIFEST_ROOT
    / "gaussian_copula_metadata_registry.csv"
)

BASELINE_METADATA_DF.to_csv(
    BASELINE_METADATA_REGISTRY_PATH,
    index=False,
)

if not BASELINE_METADATA_REGISTRY_PATH.exists():
    raise RuntimeError(
        "Gaussian Copula metadata registry was not persisted."
    )

BASELINE_METADATA_REGISTRY_SHA256 = sha256_file(
    BASELINE_METADATA_REGISTRY_PATH
)


# --------------------------------------------------------------------------------------------------
# 26. Final FITTED_BASELINE_MODELS Registry Validation
# --------------------------------------------------------------------------------------------------

if len(
    FITTED_BASELINE_MODELS
) != REPETITIONS:
    raise RuntimeError(
        "FITTED_BASELINE_MODELS repetition coverage is incomplete."
    )


for repetition in range(
    1,
    REPETITIONS + 1,
):

    if repetition not in FITTED_BASELINE_MODELS:
        raise RuntimeError(
            f"Missing repetition {repetition}."
        )

    if set(
        FITTED_BASELINE_MODELS[
            repetition
        ].keys()
    ) != set(
        DATASET_IDS
    ):
        raise RuntimeError(
            f"Repetition {repetition}: dataset registry mismatch."
        )

    for dataset_id in DATASET_IDS:

        if set(
            FITTED_BASELINE_MODELS[
                repetition
            ][
                dataset_id
            ].keys()
        ) != set(
            BASELINE_METHODS
        ):
            raise RuntimeError(
                f"Repetition {repetition}, dataset {dataset_id}: "
                "baseline registry mismatch."
            )


# --------------------------------------------------------------------------------------------------
# 27. Final Section 08 Integrity Gate
# --------------------------------------------------------------------------------------------------

if len(
    FIT_RUNTIME_RECORDS
) != 30:
    raise RuntimeError(
        "Final fitting count is not 30."
    )

if len(
    BASELINE_FIT_MANIFEST_RECORDS
) != 30:
    raise RuntimeError(
        "Final fit manifest count is not 30."
    )

if len(
    BASELINE_METADATA_ARTIFACTS
) != 3:
    raise RuntimeError(
        "Final canonical metadata count is not 3."
    )

if len(
    set(
        experiment_seeds
    )
) != 30:
    raise RuntimeError(
        "Final experiment seed uniqueness check failed."
    )


# --------------------------------------------------------------------------------------------------
# 28. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 8 — FITTING SUMMARY")
print("=" * 100)

print(
    f"✓ SDV version                  : "
    f"{getattr(sdv, '__version__', 'unknown')}"
)

print(
    f"✓ Python version               : "
    f"{sys.version.split()[0]}"
)

print(
    f"✓ Datasets fitted              : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset        : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                  : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected experimental runs   : "
    f"{EXPECTED_EXPERIMENT_COUNT}"
)

print(
    f"✓ Total fitted runs            : "
    f"{len(FIT_RUNTIME_RECORDS)}"
)

print(
    f"✓ Runtime records              : "
    f"{len(FIT_RUNTIME_RECORDS)}"
)

print(
    f"✓ Model artifacts              : "
    f"{len(BASELINE_FIT_MANIFEST_RECORDS)}"
)

print(
    f"✓ Canonical metadata artifacts : "
    f"{len(BASELINE_METADATA_ARTIFACTS)}"
)

print(
    "✓ Native TRAIN-only fitting    : PASS"
)

print(
    "✓ Generative schema validation : PASS"
)

print(
    "✓ 5-repetition coverage        : PASS"
)

print(
    "✓ 30-run experimental coverage : PASS"
)

print(
    "✓ Experiment seed traceability : PASS"
)

print(
    "✓ Experiment seed uniqueness   : PASS"
)

print(
    "✓ Model artifact persistence   : PASS"
)

print(
    "✓ Model artifact integrity     : PASS"
)

print(
    "✓ Metadata persistence         : PASS"
)

print(
    "✓ Metadata integrity           : PASS"
)

print(
    "✓ Fit registry integrity       : PASS"
)

print(
    f"✓ Fit manifest                 : "
    f"{BASELINE_FIT_MANIFEST_PATH}"
)

print(
    f"✓ Fit manifest SHA256          : "
    f"{BASELINE_FIT_MANIFEST_SHA256}"
)

print(
    f"✓ Runtime manifest             : "
    f"{FIT_RUNTIME_PATH}"
)

print(
    f"✓ Runtime manifest SHA256      : "
    f"{FIT_RUNTIME_SHA256}"
)

print(
    f"✓ Metadata registry            : "
    f"{BASELINE_METADATA_REGISTRY_PATH}"
)

print(
    f"✓ Metadata registry SHA256     : "
    f"{BASELINE_METADATA_REGISTRY_SHA256}"
)

print()
print(
    "✓ Section 8 fitting completed."
)

SECTION 8 — FIT BASELINE DISTRIBUTIONS
✓ SDV version    : 1.38.3
✓ Python version : 3.13.15
✓ Baseline artifact root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines
✓ Model artifact root    : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/models
✓ Metadata artifact root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/metadata
✓ Manifest artifact root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/manifests

REPETITION 1 / 5
Authoritative repetition seed : 3026

----------------------------------------------------------------------------------------------------
FITTING — adult_income | REPETITION 1
----------------------------------------------------------------------------------------------------
✓ repetition=1 | adult_income       | independent_marginal     | seed=3136 | rows=34,189 | columns=15 | fit_runtime=0.305s
  Model    : /content/drive/MyDrive/SPP_GAN_Res

In [78]:
# ==================================================================================================
# SECTION 9 — GENERATE SYNTHETIC DATA
# RAM-SAFE + RESTART-SAFE + 30-RUN EXPERIMENTAL DESIGN
# ==================================================================================================

print("=" * 100)
print("SECTION 9 — GENERATE SYNTHETIC DATA")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import os
import gc
import time
import json
import hashlib
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "DATASET_IDS",
    "BASELINE_METHODS",
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
    "REPETITIONS",
    "REPETITION_SEED_REGISTRY",
    "EXPERIMENT_SEED_REGISTRY",
]

_missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if _missing_runtime_objects:
    raise RuntimeError(
        "Section 09 is missing required runtime objects: "
        f"{_missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if list(BASELINE_METHODS) != EXPECTED_BASELINES:
    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 "
        f"baseline registry.\n"
        f"Expected: {EXPECTED_BASELINES}\n"
        f"Found   : {list(BASELINE_METHODS)}"
    )


if int(REPETITIONS) != 5:
    raise RuntimeError(
        f"Notebook 04 requires 5 repetitions, found {REPETITIONS}."
    )


EXPECTED_RUN_COUNT = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)


if EXPECTED_RUN_COUNT != 30:
    raise RuntimeError(
        "The frozen Notebook 04 design must contain exactly 30 "
        f"experimental generation runs. Found {EXPECTED_RUN_COUNT}."
    )


# --------------------------------------------------------------------------------------------------
# 4. Authoritative Notebook 04 Artifact Roots
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NOTEBOOK_04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

BASELINE_ROOT = (
    NOTEBOOK_04_ROOT
    / "baselines"
)

MODEL_ROOT = (
    BASELINE_ROOT
    / "models"
)

SYNTHETIC_ROOT = (
    BASELINE_ROOT
    / "synthetic"
)

MANIFEST_ROOT = (
    BASELINE_ROOT
    / "manifests"
)


# --------------------------------------------------------------------------------------------------
# 5. Verify Artifact Roots
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("ARTIFACT ROOT VALIDATION")
print("-" * 100)

print(
    f"✓ Project root          : {PROJECT_ROOT}"
)

print(
    f"✓ Section 08 model root : {MODEL_ROOT}"
)

print(
    f"✓ Synthetic output root : {SYNTHETIC_ROOT}"
)

print(
    f"✓ Manifest root         : {MANIFEST_ROOT}"
)


if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Project root does not exist: {PROJECT_ROOT}"
)


if not MODEL_ROOT.exists():
    raise RuntimeError(
        f"Section 08 model root does not exist: {MODEL_ROOT}"
)


SYNTHETIC_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 6. Normalize and Validate Repetition Seeds
# --------------------------------------------------------------------------------------------------

EXPECTED_REPETITION_SEEDS = {
    1: 3026,
    2: 3027,
    3: 3028,
    4: 3029,
    5: 3030,
}


NORMALIZED_REPETITION_SEEDS = {
    int(key): int(value)
    for key, value in REPETITION_SEED_REGISTRY.items()
}


if NORMALIZED_REPETITION_SEEDS != EXPECTED_REPETITION_SEEDS:
    raise RuntimeError(
        "Repetition seed registry does not match the frozen "
        "Notebook 00 configuration.\n"
        f"Expected: {EXPECTED_REPETITION_SEEDS}\n"
        f"Found   : {NORMALIZED_REPETITION_SEEDS}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Normalize Experiment Seed Registry
# --------------------------------------------------------------------------------------------------

def normalize_experiment_seed_registry(registry):
    """
    Normalize the authoritative Section 07 experiment seed registry.

    Expected structure:

        {
            repetition: {
                dataset_id: {
                    baseline_name: seed
                }
            }
        }
    """

    normalized = {}

    for repetition_key, dataset_records in registry.items():

        repetition = int(
            repetition_key
        )

        normalized[
            repetition
        ] = {}

        for dataset_id, baseline_records in dataset_records.items():

            normalized[
                repetition
            ][
                dataset_id
            ] = {}

            for baseline_name, seed in baseline_records.items():

                normalized[
                    repetition
                ][
                    dataset_id
                ][
                    baseline_name
                ] = int(seed)

    return normalized


EXPERIMENT_SEEDS = normalize_experiment_seed_registry(
    EXPERIMENT_SEED_REGISTRY
)


# --------------------------------------------------------------------------------------------------
# 8. Validate Complete Experiment Seed Coverage
# --------------------------------------------------------------------------------------------------

for repetition in range(
    1,
    int(REPETITIONS) + 1,
):

    if repetition not in EXPERIMENT_SEEDS:
        raise RuntimeError(
            f"Missing experiment seed registry for repetition "
            f"{repetition}."
        )

    for dataset_id in DATASET_IDS:

        if dataset_id not in EXPERIMENT_SEEDS[
            repetition
        ]:
            raise RuntimeError(
                f"Missing experiment seeds for "
                f"repetition={repetition}, "
                f"dataset={dataset_id}."
            )

        for baseline_name in BASELINE_METHODS:

            if baseline_name not in EXPERIMENT_SEEDS[
                repetition
            ][
                dataset_id
            ]:
                raise RuntimeError(
                    f"Missing experiment seed for "
                    f"repetition={repetition}, "
                    f"dataset={dataset_id}, "
                    f"baseline={baseline_name}."
                )


# --------------------------------------------------------------------------------------------------
# 9. Seed Utility
# --------------------------------------------------------------------------------------------------

def seed_everything(seed):
    """
    Set the runtime seeds used by Section 09.
    """

    seed = int(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )


# --------------------------------------------------------------------------------------------------
# 10. SHA256 Utility
# --------------------------------------------------------------------------------------------------

def sha256_file(filepath):
    """
    Compute SHA256 for a persisted file.
    """

    filepath = Path(
        filepath
    )

    digest = hashlib.sha256()

    with filepath.open(
        "rb"
    ) as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 11. Resolve Section 08 Model Artifact
# --------------------------------------------------------------------------------------------------

def get_model_artifact_path(
    repetition,
    dataset_id,
    baseline_name,
):
    """
    Resolve the exact persisted Section 08 model artifact.
    """

    return (
        MODEL_ROOT
        / f"repetition_{int(repetition)}"
        / dataset_id
        / f"{baseline_name}.pkl"
    )


# --------------------------------------------------------------------------------------------------
# 12. Resolve Synthetic Output Artifact
# --------------------------------------------------------------------------------------------------

def get_synthetic_artifact_path(
    repetition,
    dataset_id,
    baseline_name,
):
    """
    Resolve the canonical Section 09 synthetic-data output path.
    """

    return (
        SYNTHETIC_ROOT
        / f"repetition_{int(repetition)}"
        / dataset_id
        / f"{baseline_name}.csv"
    )


# --------------------------------------------------------------------------------------------------
# 13. Load One Persisted Section 08 Model
# --------------------------------------------------------------------------------------------------

def load_persisted_model(
    repetition,
    dataset_id,
    baseline_name,
):
    """
    Load exactly one persisted Section 08 model.

    The model is intentionally loaded one run at a time
    to minimize RAM usage.
    """

    model_path = get_model_artifact_path(
        repetition=repetition,
        dataset_id=dataset_id,
        baseline_name=baseline_name,
    )

    if not model_path.exists():
        raise FileNotFoundError(
            "Required Section 08 model artifact is missing:\n"
            f"{model_path}"
        )

    with model_path.open(
        "rb"
    ) as handle:

        model = pickle.load(
            handle
        )

    return (
        model,
        model_path,
    )


# --------------------------------------------------------------------------------------------------
# 14. Generate Independent Marginal Synthetic Data
# --------------------------------------------------------------------------------------------------

def generate_independent_marginal(
    model,
    generative_columns,
    training_rows,
    seed,
    dataset_id,
):
    """
    Generate synthetic data from the persisted empirical
    marginal distributions.
    """

    if not isinstance(
        model,
        dict,
    ):
        raise RuntimeError(
            f"{dataset_id}: Independent Marginal model must "
            "be a dictionary."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    synthetic_columns = {}

    for column in generative_columns:

        if column not in model:
            raise RuntimeError(
                f"{dataset_id}: Independent Marginal model "
                f"is missing column '{column}'."
            )

        column_record = model[
            column
        ]

        if not isinstance(
            column_record,
            dict,
        ):
            raise RuntimeError(
                f"{dataset_id}: invalid model record for "
                f"column '{column}'."
            )

        if "values" not in column_record:
            raise RuntimeError(
                f"{dataset_id}: model has no values for "
                f"column '{column}'."
            )

        if "probabilities" not in column_record:
            raise RuntimeError(
                f"{dataset_id}: model has no probabilities for "
                f"column '{column}'."
            )

        values = np.asarray(
            column_record[
                "values"
            ],
            dtype=object,
        )

        probabilities = np.asarray(
            column_record[
                "probabilities"
            ],
            dtype=float,
        )

        if len(values) == 0:
            raise RuntimeError(
                f"{dataset_id}: no empirical values available "
                f"for column '{column}'."
            )

        if len(values) != len(probabilities):
            raise RuntimeError(
                f"{dataset_id}: value/probability length mismatch "
                f"for column '{column}'."
            )

        if not np.all(
            np.isfinite(
                probabilities
            )
        ):
            raise RuntimeError(
                f"{dataset_id}: non-finite probabilities detected "
                f"for column '{column}'."
            )

        probability_sum = float(
            probabilities.sum()
        )

        if probability_sum <= 0:
            raise RuntimeError(
                f"{dataset_id}: invalid probability distribution "
                f"for column '{column}'."
            )

        probabilities = (
            probabilities
            / probability_sum
        )

        sampled = rng.choice(
            values,
            size=int(
                training_rows
            ),
            replace=True,
            p=probabilities,
        )

        sampled_series = pd.Series(
            sampled,
            dtype=object,
        )

        sampled_series = (
            sampled_series
            .replace(
                "__NB04_MISSING__",
                np.nan,
            )
        )

        synthetic_columns[
            column
        ] = sampled_series

    synthetic_df = pd.DataFrame(
        synthetic_columns,
        columns=list(
            generative_columns
        ),
    )

    return synthetic_df


# --------------------------------------------------------------------------------------------------
# 15. Generate Gaussian Copula Synthetic Data
# --------------------------------------------------------------------------------------------------

def generate_gaussian_copula(
    synthesizer,
    generative_columns,
    training_rows,
    dataset_id,
):
    """
    Generate synthetic data from the persisted SDV
    Gaussian Copula synthesizer.
    """

    if synthesizer is None:
        raise RuntimeError(
            f"{dataset_id}: Gaussian Copula synthesizer is None."
        )

    try:

        synthetic_df = synthesizer.sample(
            num_rows=int(
                training_rows
            ),
            randomize_samples=True,
        )

    except TypeError:

        synthetic_df = synthesizer.sample(
            num_rows=int(
                training_rows
            )
        )

    if not isinstance(
        synthetic_df,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{dataset_id}: Gaussian Copula generated output "
            "is not a pandas DataFrame."
        )

    missing_columns = [
        column
        for column in generative_columns
        if column not in synthetic_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"{dataset_id}: Gaussian Copula generated data "
            f"is missing column(s): {missing_columns}"
        )

    synthetic_df = (
        synthetic_df[
            list(
                generative_columns
            )
        ]
        .copy()
    )

    return synthetic_df


# --------------------------------------------------------------------------------------------------
# 16. Validate Existing Synthetic Artifact
# --------------------------------------------------------------------------------------------------

def validate_existing_synthetic_artifact(
    output_path,
    dataset_id,
    baseline_name,
    expected_rows,
    expected_columns,
):
    """
    Validate an existing CSV without loading the complete file into RAM.

    Used to make Section 09 restart-safe.
    """

    output_path = Path(
        output_path
    )

    if not output_path.exists():
        return False

    try:

        # ------------------------------------------------------------------------------------------
        # 16.1 Validate header
        # ------------------------------------------------------------------------------------------

        header_df = pd.read_csv(
            output_path,
            nrows=0,
        )

        if list(
            header_df.columns
        ) != list(
            expected_columns
        ):
            return False

        del header_df

        # ------------------------------------------------------------------------------------------
        # 16.2 Validate row count without loading the entire CSV
        # ------------------------------------------------------------------------------------------

        row_count = 0

        with output_path.open(
            "rb"
        ) as handle:

            for chunk in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):

                row_count += chunk.count(
                    b"\n"
                )

        # One header line is not a data row.
        data_rows = max(
            0,
            row_count - 1,
        )

        if data_rows != int(
            expected_rows
        ):
            return False

        # ------------------------------------------------------------------------------------------
        # 16.3 Validate SHA256
        # ------------------------------------------------------------------------------------------

        _ = sha256_file(
            output_path
        )

        return True

    except Exception:

        return False


# --------------------------------------------------------------------------------------------------
# 17. Runtime Containers
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
# No generated DataFrames are retained here.
#
# Only small metadata/runtime records remain in RAM.
#

GENERATION_RUNTIME_RECORDS = []

GENERATION_MANIFEST_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 18. Generate One Experimental Unit at a Time
# --------------------------------------------------------------------------------------------------

for repetition in range(
    1,
    int(REPETITIONS) + 1,
):

    repetition_seed = (
        EXPECTED_REPETITION_SEEDS[
            repetition
        ]
    )

    print()
    print("=" * 100)
    print(
        f"REPETITION {repetition} / {REPETITIONS}"
    )
    print(
        f"Authoritative repetition seed : {repetition_seed}"
    )
    print("=" * 100)

    for dataset_index, dataset_id in enumerate(
        DATASET_IDS
    ):

        # ------------------------------------------------------------------------------------------
        # Authoritative training dataset
        # ------------------------------------------------------------------------------------------

        if dataset_id not in TRAINING_DATA:
            raise RuntimeError(
                f"{dataset_id}: training data is not available."
            )

        train_df = TRAINING_DATA[
            dataset_id
        ]

        if not isinstance(
            train_df,
            pd.DataFrame,
        ):
            raise RuntimeError(
                f"{dataset_id}: training data is not a "
                "pandas DataFrame."
            )

        if len(train_df) == 0:
            raise RuntimeError(
                f"{dataset_id}: training dataset is empty."
            )

        training_rows = int(
            len(train_df)
        )

        generative_columns = list(
            TRAINING_GENERATIVE_COLUMNS[
                dataset_id
            ]
        )

        target_column = (
            TRAINING_TARGET_COLUMNS[
                dataset_id
            ]
        )

        provenance_column = (
            TRAINING_PROVENANCE_COLUMNS[
                dataset_id
            ]
        )

        identifier_columns = list(
            TRAINING_IDENTIFIER_COLUMNS[
                dataset_id
            ]
        )

        # ------------------------------------------------------------------------------------------
        # Validate training schema
        # ------------------------------------------------------------------------------------------

        missing_training_columns = [
            column
            for column in generative_columns
            if column not in train_df.columns
        ]

        if missing_training_columns:
            raise RuntimeError(
                f"{dataset_id}: training data is missing "
                f"generative column(s): "
                f"{missing_training_columns}"
            )

        if target_column not in generative_columns:
            raise RuntimeError(
                f"{dataset_id}: target '{target_column}' "
                "is not present in the generative schema."
            )

        # ------------------------------------------------------------------------------------------
        # Baseline loop
        # ------------------------------------------------------------------------------------------

        for baseline_name in BASELINE_METHODS:

            print()
            print("-" * 100)
            print(
                f"GENERATING — {dataset_id} | "
                f"REPETITION {repetition} | "
                f"{baseline_name}"
            )
            print("-" * 100)

            # --------------------------------------------------------------------------------------
            # Resolve authoritative experiment seed
            # --------------------------------------------------------------------------------------

            seed = int(
                EXPERIMENT_SEEDS[
                    repetition
                ][
                    dataset_id
                ][
                    baseline_name
                ]
            )

            seed_everything(
                seed
            )

            # --------------------------------------------------------------------------------------
            # Resolve output artifact
            # --------------------------------------------------------------------------------------

            output_path = (
                get_synthetic_artifact_path(
                    repetition=repetition,
                    dataset_id=dataset_id,
                    baseline_name=baseline_name,
                )
            )

            output_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            # --------------------------------------------------------------------------------------
            # Restart-safe check
            # --------------------------------------------------------------------------------------

            existing_output_valid = (
                validate_existing_synthetic_artifact(
                    output_path=output_path,
                    dataset_id=dataset_id,
                    baseline_name=baseline_name,
                    expected_rows=training_rows,
                    expected_columns=generative_columns,
                )
            )

            if existing_output_valid:

                output_sha256 = sha256_file(
                    output_path
                )

                model_path = (
                    get_model_artifact_path(
                        repetition=repetition,
                        dataset_id=dataset_id,
                        baseline_name=baseline_name,
                    )
                )

                if not model_path.exists():
                    raise RuntimeError(
                        "Synthetic artifact exists but its corresponding "
                        f"Section 08 model is missing:\n{model_path}"
                    )

                model_sha256 = sha256_file(
                    model_path
                )

                print(
                    "✓ Existing valid synthetic artifact detected."
                )

                print(
                    f"  Output : {output_path}"
                )

                print(
                    f"  SHA256 : {output_sha256}"
                )

                print(
                    "  Action : SKIPPED — restart-safe reuse"
                )

                GENERATION_RUNTIME_RECORDS.append(
                    {
                        "repetition": int(
                            repetition
                        ),
                        "repetition_seed": int(
                            repetition_seed
                        ),
                        "dataset_id": dataset_id,
                        "baseline": baseline_name,
                        "experiment_seed": int(
                            seed
                        ),
                        "training_rows": int(
                            training_rows
                        ),
                        "rows_requested": int(
                            training_rows
                        ),
                        "rows_generated": int(
                            training_rows
                        ),
                        "generative_columns": int(
                            len(
                                generative_columns
                            )
                        ),
                        "target_column": target_column,
                        "generation_runtime_seconds": 0.0,
                        "model_artifact": str(
                            model_path
                        ),
                        "model_sha256": model_sha256,
                        "synthetic_artifact": str(
                            output_path
                        ),
                        "synthetic_sha256": output_sha256,
                        "execution_status": "REUSED_EXISTING",
                        "status": "PASS",
                    }
                )

                GENERATION_MANIFEST_RECORDS.append(
                    {
                        "repetition": int(
                            repetition
                        ),
                        "repetition_seed": int(
                            repetition_seed
                        ),
                        "dataset_id": dataset_id,
                        "baseline": baseline_name,
                        "experiment_seed": int(
                            seed
                        ),
                        "training_rows": int(
                            training_rows
                        ),
                        "rows_generated": int(
                            training_rows
                        ),
                        "generative_columns": int(
                            len(
                                generative_columns
                            )
                        ),
                        "target_column": target_column,
                        "provenance_column": (
                            provenance_column
                            if provenance_column is not None
                            else ""
                        ),
                        "identifier_columns": json.dumps(
                            identifier_columns,
                            ensure_ascii=False,
                        ),
                        "model_artifact": str(
                            model_path
                        ),
                        "model_sha256": model_sha256,
                        "synthetic_artifact": str(
                            output_path
                        ),
                        "synthetic_sha256": output_sha256,
                        "generation_runtime_seconds": 0.0,
                        "execution_status": "REUSED_EXISTING",
                        "status": "PASS",
                    }
                )

                gc.collect()

                continue

            # --------------------------------------------------------------------------------------
            # Start runtime measurement
            # --------------------------------------------------------------------------------------

            start_time = time.perf_counter()

            # --------------------------------------------------------------------------------------
            # Load exactly one Section 08 model
            # --------------------------------------------------------------------------------------

            model, model_path = (
                load_persisted_model(
                    repetition=repetition,
                    dataset_id=dataset_id,
                    baseline_name=baseline_name,
                )
            )

            model_sha256 = sha256_file(
                model_path
            )

            print(
                f"✓ Model loaded | SHA256={model_sha256}"
            )

            # --------------------------------------------------------------------------------------
            # Generate synthetic data
            # --------------------------------------------------------------------------------------

            if baseline_name == "independent_marginal":

                synthetic_df = (
                    generate_independent_marginal(
                        model=model,
                        generative_columns=generative_columns,
                        training_rows=training_rows,
                        seed=seed,
                        dataset_id=dataset_id,
                    )
                )

            elif baseline_name == "gaussian_copula":

                synthetic_df = (
                    generate_gaussian_copula(
                        synthesizer=model,
                        generative_columns=generative_columns,
                        training_rows=training_rows,
                        dataset_id=dataset_id,
                    )
                )

            else:

                raise RuntimeError(
                    f"Unknown statistical baseline: "
                    f"{baseline_name}"
                )

            # --------------------------------------------------------------------------------------
            # Model no longer needed
            # --------------------------------------------------------------------------------------
            #
            # Free the fitted model immediately after sampling.
            #

            del model

            gc.collect()

            # --------------------------------------------------------------------------------------
            # Validate generated object
            # --------------------------------------------------------------------------------------

            if not isinstance(
                synthetic_df,
                pd.DataFrame,
            ):
                raise RuntimeError(
                    f"{dataset_id} / {baseline_name}: "
                    "synthetic output is not a pandas DataFrame."
                )

            # --------------------------------------------------------------------------------------
            # Validate row count
            # --------------------------------------------------------------------------------------

            rows_generated = int(
                len(
                    synthetic_df
                )
            )

            if rows_generated != training_rows:
                raise RuntimeError(
                    f"{dataset_id} / {baseline_name}: "
                    f"expected {training_rows:,} rows but generated "
                    f"{rows_generated:,}."
                )

            # --------------------------------------------------------------------------------------
            # Validate exact schema
            # --------------------------------------------------------------------------------------

            if list(
                synthetic_df.columns
            ) != list(
                generative_columns
            ):
                raise RuntimeError(
                    f"{dataset_id} / {baseline_name}: "
                    "generated schema does not exactly match "
                    "the frozen generative schema."
                )

            # --------------------------------------------------------------------------------------
            # Validate target retention
            # --------------------------------------------------------------------------------------

            if target_column not in synthetic_df.columns:
                raise RuntimeError(
                    f"{dataset_id} / {baseline_name}: "
                    f"target '{target_column}' is missing "
                    "from generated data."
                )

            # --------------------------------------------------------------------------------------
            # Validate provenance exclusion
            # --------------------------------------------------------------------------------------

            if provenance_column is not None:

                if provenance_column in synthetic_df.columns:
                    raise RuntimeError(
                        f"{dataset_id} / {baseline_name}: "
                        f"provenance column '{provenance_column}' "
                        "must not be generated."
                    )

            # --------------------------------------------------------------------------------------
            # Validate identifier exclusion
            # --------------------------------------------------------------------------------------

            leaked_identifiers = [
                column
                for column in identifier_columns
                if column in synthetic_df.columns
            ]

            if leaked_identifiers:
                raise RuntimeError(
                    f"{dataset_id} / {baseline_name}: "
                    "identifier column(s) leaked into synthetic data: "
                    f"{leaked_identifiers}"
                )

            # --------------------------------------------------------------------------------------
            # Persist synthetic data
            # --------------------------------------------------------------------------------------

            temporary_output_path = (
                output_path.parent
                / f".{output_path.stem}.tmp.csv"
            )

            # Remove stale temporary artifact if present.
            if temporary_output_path.exists():
                temporary_output_path.unlink()

            synthetic_df.to_csv(
                temporary_output_path,
                index=False,
            )

            # Atomic replacement.
            os.replace(
                temporary_output_path,
                output_path,
            )

            # --------------------------------------------------------------------------------------
            # Verify persisted artifact
            # --------------------------------------------------------------------------------------

            if not output_path.exists():
                raise RuntimeError(
                    f"Synthetic output was not persisted: "
                    f"{output_path}"
                )

            output_sha256 = sha256_file(
                output_path
            )

            # --------------------------------------------------------------------------------------
            # Persisted row/schema verification
            # --------------------------------------------------------------------------------------

            verification_header = pd.read_csv(
                output_path,
                nrows=0,
            )

            if list(
                verification_header.columns
            ) != list(
                generative_columns
            ):
                raise RuntimeError(
                    f"{dataset_id} / {baseline_name}: "
                    "persisted synthetic schema validation failed."
                )

            del verification_header

            # --------------------------------------------------------------------------------------
            # Runtime
            # --------------------------------------------------------------------------------------

            elapsed = (
                time.perf_counter()
                - start_time
            )

            # --------------------------------------------------------------------------------------
            # Runtime record
            # --------------------------------------------------------------------------------------

            GENERATION_RUNTIME_RECORDS.append(
                {
                    "repetition": int(
                        repetition
                    ),
                    "repetition_seed": int(
                        repetition_seed
                    ),
                    "dataset_id": dataset_id,
                    "baseline": baseline_name,
                    "experiment_seed": int(
                        seed
                    ),
                    "training_rows": int(
                        training_rows
                    ),
                    "rows_requested": int(
                        training_rows
                    ),
                    "rows_generated": int(
                        rows_generated
                    ),
                    "generative_columns": int(
                        len(
                            generative_columns
                        )
                    ),
                    "target_column": target_column,
                    "generation_runtime_seconds": float(
                        elapsed
                    ),
                    "model_artifact": str(
                        model_path
                    ),
                    "model_sha256": model_sha256,
                    "synthetic_artifact": str(
                        output_path
                    ),
                    "synthetic_sha256": output_sha256,
                    "execution_status": "GENERATED",
                    "status": "PASS",
                }
            )

            # --------------------------------------------------------------------------------------
            # Manifest record
            # --------------------------------------------------------------------------------------

            GENERATION_MANIFEST_RECORDS.append(
                {
                    "repetition": int(
                        repetition
                    ),
                    "repetition_seed": int(
                        repetition_seed
                    ),
                    "dataset_id": dataset_id,
                    "baseline": baseline_name,
                    "experiment_seed": int(
                        seed
                    ),
                    "training_rows": int(
                        training_rows
                    ),
                    "rows_generated": int(
                        rows_generated
                    ),
                    "generative_columns": int(
                        len(
                            generative_columns
                        )
                    ),
                    "target_column": target_column,
                    "provenance_column": (
                        provenance_column
                        if provenance_column is not None
                        else ""
                    ),
                    "identifier_columns": json.dumps(
                        identifier_columns,
                        ensure_ascii=False,
                    ),
                    "model_artifact": str(
                        model_path
                    ),
                    "model_sha256": model_sha256,
                    "synthetic_artifact": str(
                        output_path
                    ),
                    "synthetic_sha256": output_sha256,
                    "generation_runtime_seconds": float(
                        elapsed
                    ),
                    "execution_status": "GENERATED",
                    "status": "PASS",
                }
            )

            # --------------------------------------------------------------------------------------
            # Console output
            # --------------------------------------------------------------------------------------

            print(
                f"✓ Generated | "
                f"rows={rows_generated:,} | "
                f"columns={len(generative_columns)} | "
                f"runtime={elapsed:.3f}s"
            )

            print(
                f"  Synthetic : {output_path}"
            )

            print(
                f"  SHA256    : {output_sha256}"
            )

            # --------------------------------------------------------------------------------------
            # CRITICAL RAM CLEANUP
            # --------------------------------------------------------------------------------------

            del synthetic_df

            gc.collect()

            print(
                "✓ RAM cleanup completed."
            )


# --------------------------------------------------------------------------------------------------
# 19. Build Runtime DataFrames
# --------------------------------------------------------------------------------------------------

runtime_df = pd.DataFrame(
    GENERATION_RUNTIME_RECORDS
)

manifest_df = pd.DataFrame(
    GENERATION_MANIFEST_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 20. Validate 30-Run Coverage
# --------------------------------------------------------------------------------------------------

if len(runtime_df) != EXPECTED_RUN_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RUN_COUNT} runtime records, "
        f"found {len(runtime_df)}."
    )


if len(manifest_df) != EXPECTED_RUN_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RUN_COUNT} manifest records, "
        f"found {len(manifest_df)}."
    )


# --------------------------------------------------------------------------------------------------
# 21. Validate Unique Experimental Units
# --------------------------------------------------------------------------------------------------

EXPERIMENTAL_UNIT_COLUMNS = [
    "repetition",
    "dataset_id",
    "baseline",
]


if runtime_df.duplicated(
    subset=EXPERIMENTAL_UNIT_COLUMNS,
    keep=False,
).any():

    raise RuntimeError(
        "Duplicate experimental units detected "
        "in generation runtime records."
    )


if manifest_df.duplicated(
    subset=EXPERIMENTAL_UNIT_COLUMNS,
    keep=False,
).any():

    raise RuntimeError(
        "Duplicate experimental units detected "
        "in generation manifest."
    )


# --------------------------------------------------------------------------------------------------
# 22. Validate Exact Experimental Coverage
# --------------------------------------------------------------------------------------------------

expected_units = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


actual_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in runtime_df.iterrows()
}


if actual_units != expected_units:

    missing_units = sorted(
        expected_units
        - actual_units
    )

    extra_units = sorted(
        actual_units
        - expected_units
    )

    raise RuntimeError(
        "Generation experimental coverage mismatch.\n"
        f"Missing units    : {missing_units}\n"
        f"Unexpected units : {extra_units}"
    )


# --------------------------------------------------------------------------------------------------
# 23. Validate All Persisted Synthetic Artifacts
# --------------------------------------------------------------------------------------------------

for _, record in manifest_df.iterrows():

    synthetic_path = Path(
        record[
            "synthetic_artifact"
        ]
    )

    if not synthetic_path.exists():
        raise RuntimeError(
            f"Missing synthetic artifact: "
            f"{synthetic_path}"
        )

    expected_rows = int(
        record[
            "training_rows"
        ]
    )

    dataset_id = record[
        "dataset_id"
    ]

    baseline_name = record[
        "baseline"
    ]

    expected_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    if not validate_existing_synthetic_artifact(
        output_path=synthetic_path,
        dataset_id=dataset_id,
        baseline_name=baseline_name,
        expected_rows=expected_rows,
        expected_columns=expected_columns,
    ):
        raise RuntimeError(
            f"Persisted synthetic artifact validation failed:\n"
            f"{synthetic_path}"
        )

    persisted_sha256 = sha256_file(
        synthetic_path
    )

    if persisted_sha256 != record[
        "synthetic_sha256"
    ]:
        raise RuntimeError(
            f"Synthetic artifact SHA256 mismatch:\n"
            f"{synthetic_path}"
        )


# --------------------------------------------------------------------------------------------------
# 24. Persist Generation Manifest
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "baseline_generation_manifest.csv"
)

GENERATION_RUNTIME_PATH = (
    MANIFEST_ROOT
    / "baseline_generation_runtime.csv"
)


manifest_df = (
    manifest_df
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


runtime_df = (
    runtime_df
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


manifest_df.to_csv(
    GENERATION_MANIFEST_PATH,
    index=False,
)


runtime_df.to_csv(
    GENERATION_RUNTIME_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 25. Manifest Integrity
# --------------------------------------------------------------------------------------------------

generation_manifest_sha256 = (
    sha256_file(
        GENERATION_MANIFEST_PATH
    )
)

generation_runtime_sha256 = (
    sha256_file(
        GENERATION_RUNTIME_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 26. Final Dataset-Level Summary
# --------------------------------------------------------------------------------------------------

dataset_summary_records = []

for dataset_id in DATASET_IDS:

    expected_rows = int(
        len(
            TRAINING_DATA[
                dataset_id
            ]
        )
    )

    expected_columns = len(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    dataset_summary_records.append(
        {
            "dataset_id": dataset_id,
            "training_rows": expected_rows,
            "generative_columns": expected_columns,
        }
    )


dataset_summary_df = pd.DataFrame(
    dataset_summary_records
)


# --------------------------------------------------------------------------------------------------
# 27. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 9 — GENERATION SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets                         : {len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset             : {len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                       : {REPETITIONS}"
)

print(
    f"✓ Expected experimental runs        : {EXPECTED_RUN_COUNT}"
)

print(
    f"✓ Generation runtime records        : {len(runtime_df)}"
)

print(
    f"✓ Generation manifest records       : {len(manifest_df)}"
)

print(
    "✓ Training-sized generation         : PASS"
)

print(
    "✓ Exact generative schema           : PASS"
)

print(
    "✓ Target retained                   : PASS"
)

print(
    "✓ Provenance exclusion              : PASS"
)

print(
    "✓ Identifier exclusion              : PASS"
)

print(
    "✓ 5-repetition coverage             : PASS"
)

print(
    "✓ 30-run experimental coverage      : PASS"
)

print(
    "✓ Persisted synthetic artifacts     : PASS"
)

print(
    "✓ Synthetic artifact integrity      : PASS"
)

print(
    f"✓ Generation manifest               : "
    f"{GENERATION_MANIFEST_PATH}"
)

print(
    f"✓ Generation manifest SHA256        : "
    f"{generation_manifest_sha256}"
)

print(
    f"✓ Runtime manifest                  : "
    f"{GENERATION_RUNTIME_PATH}"
)

print(
    f"✓ Runtime manifest SHA256           : "
    f"{generation_runtime_sha256}"
)

print()
print(
    "✓ Section 9 synthetic-data generation "
    "completed successfully."
)


# --------------------------------------------------------------------------------------------------
# 28. Release Temporary Runtime Objects
# --------------------------------------------------------------------------------------------------

gc.collect()

print(
    "✓ Final RAM cleanup completed."
)

SECTION 9 — GENERATE SYNTHETIC DATA

----------------------------------------------------------------------------------------------------
ARTIFACT ROOT VALIDATION
----------------------------------------------------------------------------------------------------
✓ Project root          : /content/drive/MyDrive/SPP_GAN_Research
✓ Section 08 model root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/models
✓ Synthetic output root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/synthetic
✓ Manifest root         : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/manifests

REPETITION 1 / 5
Authoritative repetition seed : 3026

----------------------------------------------------------------------------------------------------
GENERATING — adult_income | REPETITION 1 | independent_marginal
----------------------------------------------------------------------------------------------------
✓ Existing v

In [80]:
# ==================================================================================================
# SECTION 10 — VALIDATE SYNTHETIC SCHEMA
# RAM-SAFE + RESTART-SAFE
# ==================================================================================================

print("=" * 100)
print("SECTION 10 — VALIDATE SYNTHETIC SCHEMA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import gc
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "DATASET_IDS",
    "BASELINE_METHODS",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
    "REPETITIONS",
]

_missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if _missing_runtime_objects:
    raise RuntimeError(
        "Section 10 is missing required runtime objects: "
        f"{_missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if list(BASELINE_METHODS) != EXPECTED_BASELINES:
    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 "
        f"baseline registry.\n"
        f"Expected: {EXPECTED_BASELINES}\n"
        f"Found   : {list(BASELINE_METHODS)}"
    )


if int(REPETITIONS) != 5:
    raise RuntimeError(
        f"Notebook 04 requires 5 repetitions, found {REPETITIONS}."
    )


EXPECTED_VALIDATION_COUNT = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)


if EXPECTED_VALIDATION_COUNT != 30:
    raise RuntimeError(
        "Section 10 must validate exactly 30 experimental units. "
        f"Found {EXPECTED_VALIDATION_COUNT}."
    )


# --------------------------------------------------------------------------------------------------
# 4. Authoritative Paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NOTEBOOK_04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

BASELINE_ROOT = (
    NOTEBOOK_04_ROOT
    / "baselines"
)

SYNTHETIC_ROOT = (
    BASELINE_ROOT
    / "synthetic"
)

MANIFEST_ROOT = (
    BASELINE_ROOT
    / "manifests"
)

GENERATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "baseline_generation_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Section 09 Generation Manifest
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("SECTION 09 ARTIFACT VALIDATION")
print("-" * 100)

print(
    f"✓ Synthetic root    : {SYNTHETIC_ROOT}"
)

print(
    f"✓ Generation manifest: {GENERATION_MANIFEST_PATH}"
)


if not GENERATION_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Section 09 generation manifest was not found:\n"
        f"{GENERATION_MANIFEST_PATH}"
    )


GENERATION_MANIFEST_DF = pd.read_csv(
    GENERATION_MANIFEST_PATH
)


if len(
    GENERATION_MANIFEST_DF
) != EXPECTED_VALIDATION_COUNT:

    raise RuntimeError(
        "Section 09 generation manifest does not contain "
        f"{EXPECTED_VALIDATION_COUNT} records.\n"
        f"Found: {len(GENERATION_MANIFEST_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Validate Required Manifest Columns
# --------------------------------------------------------------------------------------------------

REQUIRED_MANIFEST_COLUMNS = [
    "repetition",
    "repetition_seed",
    "dataset_id",
    "baseline",
    "experiment_seed",
    "training_rows",
    "rows_generated",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "synthetic_artifact",
    "synthetic_sha256",
    "status",
]


missing_manifest_columns = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in GENERATION_MANIFEST_DF.columns
]


if missing_manifest_columns:
    raise RuntimeError(
        "Section 09 generation manifest is missing "
        f"required column(s): {missing_manifest_columns}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate Manifest Experimental Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_UNITS = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


ACTUAL_UNITS = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_MANIFEST_DF.iterrows()
}


if ACTUAL_UNITS != EXPECTED_UNITS:

    missing_units = sorted(
        EXPECTED_UNITS
        - ACTUAL_UNITS
    )

    unexpected_units = sorted(
        ACTUAL_UNITS
        - EXPECTED_UNITS
    )

    raise RuntimeError(
        "Section 09 generation manifest coverage mismatch.\n"
        f"Missing units    : {missing_units}\n"
        f"Unexpected units : {unexpected_units}"
    )


print(
    "✓ Section 09 manifest coverage : PASS"
)

print(
    f"✓ Manifest records             : "
    f"{len(GENERATION_MANIFEST_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 8. Runtime Validation Container
# --------------------------------------------------------------------------------------------------

SYNTHETIC_SCHEMA_VALIDATION = []


# --------------------------------------------------------------------------------------------------
# 9. Validate Each Persisted Synthetic Dataset
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# Synthetic CSV files are loaded ONE AT A TIME.
#
# No complete synthetic dataset collection is retained in memory.
#

for repetition in range(
    1,
    int(REPETITIONS) + 1,
):

    print()
    print("=" * 100)
    print(
        f"REPETITION {repetition} / {REPETITIONS}"
    )
    print("=" * 100)

    repetition_manifest = (
        GENERATION_MANIFEST_DF[
            GENERATION_MANIFEST_DF[
                "repetition"
            ].astype(int)
            == repetition
        ]
    )

    if len(
        repetition_manifest
    ) != (
        len(DATASET_IDS)
        * len(BASELINE_METHODS)
    ):
        raise RuntimeError(
            f"Repetition {repetition}: expected 6 "
            f"manifest records, found "
            f"{len(repetition_manifest)}."
        )

    for _, manifest_record in (
        repetition_manifest
        .sort_values(
            by=[
                "dataset_id",
                "baseline",
            ]
        )
        .iterrows()
    ):

        dataset_id = manifest_record[
            "dataset_id"
        ]

        baseline_name = manifest_record[
            "baseline"
        ]

        print()
        print("-" * 100)
        print(
            f"VALIDATING — {dataset_id} | "
            f"REPETITION {repetition} | "
            f"{baseline_name}"
        )
        print("-" * 100)

        # ------------------------------------------------------------------------------------------
        # 9.1 Validate dataset registration
        # ------------------------------------------------------------------------------------------

        if dataset_id not in DATASET_IDS:
            raise RuntimeError(
                f"Unknown dataset '{dataset_id}' "
                "found in Section 09 manifest."
            )

        if baseline_name not in BASELINE_METHODS:
            raise RuntimeError(
                f"Unknown baseline '{baseline_name}' "
                "found in Section 09 manifest."
            )

        # ------------------------------------------------------------------------------------------
        # 9.2 Authoritative schema
        # ------------------------------------------------------------------------------------------

        expected_columns = list(
            TRAINING_GENERATIVE_COLUMNS[
                dataset_id
            ]
        )

        target = (
            TRAINING_TARGET_COLUMNS[
                dataset_id
            ]
        )

        provenance = (
            TRAINING_PROVENANCE_COLUMNS[
                dataset_id
            ]
        )

        identifiers = list(
            TRAINING_IDENTIFIER_COLUMNS[
                dataset_id
            ]
        )

        # ------------------------------------------------------------------------------------------
        # 9.3 Expected training row count
        # ------------------------------------------------------------------------------------------

        expected_rows = int(
            manifest_record[
                "training_rows"
            ]
        )

        expected_generated_rows = int(
            manifest_record[
                "rows_generated"
            ]
        )

        if expected_generated_rows != expected_rows:
            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "Section 09 manifest row-count mismatch."
            )

        # ------------------------------------------------------------------------------------------
        # 9.4 Synthetic artifact path
        # ------------------------------------------------------------------------------------------

        synthetic_path = Path(
            manifest_record[
                "synthetic_artifact"
            ]
        )

        if not synthetic_path.exists():
            raise FileNotFoundError(
                f"Synthetic artifact does not exist:\n"
                f"{synthetic_path}"
            )

        # ------------------------------------------------------------------------------------------
        # 9.5 Validate persisted SHA256
        # ------------------------------------------------------------------------------------------

        import hashlib

        digest = hashlib.sha256()

        with synthetic_path.open(
            "rb"
        ) as handle:

            for chunk in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(
                    chunk
                )

        persisted_sha256 = digest.hexdigest()

        expected_sha256 = str(
            manifest_record[
                "synthetic_sha256"
            ]
        )

        if persisted_sha256 != expected_sha256:
            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "synthetic artifact SHA256 mismatch."
            )

        # ------------------------------------------------------------------------------------------
        # 9.6 Read synthetic dataset
        # ------------------------------------------------------------------------------------------
        #
        # One file at a time.
        #

        synthetic_df = pd.read_csv(
            synthetic_path
        )

        # ------------------------------------------------------------------------------------------
        # 9.7 Exact column order
        # ------------------------------------------------------------------------------------------

        if list(
            synthetic_df.columns
        ) != expected_columns:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "synthetic column schema mismatch."
            )

        # ------------------------------------------------------------------------------------------
        # 9.8 Row count
        # ------------------------------------------------------------------------------------------

        actual_rows = int(
            len(
                synthetic_df
            )
        )

        if actual_rows != expected_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                f"expected {expected_rows:,} rows, "
                f"found {actual_rows:,}."
            )

        # ------------------------------------------------------------------------------------------
        # 9.9 Target retained
        # ------------------------------------------------------------------------------------------

        target_present = (
            target in synthetic_df.columns
        )

        if not target_present:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                f"target column '{target}' is missing."
            )

        # ------------------------------------------------------------------------------------------
        # 9.10 Provenance excluded
        # ------------------------------------------------------------------------------------------

        provenance_excluded = True

        if provenance is not None:

            if provenance in synthetic_df.columns:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    f"provenance leakage detected: "
                    f"'{provenance}'."
                )

        # ------------------------------------------------------------------------------------------
        # 9.11 Identifier exclusion
        # ------------------------------------------------------------------------------------------

        identifier_leakage = (
            set(
                identifiers
            )
            .intersection(
                synthetic_df.columns
            )
        )

        if identifier_leakage:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "identifier leakage detected: "
                f"{sorted(identifier_leakage)}"
            )

        # ------------------------------------------------------------------------------------------
        # 9.12 Validate manifest metadata
        # ------------------------------------------------------------------------------------------

        manifest_target = str(
            manifest_record[
                "target_column"
            ]
        )

        if manifest_target != str(
            target
        ):
            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "manifest target does not match "
                "authoritative target."
            )

        manifest_column_count = int(
            manifest_record[
                "generative_columns"
            ]
        )

        if manifest_column_count != len(
            expected_columns
        ):
            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "manifest generative-column count "
                "does not match authoritative schema."
            )

        # ------------------------------------------------------------------------------------------
        # 9.13 Record successful validation
        # ------------------------------------------------------------------------------------------

        SYNTHETIC_SCHEMA_VALIDATION.append(
            {
                "repetition": int(
                    repetition
                ),
                "repetition_seed": int(
                    manifest_record[
                        "repetition_seed"
                    ]
                ),
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "experiment_seed": int(
                    manifest_record[
                        "experiment_seed"
                    ]
                ),
                "schema_pass": True,
                "rows_expected": int(
                    expected_rows
                ),
                "rows_actual": int(
                    actual_rows
                ),
                "columns": int(
                    len(
                        synthetic_df.columns
                    )
                ),
                "target_present": True,
                "provenance_excluded": True,
                "identifiers_excluded": True,
                "sha256_verified": True,
                "status": "PASS",
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"rows={actual_rows:>8,} | "
            f"columns={len(expected_columns):>4} | "
            f"schema PASS"
        )

        # ------------------------------------------------------------------------------------------
        # 9.14 Critical RAM cleanup
        # ------------------------------------------------------------------------------------------

        del synthetic_df

        gc.collect()


# --------------------------------------------------------------------------------------------------
# 10. Build Validation DataFrame
# --------------------------------------------------------------------------------------------------

SYNTHETIC_SCHEMA_VALIDATION_DF = pd.DataFrame(
    SYNTHETIC_SCHEMA_VALIDATION
)


# --------------------------------------------------------------------------------------------------
# 11. Validate Final Record Count
# --------------------------------------------------------------------------------------------------

if len(
    SYNTHETIC_SCHEMA_VALIDATION_DF
) != EXPECTED_VALIDATION_COUNT:

    raise RuntimeError(
        "Synthetic schema validation record count mismatch.\n"
        f"Expected: {EXPECTED_VALIDATION_COUNT}\n"
        f"Found   : {len(SYNTHETIC_SCHEMA_VALIDATION_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 12. Validate Unique Experimental Units
# --------------------------------------------------------------------------------------------------

validation_units = (
    SYNTHETIC_SCHEMA_VALIDATION_DF[
        [
            "repetition",
            "dataset_id",
            "baseline",
        ]
    ]
    .drop_duplicates()
)


if len(
    validation_units
) != EXPECTED_VALIDATION_COUNT:

    raise RuntimeError(
        "Duplicate or missing experimental units detected "
        "in synthetic schema validation."
    )


# --------------------------------------------------------------------------------------------------
# 13. Validate Every Record Passed
# --------------------------------------------------------------------------------------------------

validation_pass = (
    SYNTHETIC_SCHEMA_VALIDATION_DF[
        "schema_pass"
    ]
    .all()
)

target_pass = (
    SYNTHETIC_SCHEMA_VALIDATION_DF[
        "target_present"
    ]
    .all()
)

provenance_pass = (
    SYNTHETIC_SCHEMA_VALIDATION_DF[
        "provenance_excluded"
    ]
    .all()
)

identifier_pass = (
    SYNTHETIC_SCHEMA_VALIDATION_DF[
        "identifiers_excluded"
    ]
    .all()
)

sha256_pass = (
    SYNTHETIC_SCHEMA_VALIDATION_DF[
        "sha256_verified"
    ]
    .all()
)


if not validation_pass:
    raise RuntimeError(
        "One or more synthetic schema validations failed."
    )


if not target_pass:
    raise RuntimeError(
        "Target-retention validation failed."
    )


if not provenance_pass:
    raise RuntimeError(
        "Provenance-exclusion validation failed."
    )


if not identifier_pass:
    raise RuntimeError(
        "Identifier-exclusion validation failed."
    )


if not sha256_pass:
    raise RuntimeError(
        "Synthetic artifact integrity validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 14. Persist Section 10 Validation Results
# --------------------------------------------------------------------------------------------------

SCHEMA_VALIDATION_PATH = (
    MANIFEST_ROOT
    / "synthetic_schema_validation.csv"
)


SYNTHETIC_SCHEMA_VALIDATION_DF = (
    SYNTHETIC_SCHEMA_VALIDATION_DF
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


SYNTHETIC_SCHEMA_VALIDATION_DF.to_csv(
    SCHEMA_VALIDATION_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 15. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 10 — SYNTHETIC SCHEMA VALIDATION SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets validated              : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset           : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                     : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected validation runs       : "
    f"{EXPECTED_VALIDATION_COUNT}"
)

print(
    f"✓ Actual validation records      : "
    f"{len(SYNTHETIC_SCHEMA_VALIDATION_DF)}"
)

print(
    "✓ Exact column order              : PASS"
)

print(
    "✓ Training row-count validation   : PASS"
)

print(
    "✓ Target retention                : PASS"
)

print(
    "✓ Provenance exclusion            : PASS"
)

print(
    "✓ Identifier exclusion            : PASS"
)

print(
    "✓ Synthetic SHA256 integrity      : PASS"
)

print(
    "✓ 5-repetition coverage           : PASS"
)

print(
    "✓ 30-run experimental coverage    : PASS"
)

print(
    f"✓ Validation artifact             : "
    f"{SCHEMA_VALIDATION_PATH}"
)

print()
print(
    "✓ Section 10 synthetic schema validation "
    "completed successfully."
)

print(
    "✓ Final RAM cleanup completed."
)

gc.collect()

SECTION 10 — VALIDATE SYNTHETIC SCHEMA

----------------------------------------------------------------------------------------------------
SECTION 09 ARTIFACT VALIDATION
----------------------------------------------------------------------------------------------------
✓ Synthetic root    : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/synthetic
✓ Generation manifest: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/manifests/baseline_generation_manifest.csv
✓ Section 09 manifest coverage : PASS
✓ Manifest records             : 30

REPETITION 1 / 5

----------------------------------------------------------------------------------------------------
VALIDATING — adult_income | REPETITION 1 | gaussian_copula
----------------------------------------------------------------------------------------------------
✓ adult_income         | gaussian_copula          | rows=  34,189 | columns=  15 | schema PASS

-------------------------

0

In [82]:
# ==================================================================================================
# SECTION 11 — VALIDATE SAMPLE SIZE
# RAM-SAFE + RESTART-SAFE
# ==================================================================================================

print("=" * 100)
print("SECTION 11 — VALIDATE SAMPLE SIZE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import gc
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "DATASET_IDS",
    "BASELINE_METHODS",
    "REPETITIONS",
    "TRAINING_DATA",
]

_missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if _missing_runtime_objects:

    raise RuntimeError(
        "Section 11 is missing required runtime objects: "
        f"{_missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if list(BASELINE_METHODS) != EXPECTED_BASELINES:

    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 "
        f"baseline registry.\n"
        f"Expected: {EXPECTED_BASELINES}\n"
        f"Found   : {list(BASELINE_METHODS)}"
    )


if int(REPETITIONS) != 5:

    raise RuntimeError(
        f"Notebook 04 requires 5 repetitions, found {REPETITIONS}."
    )


EXPECTED_VALIDATION_RECORDS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)


if EXPECTED_VALIDATION_RECORDS != 30:

    raise RuntimeError(
        "Section 11 must validate exactly 30 experimental units. "
        f"Found {EXPECTED_VALIDATION_RECORDS}."
    )


# --------------------------------------------------------------------------------------------------
# 4. Authoritative Project Paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NOTEBOOK_04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

BASELINE_ROOT = (
    NOTEBOOK_04_ROOT
    / "baselines"
)

MANIFEST_ROOT = (
    BASELINE_ROOT
    / "manifests"
)

GENERATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "baseline_generation_manifest.csv"
)

SCHEMA_VALIDATION_PATH = (
    MANIFEST_ROOT
    / "synthetic_schema_validation.csv"
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Section 09 Generation Manifest
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("SECTION 09 ARTIFACT VALIDATION")
print("-" * 100)

print(
    f"✓ Generation manifest : "
    f"{GENERATION_MANIFEST_PATH}"
)


if not GENERATION_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "Section 09 generation manifest was not found:\n"
        f"{GENERATION_MANIFEST_PATH}"
    )


GENERATION_MANIFEST_DF = pd.read_csv(
    GENERATION_MANIFEST_PATH
)


if len(
    GENERATION_MANIFEST_DF
) != EXPECTED_VALIDATION_RECORDS:

    raise RuntimeError(
        "Section 09 generation manifest does not contain "
        f"{EXPECTED_VALIDATION_RECORDS} records.\n"
        f"Found: {len(GENERATION_MANIFEST_DF)}"
    )


print(
    f"✓ Generation manifest records : "
    f"{len(GENERATION_MANIFEST_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 6. Required Manifest Columns
# --------------------------------------------------------------------------------------------------

REQUIRED_MANIFEST_COLUMNS = [
    "repetition",
    "repetition_seed",
    "dataset_id",
    "baseline",
    "experiment_seed",
    "training_rows",
    "rows_generated",
    "synthetic_artifact",
    "status",
]


missing_manifest_columns = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in GENERATION_MANIFEST_DF.columns
]


if missing_manifest_columns:

    raise RuntimeError(
        "Section 09 generation manifest is missing "
        f"required column(s): {missing_manifest_columns}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Validate Experimental Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_UNITS = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


ACTUAL_UNITS = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_MANIFEST_DF.iterrows()
}


if ACTUAL_UNITS != EXPECTED_UNITS:

    missing_units = sorted(
        EXPECTED_UNITS
        - ACTUAL_UNITS
    )

    unexpected_units = sorted(
        ACTUAL_UNITS
        - EXPECTED_UNITS
    )

    raise RuntimeError(
        "Section 09 generation manifest coverage mismatch.\n"
        f"Missing units    : {missing_units}\n"
        f"Unexpected units : {unexpected_units}"
    )


print(
    "✓ 5-repetition experimental coverage : PASS"
)

print(
    "✓ 30-run experimental coverage       : PASS"
)


# --------------------------------------------------------------------------------------------------
# 8. Validate Authoritative Training Row Counts
# --------------------------------------------------------------------------------------------------

TRAINING_ROW_COUNTS = {}


for dataset_id in DATASET_IDS:

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: authoritative training data "
            "is not available."
        )

    training_df = TRAINING_DATA[
        dataset_id
    ]

    if not isinstance(
        training_df,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: training data must be "
            "a pandas DataFrame."
        )

    expected_rows = int(
        len(training_df)
    )

    if expected_rows <= 0:

        raise RuntimeError(
            f"{dataset_id}: invalid training-row count: "
            f"{expected_rows}"
        )

    TRAINING_ROW_COUNTS[
        dataset_id
    ] = expected_rows

    print(
        f"✓ {dataset_id:<20} | "
        f"training rows = {expected_rows:>8,}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Initialize Validation Container
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_VALIDATION = []


# --------------------------------------------------------------------------------------------------
# 10. Validate Persisted Synthetic Sample Sizes
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# Synthetic files are loaded ONE AT A TIME.
#
# No complete synthetic-data collection is retained in memory.
#

for repetition in range(
    1,
    int(REPETITIONS) + 1,
):

    print()
    print("=" * 100)
    print(
        f"REPETITION {repetition} / {REPETITIONS}"
    )
    print("=" * 100)

    repetition_manifest = (
        GENERATION_MANIFEST_DF[
            GENERATION_MANIFEST_DF[
                "repetition"
            ].astype(int)
            == repetition
        ]
    )

    expected_repetition_records = (
        len(DATASET_IDS)
        * len(BASELINE_METHODS)
    )

    if len(
        repetition_manifest
    ) != expected_repetition_records:

        raise RuntimeError(
            f"Repetition {repetition}: expected "
            f"{expected_repetition_records} generation records, "
            f"found {len(repetition_manifest)}."
        )

    for _, manifest_record in (
        repetition_manifest
        .sort_values(
            by=[
                "dataset_id",
                "baseline",
            ]
        )
        .iterrows()
    ):

        dataset_id = manifest_record[
            "dataset_id"
        ]

        baseline_name = manifest_record[
            "baseline"
        ]

        print()
        print("-" * 100)
        print(
            f"VALIDATING SAMPLE SIZE — "
            f"{dataset_id} | "
            f"REPETITION {repetition} | "
            f"{baseline_name}"
        )
        print("-" * 100)

        # ------------------------------------------------------------------------------------------
        # 10.1 Validate dataset and baseline
        # ------------------------------------------------------------------------------------------

        if dataset_id not in DATASET_IDS:

            raise RuntimeError(
                f"Unknown dataset '{dataset_id}' "
                "found in generation manifest."
            )

        if baseline_name not in BASELINE_METHODS:

            raise RuntimeError(
                f"Unknown baseline '{baseline_name}' "
                "found in generation manifest."
            )

        # ------------------------------------------------------------------------------------------
        # 10.2 Authoritative expected sample size
        # ------------------------------------------------------------------------------------------

        expected_rows = TRAINING_ROW_COUNTS[
            dataset_id
        ]

        # ------------------------------------------------------------------------------------------
        # 10.3 Manifest training-row count
        # ------------------------------------------------------------------------------------------

        manifest_training_rows = int(
            manifest_record[
                "training_rows"
            ]
        )

        if manifest_training_rows != expected_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "manifest training-row count does not "
                "match authoritative TRAINING_DATA.\n"
                f"Expected: {expected_rows:,}\n"
                f"Manifest : {manifest_training_rows:,}"
            )

        # ------------------------------------------------------------------------------------------
        # 10.4 Manifest generated-row count
        # ------------------------------------------------------------------------------------------

        manifest_generated_rows = int(
            manifest_record[
                "rows_generated"
            ]
        )

        if manifest_generated_rows != expected_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "Section 09 manifest reports an incorrect "
                "synthetic sample size.\n"
                f"Expected: {expected_rows:,}\n"
                f"Manifest : {manifest_generated_rows:,}"
            )

        # ------------------------------------------------------------------------------------------
        # 10.5 Synthetic artifact path
        # ------------------------------------------------------------------------------------------

        synthetic_path = Path(
            manifest_record[
                "synthetic_artifact"
            ]
        )

        if not synthetic_path.exists():

            raise FileNotFoundError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "synthetic artifact does not exist:\n"
                f"{synthetic_path}"
            )

        # ------------------------------------------------------------------------------------------
        # 10.6 Load one synthetic artifact
        # ------------------------------------------------------------------------------------------

        synthetic_df = pd.read_csv(
            synthetic_path
        )

        # ------------------------------------------------------------------------------------------
        # 10.7 Actual sample size
        # ------------------------------------------------------------------------------------------

        actual_rows = int(
            len(synthetic_df)
        )

        # ------------------------------------------------------------------------------------------
        # 10.8 Exact sample-size validation
        # ------------------------------------------------------------------------------------------

        if actual_rows != expected_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "sample-size mismatch.\n"
                f"Expected: {expected_rows:,}\n"
                f"Found   : {actual_rows:,}"
            )

        # ------------------------------------------------------------------------------------------
        # 10.9 Record successful validation
        # ------------------------------------------------------------------------------------------

        SAMPLE_SIZE_VALIDATION.append(
            {
                "repetition": int(
                    repetition
                ),
                "repetition_seed": int(
                    manifest_record[
                        "repetition_seed"
                    ]
                ),
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "experiment_seed": int(
                    manifest_record[
                        "experiment_seed"
                    ]
                ),
                "expected_rows": int(
                    expected_rows
                ),
                "actual_rows": int(
                    actual_rows
                ),
                "difference": int(
                    actual_rows
                    - expected_rows
                ),
                "sample_size_pass": True,
                "status": "PASS",
            }
        )

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"{actual_rows:>8,} rows PASS"
        )

        # ------------------------------------------------------------------------------------------
        # 10.10 RAM cleanup
        # ------------------------------------------------------------------------------------------

        del synthetic_df

        gc.collect()


# --------------------------------------------------------------------------------------------------
# 11. Final Validation DataFrame
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_VALIDATION_DF = pd.DataFrame(
    SAMPLE_SIZE_VALIDATION
)


# --------------------------------------------------------------------------------------------------
# 12. Final Record-Count Gate
# --------------------------------------------------------------------------------------------------

if len(
    SAMPLE_SIZE_VALIDATION_DF
) != EXPECTED_VALIDATION_RECORDS:

    raise RuntimeError(
        "Unexpected number of sample-size validation records.\n"
        f"Expected: {EXPECTED_VALIDATION_RECORDS}\n"
        f"Found   : {len(SAMPLE_SIZE_VALIDATION_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 13. Experimental-Unit Uniqueness Gate
# --------------------------------------------------------------------------------------------------

validation_units = (
    SAMPLE_SIZE_VALIDATION_DF[
        [
            "repetition",
            "dataset_id",
            "baseline",
        ]
    ]
    .drop_duplicates()
)


if len(
    validation_units
) != EXPECTED_VALIDATION_RECORDS:

    raise RuntimeError(
        "Duplicate or missing experimental units detected "
        "in sample-size validation."
    )


# --------------------------------------------------------------------------------------------------
# 14. Final PASS Gate
# --------------------------------------------------------------------------------------------------

if not SAMPLE_SIZE_VALIDATION_DF[
    "sample_size_pass"
].all():

    raise RuntimeError(
        "One or more sample-size validations failed."
    )


# --------------------------------------------------------------------------------------------------
# 15. Difference Gate
# --------------------------------------------------------------------------------------------------

if not (
    SAMPLE_SIZE_VALIDATION_DF[
        "difference"
    ] == 0
).all():

    raise RuntimeError(
        "One or more synthetic datasets have a "
        "non-zero sample-size difference."
    )


# --------------------------------------------------------------------------------------------------
# 16. Persist Section 11 Results
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_VALIDATION_PATH = (
    MANIFEST_ROOT
    / "synthetic_sample_size_validation.csv"
)


SAMPLE_SIZE_VALIDATION_DF = (
    SAMPLE_SIZE_VALIDATION_DF
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


SAMPLE_SIZE_VALIDATION_DF.to_csv(
    SAMPLE_SIZE_VALIDATION_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 17. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 11 — SAMPLE SIZE VALIDATION SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets validated              : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset           : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                     : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected validation runs        : "
    f"{EXPECTED_VALIDATION_RECORDS}"
)

print(
    f"✓ Actual validation records       : "
    f"{len(SAMPLE_SIZE_VALIDATION_DF)}"
)

print(
    "✓ Training-row reference          : "
    "TRAINING_DATA"
)

print(
    "✓ Manifest training-row agreement : PASS"
)

print(
    "✓ Exact synthetic sample size     : PASS"
)

print(
    "✓ Zero row-count difference       : PASS"
)

print(
    "✓ 5-repetition coverage           : PASS"
)

print(
    "✓ 30-run experimental coverage    : PASS"
)

print(
    f"✓ Validation artifact             : "
    f"{SAMPLE_SIZE_VALIDATION_PATH}"
)

print()
print(
    "✓ Section 11 sample-size validation "
    "completed successfully."
)

print(
    "✓ Final RAM cleanup completed."
)

gc.collect()

SECTION 11 — VALIDATE SAMPLE SIZE

----------------------------------------------------------------------------------------------------
SECTION 09 ARTIFACT VALIDATION
----------------------------------------------------------------------------------------------------
✓ Generation manifest : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/manifests/baseline_generation_manifest.csv
✓ Generation manifest records : 30
✓ 5-repetition experimental coverage : PASS
✓ 30-run experimental coverage       : PASS
✓ adult_income         | training rows =   34,189
✓ bank_marketing       | training rows =   31,647
✓ diabetes_130us       | training rows =   71,236

REPETITION 1 / 5

----------------------------------------------------------------------------------------------------
VALIDATING SAMPLE SIZE — adult_income | REPETITION 1 | gaussian_copula
----------------------------------------------------------------------------------------------------
✓ adult_income         

0

In [86]:
# ==================================================================================================
# SECTION 12 — RECORD RUNTIME
# 30-RUN EXPERIMENTAL DESIGN
# AUTHORITATIVE SECTION 08/09 RUNTIME SCHEMAS
# ==================================================================================================

print("=" * 100)
print("SECTION 12 — RECORD RUNTIME")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import gc
from pathlib import Path

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "DATASET_IDS",
    "BASELINE_METHODS",
    "REPETITIONS",
    "FIT_RUNTIME_RECORDS",
    "GENERATION_RUNTIME_RECORDS",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:

    raise RuntimeError(
        "Section 12 is missing required runtime objects: "
        f"{missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if list(BASELINE_METHODS) != EXPECTED_BASELINES:

    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 "
        f"baseline registry.\n"
        f"Expected: {EXPECTED_BASELINES}\n"
        f"Found   : {list(BASELINE_METHODS)}"
    )


if int(REPETITIONS) != 5:

    raise RuntimeError(
        f"Notebook 04 requires 5 repetitions, found {REPETITIONS}."
    )


# --------------------------------------------------------------------------------------------------
# 4. Expected Experimental Unit
# --------------------------------------------------------------------------------------------------

EXPECTED_RUNTIME_RECORDS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)


if EXPECTED_RUNTIME_RECORDS != 30:

    raise RuntimeError(
        "Notebook 04 runtime design must contain exactly "
        f"30 experimental units. Found {EXPECTED_RUNTIME_RECORDS}."
    )


print()
print(
    f"✓ Datasets                    : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines                   : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                 : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected runtime records    : "
    f"{EXPECTED_RUNTIME_RECORDS}"
)


# --------------------------------------------------------------------------------------------------
# 5. Convert Runtime Records to DataFrames
# --------------------------------------------------------------------------------------------------

FIT_RUNTIME_DF = pd.DataFrame(
    FIT_RUNTIME_RECORDS
).copy()

GENERATION_RUNTIME_DF = pd.DataFrame(
    GENERATION_RUNTIME_RECORDS
).copy()


# --------------------------------------------------------------------------------------------------
# 6. Validate Runtime Record Counts
# --------------------------------------------------------------------------------------------------

if len(
    FIT_RUNTIME_DF
) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNTIME_RECORDS} fit runtime records, "
        f"found {len(FIT_RUNTIME_DF)}."
    )


if len(
    GENERATION_RUNTIME_DF
) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNTIME_RECORDS} generation runtime records, "
        f"found {len(GENERATION_RUNTIME_DF)}."
    )


print()
print(
    f"✓ Fit runtime records          : "
    f"{len(FIT_RUNTIME_DF)}"
)

print(
    f"✓ Generation runtime records   : "
    f"{len(GENERATION_RUNTIME_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 7. Validate Authoritative Fit Runtime Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_FIT_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "dataset_id",
    "dataset_index",
    "baseline",
    "baseline_index",
    "training_rows",
    "training_columns",
    "generative_columns",
    "target_column",
    "fit_data",
    "input_schema",
    "provenance_policy",
    "identifier_policy",
    "fit_runtime_seconds",
    "model_artifact_path",
    "model_artifact_size_bytes",
    "model_artifact_sha256",
    "status",
]

missing_fit_columns = [
    column
    for column in REQUIRED_FIT_COLUMNS
    if column not in FIT_RUNTIME_DF.columns
]

if missing_fit_columns:

    raise RuntimeError(
        "Missing required fit-runtime column(s): "
        f"{missing_fit_columns}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Validate Authoritative Generation Runtime Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_GENERATION_COLUMNS = [
    "repetition",
    "repetition_seed",
    "dataset_id",
    "baseline",
    "experiment_seed",
    "training_rows",
    "rows_requested",
    "rows_generated",
    "generative_columns",
    "target_column",
    "generation_runtime_seconds",
    "model_artifact",
    "model_sha256",
    "synthetic_artifact",
    "synthetic_sha256",
    "execution_status",
    "status",
]

missing_generation_columns = [
    column
    for column in REQUIRED_GENERATION_COLUMNS
    if column not in GENERATION_RUNTIME_DF.columns
]

if missing_generation_columns:

    raise RuntimeError(
        "Missing required generation-runtime column(s): "
        f"{missing_generation_columns}"
    )


print(
    "✓ Fit runtime schema           : PASS"
)

print(
    "✓ Generation runtime schema    : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9. Validate Runtime Key Uniqueness
# --------------------------------------------------------------------------------------------------

RUNTIME_KEYS = [
    "repetition",
    "dataset_id",
    "baseline",
    "experiment_seed",
]


if FIT_RUNTIME_DF.duplicated(
    subset=RUNTIME_KEYS
).any():

    raise RuntimeError(
        "Duplicate fit runtime records detected."
    )


if GENERATION_RUNTIME_DF.duplicated(
    subset=RUNTIME_KEYS
).any():

    raise RuntimeError(
        "Duplicate generation runtime records detected."
    )


print(
    "✓ Runtime key uniqueness       : PASS"
)


# --------------------------------------------------------------------------------------------------
# 10. Validate Repetition Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_REPETITIONS = set(
    range(
        1,
        int(REPETITIONS) + 1,
    )
)


actual_fit_repetitions = set(
    FIT_RUNTIME_DF[
        "repetition"
    ]
    .astype(int)
    .unique()
)

actual_generation_repetitions = set(
    GENERATION_RUNTIME_DF[
        "repetition"
    ]
    .astype(int)
    .unique()
)


if actual_fit_repetitions != EXPECTED_REPETITIONS:

    raise RuntimeError(
        "Fit runtime repetition coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_REPETITIONS)}\n"
        f"Found   : {sorted(actual_fit_repetitions)}"
    )


if actual_generation_repetitions != EXPECTED_REPETITIONS:

    raise RuntimeError(
        "Generation runtime repetition coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_REPETITIONS)}\n"
        f"Found   : {sorted(actual_generation_repetitions)}"
    )


print(
    "✓ Repetition coverage          : PASS"
)


# --------------------------------------------------------------------------------------------------
# 11. Validate Dataset/Baseline Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_UNITS = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


actual_fit_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in FIT_RUNTIME_DF.iterrows()
}


actual_generation_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_RUNTIME_DF.iterrows()
}


if actual_fit_units != EXPECTED_UNITS:

    missing_units = sorted(
        EXPECTED_UNITS
        - actual_fit_units
    )

    unexpected_units = sorted(
        actual_fit_units
        - EXPECTED_UNITS
    )

    raise RuntimeError(
        "Fit runtime experimental coverage mismatch.\n"
        f"Missing    : {missing_units}\n"
        f"Unexpected : {unexpected_units}"
    )


if actual_generation_units != EXPECTED_UNITS:

    missing_units = sorted(
        EXPECTED_UNITS
        - actual_generation_units
    )

    unexpected_units = sorted(
        actual_generation_units
        - EXPECTED_UNITS
    )

    raise RuntimeError(
        "Generation runtime experimental coverage mismatch.\n"
        f"Missing    : {missing_units}\n"
        f"Unexpected : {unexpected_units}"
    )


print(
    "✓ Dataset/baseline coverage    : PASS"
)

print(
    "✓ 30-run experimental coverage : PASS"
)


# --------------------------------------------------------------------------------------------------
# 12. Validate Repetition Seed Consistency
# --------------------------------------------------------------------------------------------------

fit_repetition_seed = {
    int(row["repetition"]): int(row["repetition_seed"])
    for _, row in FIT_RUNTIME_DF.iterrows()
}


generation_repetition_seed = {
    int(row["repetition"]): int(row["repetition_seed"])
    for _, row in GENERATION_RUNTIME_DF.iterrows()
}


if fit_repetition_seed != generation_repetition_seed:

    raise RuntimeError(
        "Fit and generation repetition-seed registries "
        "are inconsistent."
    )


# --------------------------------------------------------------------------------------------------
# 13. Validate Experiment Seed Consistency
# --------------------------------------------------------------------------------------------------

FIT_SEED_MAP = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    ): int(row["experiment_seed"])
    for _, row in FIT_RUNTIME_DF.iterrows()
}


GENERATION_SEED_MAP = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    ): int(row["experiment_seed"])
    for _, row in GENERATION_RUNTIME_DF.iterrows()
}


if FIT_SEED_MAP != GENERATION_SEED_MAP:

    raise RuntimeError(
        "Fit and generation experiment seeds are inconsistent."
    )


print(
    "✓ Repetition-seed consistency   : PASS"
)

print(
    "✓ Experiment-seed consistency   : PASS"
)


# --------------------------------------------------------------------------------------------------
# 14. Validate Runtime Numeric Fields
# --------------------------------------------------------------------------------------------------

FIT_NUMERIC_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "training_rows",
    "training_columns",
    "generative_columns",
    "fit_runtime_seconds",
    "model_artifact_size_bytes",
]


GENERATION_NUMERIC_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "training_rows",
    "rows_requested",
    "rows_generated",
    "generative_columns",
    "generation_runtime_seconds",
]


for column in FIT_NUMERIC_COLUMNS:

    FIT_RUNTIME_DF[column] = pd.to_numeric(
        FIT_RUNTIME_DF[column],
        errors="raise",
    )


for column in GENERATION_NUMERIC_COLUMNS:

    GENERATION_RUNTIME_DF[column] = pd.to_numeric(
        GENERATION_RUNTIME_DF[column],
        errors="raise",
    )


# --------------------------------------------------------------------------------------------------
# 15. Validate Runtime Values
# --------------------------------------------------------------------------------------------------

for column in [
    "fit_runtime_seconds",
]:

    if not np.isfinite(
        FIT_RUNTIME_DF[column]
    ).all():

        raise RuntimeError(
            f"Non-finite fit runtime value detected "
            f"in '{column}'."
        )

    if (
        FIT_RUNTIME_DF[column] < 0
    ).any():

        raise RuntimeError(
            f"Negative fit runtime value detected "
            f"in '{column}'."
        )


for column in [
    "generation_runtime_seconds",
]:

    if not np.isfinite(
        GENERATION_RUNTIME_DF[column]
    ).all():

        raise RuntimeError(
            f"Non-finite generation runtime value detected "
            f"in '{column}'."
        )

    if (
        GENERATION_RUNTIME_DF[column] < 0
    ).any():

        raise RuntimeError(
            f"Negative generation runtime value detected "
            f"in '{column}'."
        )


print(
    "✓ Runtime value validation      : PASS"
)


# --------------------------------------------------------------------------------------------------
# 16. Validate Training Rows Between Fit and Generation
# --------------------------------------------------------------------------------------------------

FIT_TRAINING_ROWS = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    ): int(row["training_rows"])
    for _, row in FIT_RUNTIME_DF.iterrows()
}


GENERATION_TRAINING_ROWS = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    ): int(row["training_rows"])
    for _, row in GENERATION_RUNTIME_DF.iterrows()
}


if FIT_TRAINING_ROWS != GENERATION_TRAINING_ROWS:

    raise RuntimeError(
        "Fit and generation training-row counts are inconsistent."
    )


# --------------------------------------------------------------------------------------------------
# 17. Validate Training Rows Against TRAINING_DATA
# --------------------------------------------------------------------------------------------------

if "TRAINING_DATA" not in globals():

    raise RuntimeError(
        "TRAINING_DATA is required for authoritative "
        "training-row validation."
    )


AUTHORITATIVE_TRAINING_ROWS = {}


for dataset_id in DATASET_IDS:

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: training data is not available."
        )

    training_df = TRAINING_DATA[
        dataset_id
    ]

    if not isinstance(
        training_df,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: training data must be "
            "a pandas DataFrame."
        )

    AUTHORITATIVE_TRAINING_ROWS[
        dataset_id
    ] = int(
        len(training_df)
    )


for (
    repetition,
    dataset_id,
    baseline,
), recorded_rows in GENERATION_TRAINING_ROWS.items():

    expected_rows = AUTHORITATIVE_TRAINING_ROWS[
        dataset_id
    ]

    if recorded_rows != expected_rows:

        raise RuntimeError(
            f"{dataset_id}/{baseline}/"
            f"repetition {repetition}: "
            "runtime training-row count does not match "
            "authoritative TRAINING_DATA.\n"
            f"Expected: {expected_rows:,}\n"
            f"Recorded: {recorded_rows:,}"
        )


print(
    "✓ Training-row reference        : PASS"
)


# --------------------------------------------------------------------------------------------------
# 18. Validate Requested vs Generated Rows
# --------------------------------------------------------------------------------------------------

if not (
    GENERATION_RUNTIME_DF[
        "rows_requested"
    ]
    ==
    GENERATION_RUNTIME_DF[
        "rows_generated"
    ]
).all():

    raise RuntimeError(
        "One or more generation runtime records have "
        "inconsistent requested/generated row counts."
    )


# --------------------------------------------------------------------------------------------------
# 19. Validate Generation Size Against Training Size
# --------------------------------------------------------------------------------------------------

for _, row in GENERATION_RUNTIME_DF.iterrows():

    dataset_id = row[
        "dataset_id"
    ]

    expected_rows = AUTHORITATIVE_TRAINING_ROWS[
        dataset_id
    ]

    if int(
        row["rows_requested"]
    ) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}/{row['baseline']}/"
            f"repetition {int(row['repetition'])}: "
            "requested generation size does not match "
            "training size.\n"
            f"Expected: {expected_rows:,}\n"
            f"Requested: {int(row['rows_requested']):,}"
        )


print(
    "✓ Requested/generated row count : PASS"
)

print(
    "✓ Training-size generation      : PASS"
)


# --------------------------------------------------------------------------------------------------
# 20. Merge Fit and Generation Runtime Records
# --------------------------------------------------------------------------------------------------

RUNTIME_DF = (
    FIT_RUNTIME_DF
    .merge(
        GENERATION_RUNTIME_DF[
            [
                "repetition",
                "dataset_id",
                "baseline",
                "experiment_seed",
                "rows_requested",
                "rows_generated",
                "generation_runtime_seconds",
                "synthetic_artifact",
                "synthetic_sha256",
                "execution_status",
            ]
        ],
        on=[
            "repetition",
            "dataset_id",
            "baseline",
            "experiment_seed",
        ],
        how="inner",
        validate="one_to_one",
    )
)


if len(
    RUNTIME_DF
) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNTIME_RECORDS} merged runtime records, "
        f"found {len(RUNTIME_DF)}."
    )


print(
    f"✓ Merged runtime records       : "
    f"{len(RUNTIME_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 21. Calculate Total Runtime
# --------------------------------------------------------------------------------------------------

RUNTIME_DF[
    "total_runtime_seconds"
] = (
    RUNTIME_DF[
        "fit_runtime_seconds"
    ]
    +
    RUNTIME_DF[
        "generation_runtime_seconds"
    ]
)


# --------------------------------------------------------------------------------------------------
# 22. Validate Total Runtime
# --------------------------------------------------------------------------------------------------

EXPECTED_TOTAL_RUNTIME = (
    RUNTIME_DF[
        "fit_runtime_seconds"
    ]
    +
    RUNTIME_DF[
        "generation_runtime_seconds"
    ]
)


if not np.allclose(
    RUNTIME_DF[
        "total_runtime_seconds"
    ],
    EXPECTED_TOTAL_RUNTIME,
    rtol=0,
    atol=1e-12,
):

    raise RuntimeError(
        "Total runtime calculation validation failed."
    )


print(
    "✓ Total-runtime calculation    : PASS"
)


# --------------------------------------------------------------------------------------------------
# 23. Select Final Runtime Schema
# --------------------------------------------------------------------------------------------------

RUNTIME_DF = RUNTIME_DF[
    [
        "repetition",
        "repetition_seed",
        "experiment_seed",
        "dataset_id",
        "dataset_index",
        "baseline",
        "baseline_index",
        "training_rows",
        "rows_requested",
        "rows_generated",
        "fit_runtime_seconds",
        "generation_runtime_seconds",
        "total_runtime_seconds",
        "execution_status",
        "model_artifact_path",
        "model_artifact_sha256",
        "synthetic_artifact",
        "synthetic_sha256",
    ]
].copy()


# --------------------------------------------------------------------------------------------------
# 24. Final Runtime Integrity Gate
# --------------------------------------------------------------------------------------------------

if len(
    RUNTIME_DF
) != EXPECTED_RUNTIME_RECORDS:

    raise RuntimeError(
        "Final runtime record-count validation failed."
    )


if RUNTIME_DF[
    "total_runtime_seconds"
].isna().any():

    raise RuntimeError(
        "Missing total runtime value detected."
    )


if RUNTIME_DF[
    "total_runtime_seconds"
].lt(0).any():

    raise RuntimeError(
        "Negative total runtime value detected."
    )


# --------------------------------------------------------------------------------------------------
# 25. Final Experimental-Unit Validation
# --------------------------------------------------------------------------------------------------

FINAL_RUNTIME_UNITS = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in RUNTIME_DF.iterrows()
}


if FINAL_RUNTIME_UNITS != EXPECTED_UNITS:

    raise RuntimeError(
        "Final runtime experimental-unit coverage failed."
    )


# --------------------------------------------------------------------------------------------------
# 26. Persist Combined Runtime Artifact
# --------------------------------------------------------------------------------------------------

RUNTIME_MANIFEST_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
    / "baselines"
    / "manifests"
)

RUNTIME_MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


RUNTIME_OUTPUT_PATH = (
    RUNTIME_MANIFEST_ROOT
    / "baseline_combined_runtime.csv"
)


RUNTIME_DF = (
    RUNTIME_DF
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


RUNTIME_DF.to_csv(
    RUNTIME_OUTPUT_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 27. Display Runtime Records
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("COMBINED RUNTIME RECORDS")
print("-" * 100)

print(
    RUNTIME_DF.to_string(
        index=False
    )
)


# --------------------------------------------------------------------------------------------------
# 28. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 12 — RUNTIME VALIDATION SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets                      : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset         : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                   : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected experimental units   : "
    f"{EXPECTED_RUNTIME_RECORDS}"
)

print(
    f"✓ Fit runtime records           : "
    f"{len(FIT_RUNTIME_DF)}"
)

print(
    f"✓ Generation runtime records    : "
    f"{len(GENERATION_RUNTIME_DF)}"
)

print(
    f"✓ Merged runtime records        : "
    f"{len(RUNTIME_DF)}"
)

print(
    "✓ Runtime schema validation     : PASS"
)

print(
    "✓ Runtime key uniqueness        : PASS"
)

print(
    "✓ Repetition coverage           : PASS"
)

print(
    "✓ Dataset/baseline coverage     : PASS"
)

print(
    "✓ Repetition-seed consistency   : PASS"
)

print(
    "✓ Experiment-seed consistency   : PASS"
)

print(
    "✓ Runtime value validation      : PASS"
)

print(
    "✓ Training-row reference        : PASS"
)

print(
    "✓ Row-count consistency         : PASS"
)

print(
    "✓ Training-size generation      : PASS"
)

print(
    "✓ Total-runtime calculation     : PASS"
)

print(
    "✓ 30-run experimental coverage  : PASS"
)

print(
    f"✓ Runtime artifact              : "
    f"{RUNTIME_OUTPUT_PATH}"
)

print()
print(
    "✓ Section 12 runtime recording "
    "completed successfully."
)


# --------------------------------------------------------------------------------------------------
# 29. RAM Cleanup
# --------------------------------------------------------------------------------------------------

gc.collect()

print(
    "✓ Final RAM cleanup completed."
)

SECTION 12 — RECORD RUNTIME

✓ Datasets                    : 3
✓ Baselines                   : 2
✓ Repetitions                 : 5
✓ Expected runtime records    : 30

✓ Fit runtime records          : 30
✓ Generation runtime records   : 30
✓ Fit runtime schema           : PASS
✓ Generation runtime schema    : PASS
✓ Runtime key uniqueness       : PASS
✓ Repetition coverage          : PASS
✓ Dataset/baseline coverage    : PASS
✓ 30-run experimental coverage : PASS
✓ Repetition-seed consistency   : PASS
✓ Experiment-seed consistency   : PASS
✓ Runtime value validation      : PASS
✓ Training-row reference        : PASS
✓ Requested/generated row count : PASS
✓ Training-size generation      : PASS
✓ Merged runtime records       : 30
✓ Total-runtime calculation    : PASS

----------------------------------------------------------------------------------------------------
COMBINED RUNTIME RECORDS
--------------------------------------------------------------------------------------------------

In [90]:
# ==================================================================================================
# SECTION 13 — SAVE / REGISTER SYNTHETIC DATA
# 30-RUN ARTIFACT REGISTRY
# AUTHORITATIVE SECTION 09 MANIFEST
# RAM-SAFE + RESTART-SAFE
# ==================================================================================================

print("=" * 100)
print("SECTION 13 — SAVE / REGISTER SYNTHETIC DATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import gc
import hashlib
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "PROJECT_ROOT",
    "DATASET_IDS",
    "BASELINE_METHODS",
    "REPETITIONS",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:

    raise RuntimeError(
        "Section 13 is missing required runtime objects: "
        f"{missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if list(BASELINE_METHODS) != EXPECTED_BASELINES:

    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 "
        f"baseline registry.\n"
        f"Expected: {EXPECTED_BASELINES}\n"
        f"Found   : {list(BASELINE_METHODS)}"
    )


if int(REPETITIONS) != 5:

    raise RuntimeError(
        f"Notebook 04 requires 5 repetitions, found {REPETITIONS}."
    )


EXPECTED_ARTIFACTS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)


if EXPECTED_ARTIFACTS != 30:

    raise RuntimeError(
        "Section 13 must register exactly 30 synthetic artifacts. "
        f"Found {EXPECTED_ARTIFACTS}."
    )


print()
print(
    f"✓ Datasets                    : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines                   : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                 : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected synthetic artifacts: "
    f"{EXPECTED_ARTIFACTS}"
)


# --------------------------------------------------------------------------------------------------
# 4. Authoritative Notebook 04 Paths
# --------------------------------------------------------------------------------------------------

NB04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

BASELINE_ROOT = (
    NB04_ROOT
    / "baselines"
)

NB04_SYNTHETIC_ROOT = (
    BASELINE_ROOT
    / "synthetic"
)

MANIFEST_ROOT = (
    BASELINE_ROOT
    / "manifests"
)

GENERATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "baseline_generation_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Authoritative Section 09 Manifest
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("SECTION 09 SYNTHETIC ARTIFACT REGISTRY")
print("-" * 100)

print(
    f"✓ Synthetic root     : "
    f"{NB04_SYNTHETIC_ROOT}"
)

print(
    f"✓ Generation manifest: "
    f"{GENERATION_MANIFEST_PATH}"
)


if not GENERATION_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "Authoritative Section 09 generation manifest was not found:\n"
        f"{GENERATION_MANIFEST_PATH}"
    )


GENERATION_MANIFEST_DF = pd.read_csv(
    GENERATION_MANIFEST_PATH
)


if len(
    GENERATION_MANIFEST_DF
) != EXPECTED_ARTIFACTS:

    raise RuntimeError(
        "Unexpected number of Section 09 synthetic generation records.\n"
        f"Expected: {EXPECTED_ARTIFACTS}\n"
        f"Found   : {len(GENERATION_MANIFEST_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Exact Authoritative Manifest Schema
# --------------------------------------------------------------------------------------------------

EXPECTED_MANIFEST_COLUMNS = [
    "repetition",
    "repetition_seed",
    "dataset_id",
    "baseline",
    "experiment_seed",
    "training_rows",
    "rows_generated",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "model_artifact",
    "model_sha256",
    "synthetic_artifact",
    "synthetic_sha256",
    "generation_runtime_seconds",
    "execution_status",
    "status",
]


if list(
    GENERATION_MANIFEST_DF.columns
) != EXPECTED_MANIFEST_COLUMNS:

    raise RuntimeError(
        "Section 09 generation manifest schema mismatch.\n"
        f"Expected: {EXPECTED_MANIFEST_COLUMNS}\n"
        f"Found   : {list(GENERATION_MANIFEST_DF.columns)}"
    )


print(
    f"✓ Manifest records : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print(
    "✓ Manifest schema   : PASS"
)


# --------------------------------------------------------------------------------------------------
# 7. Validate Experimental Unit Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_UNITS = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


ACTUAL_UNITS = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_MANIFEST_DF.iterrows()
}


if ACTUAL_UNITS != EXPECTED_UNITS:

    missing_units = sorted(
        EXPECTED_UNITS
        - ACTUAL_UNITS
    )

    unexpected_units = sorted(
        ACTUAL_UNITS
        - EXPECTED_UNITS
    )

    raise RuntimeError(
        "Section 09 synthetic artifact coverage mismatch.\n"
        f"Missing    : {missing_units}\n"
        f"Unexpected : {unexpected_units}"
    )


print(
    "✓ 5-repetition artifact coverage : PASS"
)

print(
    "✓ 30-run artifact coverage       : PASS"
)


# --------------------------------------------------------------------------------------------------
# 8. Validate Manifest Status
# --------------------------------------------------------------------------------------------------

if not (
    GENERATION_MANIFEST_DF[
        "status"
    ]
    .astype(str)
    .eq("PASS")
).all():

    failed_records = (
        GENERATION_MANIFEST_DF[
            ~GENERATION_MANIFEST_DF[
                "status"
            ]
            .astype(str)
            .eq("PASS")
        ]
    )

    raise RuntimeError(
        "Section 09 generation manifest contains "
        "non-PASS records.\n"
        f"{failed_records.to_string(index=False)}"
    )


print(
    "✓ Section 09 generation status : PASS"
)


# --------------------------------------------------------------------------------------------------
# 9. Validate Execution Status
# --------------------------------------------------------------------------------------------------

VALID_EXECUTION_STATUSES = {
    "GENERATED",
    "REUSED_EXISTING",
}


invalid_execution_statuses = set(
    GENERATION_MANIFEST_DF[
        "execution_status"
    ]
    .astype(str)
    .unique()
) - VALID_EXECUTION_STATUSES


if invalid_execution_statuses:

    raise RuntimeError(
        "Unexpected Section 09 execution status value(s): "
        f"{sorted(invalid_execution_statuses)}"
    )


print(
    "✓ Execution-status validation   : PASS"
)


# --------------------------------------------------------------------------------------------------
# 10. Reset Artifact Registry
# --------------------------------------------------------------------------------------------------

SYNTHETIC_ARTIFACT_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 11. Validate and Register Persisted Synthetic Artifacts
# --------------------------------------------------------------------------------------------------
#
# Each CSV is loaded ONE AT A TIME.
#
# Section 13 does NOT regenerate synthetic data.
# Section 09 is the authoritative producer.
#

for repetition in range(
    1,
    int(REPETITIONS) + 1,
):

    print()
    print("=" * 100)
    print(
        f"REPETITION {repetition} / {REPETITIONS}"
    )
    print("=" * 100)

    repetition_manifest = (
        GENERATION_MANIFEST_DF[
            GENERATION_MANIFEST_DF[
                "repetition"
            ].astype(int)
            == repetition
        ]
    )

    expected_repetition_records = (
        len(DATASET_IDS)
        * len(BASELINE_METHODS)
    )

    if len(
        repetition_manifest
    ) != expected_repetition_records:

        raise RuntimeError(
            f"Repetition {repetition}: expected "
            f"{expected_repetition_records} synthetic artifacts, "
            f"found {len(repetition_manifest)}."
        )


    for _, manifest_record in (
        repetition_manifest
        .sort_values(
            by=[
                "dataset_id",
                "baseline",
            ]
        )
        .iterrows()
    ):

        dataset_id = manifest_record[
            "dataset_id"
        ]

        baseline_name = manifest_record[
            "baseline"
        ]

        print()
        print("-" * 100)
        print(
            f"REGISTERING — {dataset_id} | "
            f"REPETITION {repetition} | "
            f"{baseline_name}"
        )
        print("-" * 100)


        # ------------------------------------------------------------------------------------------
        # 11.1 Validate dataset
        # ------------------------------------------------------------------------------------------

        if dataset_id not in DATASET_IDS:

            raise RuntimeError(
                f"Unknown dataset '{dataset_id}' "
                "found in Section 09 manifest."
            )


        # ------------------------------------------------------------------------------------------
        # 11.2 Validate baseline
        # ------------------------------------------------------------------------------------------

        if baseline_name not in BASELINE_METHODS:

            raise RuntimeError(
                f"Unknown baseline '{baseline_name}' "
                "found in Section 09 manifest."
            )


        # ------------------------------------------------------------------------------------------
        # 11.3 Authoritative schema
        # ------------------------------------------------------------------------------------------

        expected_generative_columns = list(
            TRAINING_GENERATIVE_COLUMNS[
                dataset_id
            ]
        )

        target_column = (
            TRAINING_TARGET_COLUMNS[
                dataset_id
            ]
        )

        provenance_column = (
            TRAINING_PROVENANCE_COLUMNS[
                dataset_id
            ]
        )

        identifier_columns = list(
            TRAINING_IDENTIFIER_COLUMNS[
                dataset_id
            ]
        )


        # ------------------------------------------------------------------------------------------
        # 11.4 Validate training-row count
        # ------------------------------------------------------------------------------------------

        training_rows = int(
            manifest_record[
                "training_rows"
            ]
        )

        rows_generated = int(
            manifest_record[
                "rows_generated"
            ]
        )

        if training_rows <= 0:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "invalid training-row count."
            )


        if rows_generated != training_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "rows_generated does not equal training_rows.\n"
                f"Training rows : {training_rows:,}\n"
                f"Generated rows: {rows_generated:,}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.5 Validate generative-column count
        # ------------------------------------------------------------------------------------------

        manifest_columns = int(
            manifest_record[
                "generative_columns"
            ]
        )

        if manifest_columns != len(
            expected_generative_columns
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "generative-column count mismatch.\n"
                f"Manifest: {manifest_columns}\n"
                f"Expected: {len(expected_generative_columns)}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.6 Validate target metadata
        # ------------------------------------------------------------------------------------------

        manifest_target = str(
            manifest_record[
                "target_column"
            ]
        )

        if manifest_target != str(
            target_column
        ):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "target-column metadata mismatch.\n"
                f"Manifest: {manifest_target}\n"
                f"Expected: {target_column}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.7 Validate provenance metadata
        # ------------------------------------------------------------------------------------------

        manifest_provenance = str(
            manifest_record[
                "provenance_column"
            ]
        )

        expected_provenance = (
            str(provenance_column)
            if provenance_column is not None
            else "None"
        )

        if manifest_provenance != expected_provenance:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "provenance-column metadata mismatch.\n"
                f"Manifest: {manifest_provenance}\n"
                f"Expected: {expected_provenance}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.8 Synthetic artifact path
        # ------------------------------------------------------------------------------------------

        artifact_path = Path(
            manifest_record[
                "synthetic_artifact"
            ]
        )


        if not artifact_path.exists():

            raise FileNotFoundError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "synthetic artifact does not exist:\n"
                f"{artifact_path}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.9 Enforce authoritative synthetic root
        # ------------------------------------------------------------------------------------------

        try:

            artifact_path.resolve().relative_to(
                NB04_SYNTHETIC_ROOT.resolve()
            )

        except ValueError:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "synthetic artifact is outside the "
                "authoritative Notebook 04 synthetic root.\n"
                f"Artifact: {artifact_path}\n"
                f"Expected root: {NB04_SYNTHETIC_ROOT}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.10 Validate file size
        # ------------------------------------------------------------------------------------------

        file_size_bytes = int(
            artifact_path.stat().st_size
        )

        if file_size_bytes <= 0:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "synthetic artifact is empty."
            )


        # ------------------------------------------------------------------------------------------
        # 11.11 Validate SHA-256
        # ------------------------------------------------------------------------------------------

        digest = hashlib.sha256()

        with artifact_path.open(
            "rb"
        ) as handle:

            for chunk in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):

                digest.update(
                    chunk
                )

        calculated_sha256 = (
            digest.hexdigest()
        )

        recorded_sha256 = str(
            manifest_record[
                "synthetic_sha256"
            ]
        )

        if calculated_sha256 != recorded_sha256:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "synthetic artifact SHA-256 mismatch.\n"
                f"Manifest : {recorded_sha256}\n"
                f"Calculated: {calculated_sha256}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.12 Load persisted synthetic data
        # ------------------------------------------------------------------------------------------

        synthetic_df = pd.read_csv(
            artifact_path
        )


        # ------------------------------------------------------------------------------------------
        # 11.13 Validate exact column order
        # ------------------------------------------------------------------------------------------

        if list(
            synthetic_df.columns
        ) != expected_generative_columns:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "persisted synthetic schema does not match "
                "the frozen generative schema."
            )


        # ------------------------------------------------------------------------------------------
        # 11.14 Validate actual row count
        # ------------------------------------------------------------------------------------------

        actual_rows = int(
            len(synthetic_df)
        )

        if actual_rows != training_rows:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "persisted synthetic row count mismatch.\n"
                f"Expected: {training_rows:,}\n"
                f"Found   : {actual_rows:,}"
            )


        if actual_rows != rows_generated:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "persisted row count does not match "
                "Section 09 rows_generated."
            )


        # ------------------------------------------------------------------------------------------
        # 11.15 Validate target retention
        # ------------------------------------------------------------------------------------------

        if target_column not in synthetic_df.columns:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                f"target column '{target_column}' is missing."
            )


        # ------------------------------------------------------------------------------------------
        # 11.16 Validate provenance exclusion
        # ------------------------------------------------------------------------------------------

        if provenance_column is not None:

            if provenance_column in synthetic_df.columns:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "provenance leakage detected."
                )


        # ------------------------------------------------------------------------------------------
        # 11.17 Validate identifier exclusion
        # ------------------------------------------------------------------------------------------

        identifier_leakage = (
            set(identifier_columns)
            .intersection(
                synthetic_df.columns
            )
        )

        if identifier_leakage:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}/"
                f"repetition {repetition}: "
                "identifier leakage detected: "
                f"{sorted(identifier_leakage)}"
            )


        # ------------------------------------------------------------------------------------------
        # 11.18 Record artifact
        # ------------------------------------------------------------------------------------------

        SYNTHETIC_ARTIFACT_RECORDS.append(
            {
                "repetition": int(
                    repetition
                ),
                "repetition_seed": int(
                    manifest_record[
                        "repetition_seed"
                    ]
                ),
                "experiment_seed": int(
                    manifest_record[
                        "experiment_seed"
                    ]
                ),
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "artifact_type": "synthetic_data",
                "path": str(
                    artifact_path
                ),
                "relative_path": str(
                    artifact_path.relative_to(
                        NB04_ROOT
                    )
                ),
                "rows": actual_rows,
                "columns": len(
                    synthetic_df.columns
                ),
                "target_column": target_column,
                "provenance_column": provenance_column,
                "identifier_columns": ",".join(
                    identifier_columns
                ),
                "provenance_excluded": True,
                "identifiers_excluded": True,
                "file_size_bytes": file_size_bytes,
                "sha256": calculated_sha256,
                "execution_status": str(
                    manifest_record[
                        "execution_status"
                    ]
                ),
                "status": "PASS",
            }
        )


        # ------------------------------------------------------------------------------------------
        # 11.19 Progress output
        # ------------------------------------------------------------------------------------------

        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"{actual_rows:>7,} rows | "
            f"{len(synthetic_df.columns):>3} cols | "
            f"SHA256 PASS | "
            f"{manifest_record['execution_status']}"
        )


        # ------------------------------------------------------------------------------------------
        # 11.20 RAM cleanup
        # ------------------------------------------------------------------------------------------

        del synthetic_df

        gc.collect()


# --------------------------------------------------------------------------------------------------
# 12. Create Final Artifact Registry
# --------------------------------------------------------------------------------------------------

SYNTHETIC_ARTIFACT_DF = pd.DataFrame(
    SYNTHETIC_ARTIFACT_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 13. Validate Final Artifact Count
# --------------------------------------------------------------------------------------------------

if len(
    SYNTHETIC_ARTIFACT_DF
) != EXPECTED_ARTIFACTS:

    raise RuntimeError(
        "Unexpected number of registered synthetic artifacts.\n"
        f"Expected: {EXPECTED_ARTIFACTS}\n"
        f"Found   : {len(SYNTHETIC_ARTIFACT_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 14. Validate Artifact Registry Uniqueness
# --------------------------------------------------------------------------------------------------

ARTIFACT_KEYS = [
    "repetition",
    "dataset_id",
    "baseline",
]


if SYNTHETIC_ARTIFACT_DF.duplicated(
    subset=ARTIFACT_KEYS
).any():

    raise RuntimeError(
        "Duplicate synthetic artifact records detected."
    )


# --------------------------------------------------------------------------------------------------
# 15. Validate Final Artifact Coverage
# --------------------------------------------------------------------------------------------------

actual_artifact_keys = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in SYNTHETIC_ARTIFACT_DF.iterrows()
}


if actual_artifact_keys != EXPECTED_UNITS:

    raise RuntimeError(
        "Final synthetic artifact coverage mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 16. Validate Every Artifact Passed
# --------------------------------------------------------------------------------------------------

if not (
    SYNTHETIC_ARTIFACT_DF[
        "status"
    ]
    == "PASS"
).all():

    raise RuntimeError(
        "One or more synthetic artifact registrations failed."
    )


if not (
    SYNTHETIC_ARTIFACT_DF[
        "provenance_excluded"
    ]
).all():

    raise RuntimeError(
        "Provenance-exclusion validation failed."
    )


if not (
    SYNTHETIC_ARTIFACT_DF[
        "identifiers_excluded"
    ]
).all():

    raise RuntimeError(
        "Identifier-exclusion validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 17. Persist Artifact Registry
# --------------------------------------------------------------------------------------------------

SYNTHETIC_ARTIFACT_REGISTRY_PATH = (
    MANIFEST_ROOT
    / "synthetic_artifact_registry.csv"
)


SYNTHETIC_ARTIFACT_DF = (
    SYNTHETIC_ARTIFACT_DF
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


SYNTHETIC_ARTIFACT_DF.to_csv(
    SYNTHETIC_ARTIFACT_REGISTRY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 18. Reload Registry for Persistence Verification
# --------------------------------------------------------------------------------------------------

RELOADED_ARTIFACT_REGISTRY_DF = pd.read_csv(
    SYNTHETIC_ARTIFACT_REGISTRY_PATH
)


if len(
    RELOADED_ARTIFACT_REGISTRY_DF
) != EXPECTED_ARTIFACTS:

    raise RuntimeError(
        "Persisted synthetic artifact registry "
        "record-count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 19. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 13 — SYNTHETIC DATA REGISTRATION SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets                       : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset          : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                    : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected synthetic artifacts  : "
    f"{EXPECTED_ARTIFACTS}"
)

print(
    f"✓ Registered synthetic artifacts: "
    f"{len(SYNTHETIC_ARTIFACT_DF)}"
)

print(
    "✓ Section 09 manifest schema          : PASS"
)

print(
    "✓ Section 09 generation status        : PASS"
)

print(
    "✓ 5-repetition artifact coverage     : PASS"
)

print(
    "✓ 30-run artifact coverage           : PASS"
)

print(
    "✓ SHA-256 integrity                  : PASS"
)

print(
    "✓ Exact generative schema             : PASS"
)

print(
    "✓ Training/generation row agreement  : PASS"
)

print(
    "✓ Target retention                   : PASS"
)

print(
    "✓ Provenance exclusion               : PASS"
)

print(
    "✓ Identifier exclusion               : PASS"
)

print(
    "✓ Artifact registry uniqueness       : PASS"
)

print(
    f"✓ Artifact registry                 : "
    f"{SYNTHETIC_ARTIFACT_REGISTRY_PATH}"
)

print()
print(
    "✓ SECTION 13 — SYNTHETIC DATA "
    "REGISTRATION: PASS"
)


# --------------------------------------------------------------------------------------------------
# 20. Final RAM Cleanup
# --------------------------------------------------------------------------------------------------

del RELOADED_ARTIFACT_REGISTRY_DF
del GENERATION_MANIFEST_DF

gc.collect()

print(
    "✓ Final RAM cleanup completed."
)

SECTION 13 — SAVE / REGISTER SYNTHETIC DATA

✓ Datasets                    : 3
✓ Baselines                   : 2
✓ Repetitions                 : 5
✓ Expected synthetic artifacts: 30

----------------------------------------------------------------------------------------------------
SECTION 09 SYNTHETIC ARTIFACT REGISTRY
----------------------------------------------------------------------------------------------------
✓ Synthetic root     : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/synthetic
✓ Generation manifest: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04/baselines/manifests/baseline_generation_manifest.csv
✓ Manifest records : 30
✓ Manifest schema   : PASS
✓ 5-repetition artifact coverage : PASS
✓ 30-run artifact coverage       : PASS
✓ Section 09 generation status : PASS
✓ Execution-status validation   : PASS

REPETITION 1 / 5

------------------------------------------------------------------------------------------------

In [92]:
# ==================================================================================================
# SECTION 14 — SAVE BASELINE METADATA
# CANONICAL BASELINE METADATA + 30-RUN EXPERIMENT METADATA REGISTRY
# ==================================================================================================

print("=" * 100)
print("SECTION 14 — SAVE BASELINE METADATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import gc
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "PROJECT_ROOT",
    "DATASET_IDS",
    "BASELINE_METHODS",
    "REPETITIONS",
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_FEATURE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
    "BASELINE_DEFINITIONS",
    "RUNTIME_DF",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:

    raise RuntimeError(
        "Section 14 is missing required runtime objects: "
        f"{missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Validate Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if list(BASELINE_METHODS) != EXPECTED_BASELINES:

    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 "
        f"baseline registry.\n"
        f"Expected: {EXPECTED_BASELINES}\n"
        f"Found   : {list(BASELINE_METHODS)}"
    )


if int(REPETITIONS) != 5:

    raise RuntimeError(
        f"Notebook 04 requires 5 repetitions, found {REPETITIONS}."
    )


EXPECTED_RUNS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)

EXPECTED_BASELINE_METADATA = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)


if EXPECTED_RUNS != 30:

    raise RuntimeError(
        "Expected exactly 30 experimental runs. "
        f"Found {EXPECTED_RUNS}."
    )


if EXPECTED_BASELINE_METADATA != 6:

    raise RuntimeError(
        "Expected exactly 6 canonical dataset × baseline metadata "
        f"records. Found {EXPECTED_BASELINE_METADATA}."
    )


print()
print(
    f"✓ Datasets                    : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines                   : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                 : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected experiment runs    : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Canonical baseline metadata : "
    f"{EXPECTED_BASELINE_METADATA}"
)


# --------------------------------------------------------------------------------------------------
# 4. Authoritative Notebook 04 Paths
# --------------------------------------------------------------------------------------------------

NB04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

BASELINE_ROOT = (
    NB04_ROOT
    / "baselines"
)

METADATA_ROOT = (
    BASELINE_ROOT
    / "metadata"
)

MANIFEST_ROOT = (
    BASELINE_ROOT
    / "manifests"
)

SYNTHETIC_ARTIFACT_REGISTRY_PATH = (
    MANIFEST_ROOT
    / "synthetic_artifact_registry.csv"
)


METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Section 13 Artifact Registry
# --------------------------------------------------------------------------------------------------

if not SYNTHETIC_ARTIFACT_REGISTRY_PATH.exists():

    raise FileNotFoundError(
        "Section 13 synthetic artifact registry was not found:\n"
        f"{SYNTHETIC_ARTIFACT_REGISTRY_PATH}"
    )


SYNTHETIC_ARTIFACT_REGISTRY_DF = pd.read_csv(
    SYNTHETIC_ARTIFACT_REGISTRY_PATH
)


REQUIRED_ARTIFACT_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "dataset_id",
    "baseline",
    "path",
    "rows",
    "columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "sha256",
    "status",
]


missing_artifact_columns = [
    column
    for column in REQUIRED_ARTIFACT_COLUMNS
    if column not in SYNTHETIC_ARTIFACT_REGISTRY_DF.columns
]


if missing_artifact_columns:

    raise RuntimeError(
        "Section 13 synthetic artifact registry is missing "
        f"column(s): {missing_artifact_columns}"
    )


if len(
    SYNTHETIC_ARTIFACT_REGISTRY_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Section 13 synthetic artifact registry must contain "
        f"{EXPECTED_RUNS} records.\n"
        f"Found: {len(SYNTHETIC_ARTIFACT_REGISTRY_DF)}"
    )


print()
print(
    f"✓ Section 13 artifact registry : "
    f"{len(SYNTHETIC_ARTIFACT_REGISTRY_DF)} records"
)


# --------------------------------------------------------------------------------------------------
# 6. Validate Runtime DataFrame
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "dataset_id",
    "baseline",
    "training_rows",
    "rows_requested",
    "rows_generated",
    "fit_runtime_seconds",
    "generation_runtime_seconds",
    "total_runtime_seconds",
    "model_artifact_path",
    "model_artifact_sha256",
    "synthetic_artifact",
    "synthetic_sha256",
]


missing_runtime_columns = [
    column
    for column in REQUIRED_RUNTIME_COLUMNS
    if column not in RUNTIME_DF.columns
]


if missing_runtime_columns:

    raise RuntimeError(
        "RUNTIME_DF is missing required column(s): "
        f"{missing_runtime_columns}"
    )


if len(
    RUNTIME_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        f"RUNTIME_DF must contain {EXPECTED_RUNS} records. "
        f"Found {len(RUNTIME_DF)}."
    )


print(
    f"✓ Runtime registry             : "
    f"{len(RUNTIME_DF)} records"
)


# --------------------------------------------------------------------------------------------------
# 7. Validate Runtime / Artifact Experimental Units
# --------------------------------------------------------------------------------------------------

RUNTIME_KEYS = [
    "repetition",
    "dataset_id",
    "baseline",
    "experiment_seed",
]


runtime_keys = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
        int(row["experiment_seed"]),
    )
    for _, row in RUNTIME_DF.iterrows()
}


artifact_keys = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
        int(row["experiment_seed"]),
    )
    for _, row in SYNTHETIC_ARTIFACT_REGISTRY_DF.iterrows()
}


if runtime_keys != artifact_keys:

    raise RuntimeError(
        "Runtime and synthetic artifact experimental-unit "
        "coverage do not match."
    )


print(
    "✓ Runtime/artifact unit linkage : PASS"
)


# --------------------------------------------------------------------------------------------------
# 8. Initialize Metadata Registries
# --------------------------------------------------------------------------------------------------

BASELINE_METADATA_RECORDS = []

EXPERIMENT_METADATA_RECORDS = []


SYNTHETIC_SAMPLE_POLICY = (
    "training_rows"
)


# --------------------------------------------------------------------------------------------------
# 9. Generate Canonical Dataset × Baseline Metadata
# --------------------------------------------------------------------------------------------------

for dataset_index, dataset_id in enumerate(
    DATASET_IDS
):

    # ----------------------------------------------------------------------------------------------
    # 9.1 Training Data
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in TRAINING_DATA:

        raise RuntimeError(
            f"{dataset_id}: training data is not available."
        )


    train_df = TRAINING_DATA[
        dataset_id
    ]


    if not isinstance(
        train_df,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{dataset_id}: training data must be a pandas DataFrame."
        )


    # ----------------------------------------------------------------------------------------------
    # 9.2 Authoritative Schema
    # ----------------------------------------------------------------------------------------------

    generative_columns = list(
        TRAINING_GENERATIVE_COLUMNS[
            dataset_id
        ]
    )

    feature_columns = list(
        TRAINING_FEATURE_COLUMNS[
            dataset_id
        ]
    )

    target = (
        TRAINING_TARGET_COLUMNS[
            dataset_id
        ]
    )

    provenance = (
        TRAINING_PROVENANCE_COLUMNS[
            dataset_id
        ]
    )

    identifiers = list(
        TRAINING_IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # 9.3 Validate Training Schema
    # ----------------------------------------------------------------------------------------------

    expected_training_columns = (
        [provenance]
        + generative_columns
    )


    if list(
        train_df.columns
    ) != expected_training_columns:

        raise RuntimeError(
            f"{dataset_id}: training schema mismatch.\n"
            f"Expected: {expected_training_columns}\n"
            f"Found   : {list(train_df.columns)}"
        )


    # ----------------------------------------------------------------------------------------------
    # 9.4 Determine Numeric / Categorical Columns
    # ----------------------------------------------------------------------------------------------

    numeric_columns = [
        column
        for column in generative_columns
        if pd.api.types.is_numeric_dtype(
            train_df[column]
        )
    ]


    categorical_columns = [
        column
        for column in generative_columns
        if column not in numeric_columns
    ]


    # ----------------------------------------------------------------------------------------------
    # 9.5 Dataset × Baseline
    # ----------------------------------------------------------------------------------------------

    for baseline_index, baseline_name in enumerate(
        BASELINE_METHODS
    ):

        if baseline_name not in BASELINE_DEFINITIONS:

            raise RuntimeError(
                f"Missing baseline definition for "
                f"'{baseline_name}'."
            )


        baseline_definition = (
            BASELINE_DEFINITIONS[
                baseline_name
            ]
        )


        # ------------------------------------------------------------------------------------------
        # 9.5.1 Retrieve All Repetition Runs
        # ------------------------------------------------------------------------------------------

        run_records = (
            RUNTIME_DF[
                (
                    RUNTIME_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    RUNTIME_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ]
            .sort_values(
                by=[
                    "repetition"
                ]
            )
            .reset_index(
                drop=True
            )
        )


        if len(
            run_records
        ) != int(REPETITIONS):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                f"expected {REPETITIONS} runtime records, "
                f"found {len(run_records)}."
            )


        # ------------------------------------------------------------------------------------------
        # 9.5.2 Validate Repetition Coverage
        # ------------------------------------------------------------------------------------------

        actual_repetitions = set(
            run_records[
                "repetition"
            ]
            .astype(int)
            .tolist()
        )


        expected_repetitions = set(
            range(
                1,
                int(REPETITIONS) + 1,
            )
        )


        if actual_repetitions != expected_repetitions:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "repetition coverage mismatch.\n"
                f"Expected: {sorted(expected_repetitions)}\n"
                f"Found   : {sorted(actual_repetitions)}"
            )


        # ------------------------------------------------------------------------------------------
        # 9.5.3 Canonical Metadata Object
        # ------------------------------------------------------------------------------------------

        metadata = {

            "notebook": {
                "id": NOTEBOOK_ID,
                "name": NOTEBOOK_NAME,
                "version": NOTEBOOK_VERSION,
            },

            "metadata_type": (
                "canonical_baseline_metadata"
            ),

            "dataset_id": dataset_id,

            "baseline": {
                "id": baseline_name,
                "name": baseline_definition[
                    "name"
                ],
                "family": baseline_definition[
                    "family"
                ],
                "definition": baseline_definition[
                    "principle"
                ],
                "dependency_model": baseline_definition[
                    "dependency_model"
                ],
                "formal_dp": False,
            },

            "data_source": {
                "notebook": "02",
                "layer": "native",
                "fit_split": "train",
                "validation_used": False,
                "test_used": False,
                "synthetic_data_used_for_fit": False,
            },

            "fit_policy": {
                "fit_data": "native_train_only",
                "fit_scope": "training_dataset_only",
                "validation_used": False,
                "test_used": False,
                "synthetic_data_used": False,
            },

            "schema": {
                "generative_columns": generative_columns,
                "feature_columns": feature_columns,
                "numeric_columns": numeric_columns,
                "categorical_columns": categorical_columns,
                "target_column": target,
                "provenance_column": provenance,
                "identifier_columns": identifiers,
                "target_retained_in_generative_schema": True,
                "target_used_as_predictor": False,
                "provenance_used": False,
                "identifiers_used": False,
            },

            "leakage_policy": {
                "target_used_as_predictor": False,
                "provenance_used": False,
                "identifiers_used": False,
            },

            "sampling": {
                "policy": SYNTHETIC_SAMPLE_POLICY,
                "training_rows": int(
                    len(train_df)
                ),
                "repetitions": int(
                    REPETITIONS
                ),
            },

            "reproducibility": {
                "master_seed": MASTER_SEED,
                "repetition_seed_policy": (
                    "Notebook_00_authoritative_repetition_seed"
                ),
                "experiment_seed_policy": (
                    "Notebook_04_Section_07_experiment_seed"
                ),
            },

            "repetition_runs": [],

            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        }


        # ------------------------------------------------------------------------------------------
        # 9.5.4 Link the Five Repetition Runs
        # ------------------------------------------------------------------------------------------

        for _, run in run_records.iterrows():

            repetition = int(
                run["repetition"]
            )

            repetition_seed = int(
                run["repetition_seed"]
            )

            experiment_seed = int(
                run["experiment_seed"]
            )


            synthetic_artifact = str(
                run["synthetic_artifact"]
            )


            model_artifact = str(
                run["model_artifact_path"]
            )


            run_metadata = {

                "repetition": repetition,

                "repetition_seed": repetition_seed,

                "experiment_seed": experiment_seed,

                "training_rows": int(
                    run["training_rows"]
                ),

                "rows_requested": int(
                    run["rows_requested"]
                ),

                "rows_generated": int(
                    run["rows_generated"]
                ),

                "fit_runtime_seconds": float(
                    run["fit_runtime_seconds"]
                ),

                "generation_runtime_seconds": float(
                    run["generation_runtime_seconds"]
                ),

                "total_runtime_seconds": float(
                    run["total_runtime_seconds"]
                ),

                "model_artifact": model_artifact,

                "model_artifact_sha256": str(
                    run["model_artifact_sha256"]
                ),

                "synthetic_artifact": synthetic_artifact,

                "synthetic_artifact_sha256": str(
                    run["synthetic_sha256"]
                ),
            }


            metadata[
                "repetition_runs"
            ].append(
                run_metadata
            )


            # --------------------------------------------------------------------------------------
            # Experiment Metadata Registry Record
            # --------------------------------------------------------------------------------------

            EXPERIMENT_METADATA_RECORDS.append(
                {
                    "repetition": repetition,
                    "repetition_seed": repetition_seed,
                    "experiment_seed": experiment_seed,
                    "dataset_id": dataset_id,
                    "baseline": baseline_name,
                    "dataset_index": dataset_index,
                    "baseline_index": baseline_index,
                    "training_rows": int(
                        run["training_rows"]
                    ),
                    "rows_requested": int(
                        run["rows_requested"]
                    ),
                    "rows_generated": int(
                        run["rows_generated"]
                    ),
                    "model_artifact": model_artifact,
                    "model_artifact_sha256": str(
                        run["model_artifact_sha256"]
                    ),
                    "synthetic_artifact": synthetic_artifact,
                    "synthetic_artifact_sha256": str(
                        run["synthetic_sha256"]
                    ),
                    "fit_runtime_seconds": float(
                        run["fit_runtime_seconds"]
                    ),
                    "generation_runtime_seconds": float(
                        run["generation_runtime_seconds"]
                    ),
                    "total_runtime_seconds": float(
                        run["total_runtime_seconds"]
                    ),
                    "metadata_type": (
                        "canonical_baseline_metadata"
                    ),
                }
            )


        # ------------------------------------------------------------------------------------------
        # 9.5.5 Metadata Output Path
        # ------------------------------------------------------------------------------------------

        metadata_path = (
            METADATA_ROOT
            / dataset_id
            / f"{baseline_name}.json"
        )


        metadata_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )


        # ------------------------------------------------------------------------------------------
        # 9.5.6 Save Canonical Metadata
        # ------------------------------------------------------------------------------------------

        with metadata_path.open(
            "w",
            encoding="utf-8",
        ) as handle:

            json.dump(
                metadata,
                handle,
                indent=2,
                ensure_ascii=False,
                default=str,
            )


        # ------------------------------------------------------------------------------------------
        # 9.5.7 Validate Persistence
        # ------------------------------------------------------------------------------------------

        if not metadata_path.exists():

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "metadata file was not created."
            )


        if metadata_path.stat().st_size <= 0:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "metadata file is empty."
            )


        # ------------------------------------------------------------------------------------------
        # 9.5.8 Reload Metadata
        # ------------------------------------------------------------------------------------------

        with metadata_path.open(
            "r",
            encoding="utf-8",
        ) as handle:

            reloaded_metadata = json.load(
                handle
            )


        if reloaded_metadata[
            "dataset_id"
        ] != dataset_id:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "persisted metadata dataset_id mismatch."
            )


        if reloaded_metadata[
            "baseline"
        ][
            "id"
        ] != baseline_name:

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "persisted metadata baseline mismatch."
            )


        if len(
            reloaded_metadata[
                "repetition_runs"
            ]
        ) != int(REPETITIONS):

            raise RuntimeError(
                f"{dataset_id}/{baseline_name}: "
                "metadata repetition-run count mismatch."
            )


        # ------------------------------------------------------------------------------------------
        # 9.5.9 Record Canonical Metadata Artifact
        # ------------------------------------------------------------------------------------------

        BASELINE_METADATA_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "baseline": baseline_name,
                "metadata_type": (
                    "canonical_baseline_metadata"
                ),
                "metadata_path": str(
                    metadata_path
                ),
                "metadata_relative_path": str(
                    metadata_path.relative_to(
                        NB04_ROOT
                    )
                ),
                "metadata_size_bytes": int(
                    metadata_path.stat().st_size
                ),
                "metadata_sha256": calculate_sha256(
                    metadata_path
                ),
                "repetitions": int(
                    REPETITIONS
                ),
                "training_rows": int(
                    len(train_df)
                ),
                "generative_columns": int(
                    len(generative_columns)
                ),
                "target_column": target,
                "provenance_column": provenance,
                "identifier_columns": ",".join(
                    identifiers
                ),
                "fit_data": "native_train_only",
                "validation_used": False,
                "test_used": False,
                "status": "PASS",
            }
        )


        print(
            f"✓ {dataset_id:<20} | "
            f"{baseline_name:<24} | "
            f"canonical metadata saved | "
            f"{REPETITIONS} repetitions linked"
        )


# --------------------------------------------------------------------------------------------------
# 10. Create Canonical Metadata Registry
# --------------------------------------------------------------------------------------------------

BASELINE_METADATA_DF = pd.DataFrame(
    BASELINE_METADATA_RECORDS
)


if len(
    BASELINE_METADATA_DF
) != EXPECTED_BASELINE_METADATA:

    raise RuntimeError(
        "Unexpected canonical baseline metadata record count.\n"
        f"Expected: {EXPECTED_BASELINE_METADATA}\n"
        f"Found   : {len(BASELINE_METADATA_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 11. Validate Canonical Metadata Uniqueness
# --------------------------------------------------------------------------------------------------

if BASELINE_METADATA_DF.duplicated(
    subset=[
        "dataset_id",
        "baseline",
    ]
).any():

    raise RuntimeError(
        "Duplicate canonical baseline metadata records detected."
    )


# --------------------------------------------------------------------------------------------------
# 12. Validate Canonical Metadata Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINE_KEYS = {
    (
        dataset_id,
        baseline_name,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


ACTUAL_BASELINE_KEYS = {
    (
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in BASELINE_METADATA_DF.iterrows()
}


if ACTUAL_BASELINE_KEYS != EXPECTED_BASELINE_KEYS:

    raise RuntimeError(
        "Canonical baseline metadata coverage mismatch."
    )


print()
print(
    f"✓ Canonical metadata records : "
    f"{len(BASELINE_METADATA_DF)}"
)

print(
    "✓ Metadata uniqueness        : PASS"
)

print(
    "✓ Metadata coverage          : PASS"
)


# --------------------------------------------------------------------------------------------------
# 13. Create Experiment Metadata Registry
# --------------------------------------------------------------------------------------------------

EXPERIMENT_METADATA_DF = pd.DataFrame(
    EXPERIMENT_METADATA_RECORDS
)


if len(
    EXPERIMENT_METADATA_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Unexpected experiment metadata record count.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {len(EXPERIMENT_METADATA_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 14. Validate Experiment Metadata Uniqueness
# --------------------------------------------------------------------------------------------------

EXPERIMENT_METADATA_KEYS = [
    "repetition",
    "dataset_id",
    "baseline",
    "experiment_seed",
]


if EXPERIMENT_METADATA_DF.duplicated(
    subset=EXPERIMENT_METADATA_KEYS
).any():

    raise RuntimeError(
        "Duplicate experiment metadata records detected."
    )


# --------------------------------------------------------------------------------------------------
# 15. Validate Experiment Metadata Coverage
# --------------------------------------------------------------------------------------------------

actual_experiment_metadata_keys = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
        int(row["experiment_seed"]),
    )
    for _, row in EXPERIMENT_METADATA_DF.iterrows()
}


if actual_experiment_metadata_keys != runtime_keys:

    raise RuntimeError(
        "Experiment metadata coverage does not match "
        "the authoritative runtime registry."
    )


print(
    f"✓ Experiment metadata records: "
    f"{len(EXPERIMENT_METADATA_DF)}"
)

print(
    "✓ Experiment metadata coverage : PASS"
)


# --------------------------------------------------------------------------------------------------
# 16. Validate Training-Row Consistency
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    authoritative_training_rows = int(
        len(
            TRAINING_DATA[
                dataset_id
            ]
        )
    )

    observed_training_rows = set(
        EXPERIMENT_METADATA_DF[
            EXPERIMENT_METADATA_DF[
                "dataset_id"
            ]
            == dataset_id
        ][
            "training_rows"
        ]
        .astype(int)
        .tolist()
    )


    if observed_training_rows != {
        authoritative_training_rows
    }:

        raise RuntimeError(
            f"{dataset_id}: metadata training-row "
            "count does not match TRAINING_DATA.\n"
            f"Expected: {authoritative_training_rows}\n"
            f"Found   : {sorted(observed_training_rows)}"
        )


print(
    "✓ Training-row consistency    : PASS"
)


# --------------------------------------------------------------------------------------------------
# 17. Persist Canonical Metadata Registry
# --------------------------------------------------------------------------------------------------

BASELINE_METADATA_REGISTRY_PATH = (
    MANIFEST_ROOT
    / "baseline_metadata_registry.csv"
)


BASELINE_METADATA_DF = (
    BASELINE_METADATA_DF
    .sort_values(
        by=[
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


BASELINE_METADATA_DF.to_csv(
    BASELINE_METADATA_REGISTRY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 18. Persist 30-Run Experiment Metadata Registry
# --------------------------------------------------------------------------------------------------

EXPERIMENT_METADATA_REGISTRY_PATH = (
    MANIFEST_ROOT
    / "baseline_experiment_metadata_registry.csv"
)


EXPERIMENT_METADATA_DF = (
    EXPERIMENT_METADATA_DF
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


EXPERIMENT_METADATA_DF.to_csv(
    EXPERIMENT_METADATA_REGISTRY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 19. Persistence Verification
# --------------------------------------------------------------------------------------------------

if not BASELINE_METADATA_REGISTRY_PATH.exists():

    raise RuntimeError(
        "Canonical baseline metadata registry was not created."
    )


if not EXPERIMENT_METADATA_REGISTRY_PATH.exists():

    raise RuntimeError(
        "Experiment metadata registry was not created."
    )


RELOADED_BASELINE_METADATA_DF = pd.read_csv(
    BASELINE_METADATA_REGISTRY_PATH
)


RELOADED_EXPERIMENT_METADATA_DF = pd.read_csv(
    EXPERIMENT_METADATA_REGISTRY_PATH
)


if len(
    RELOADED_BASELINE_METADATA_DF
) != EXPECTED_BASELINE_METADATA:

    raise RuntimeError(
        "Persisted canonical metadata registry count mismatch."
    )


if len(
    RELOADED_EXPERIMENT_METADATA_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Persisted experiment metadata registry count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 20. Final Integrity Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 14 — BASELINE METADATA SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets                       : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset          : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                    : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected experimental runs     : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Canonical metadata records     : "
    f"{len(BASELINE_METADATA_DF)}"
)

print(
    f"✓ Experiment metadata records    : "
    f"{len(EXPERIMENT_METADATA_DF)}"
)

print(
    "✓ Section 13 artifact linkage         : PASS"
)

print(
    "✓ Train-only fit policy               : PASS"
)

print(
    "✓ Validation/test exclusion           : PASS"
)

print(
    "✓ Target retention policy             : PASS"
)

print(
    "✓ Provenance exclusion policy         : PASS"
)

print(
    "✓ Identifier exclusion policy         : PASS"
)

print(
    "✓ Repetition seed traceability        : PASS"
)

print(
    "✓ Experiment seed traceability        : PASS"
)

print(
    "✓ Canonical metadata persistence      : PASS"
)

print(
    "✓ Experiment metadata persistence     : PASS"
)

print(
    "✓ Metadata reload validation          : PASS"
)

print(
    f"✓ Canonical metadata registry        : "
    f"{BASELINE_METADATA_REGISTRY_PATH}"
)

print(
    f"✓ Experiment metadata registry       : "
    f"{EXPERIMENT_METADATA_REGISTRY_PATH}"
)

print()
print(
    "✓ SECTION 14 — BASELINE METADATA: PASS"
)


# --------------------------------------------------------------------------------------------------
# 21. RAM Cleanup
# --------------------------------------------------------------------------------------------------

del RELOADED_BASELINE_METADATA_DF
del RELOADED_EXPERIMENT_METADATA_DF
del SYNTHETIC_ARTIFACT_REGISTRY_DF

gc.collect()

print(
    "✓ Final RAM cleanup completed."
)

SECTION 14 — SAVE BASELINE METADATA

✓ Datasets                    : 3
✓ Baselines                   : 2
✓ Repetitions                 : 5
✓ Expected experiment runs    : 30
✓ Canonical baseline metadata : 6

✓ Section 13 artifact registry : 30 records
✓ Runtime registry             : 30 records
✓ Runtime/artifact unit linkage : PASS
✓ adult_income         | independent_marginal     | canonical metadata saved | 5 repetitions linked
✓ adult_income         | gaussian_copula          | canonical metadata saved | 5 repetitions linked
✓ bank_marketing       | independent_marginal     | canonical metadata saved | 5 repetitions linked
✓ bank_marketing       | gaussian_copula          | canonical metadata saved | 5 repetitions linked
✓ diabetes_130us       | independent_marginal     | canonical metadata saved | 5 repetitions linked
✓ diabetes_130us       | gaussian_copula          | canonical metadata saved | 5 repetitions linked

✓ Canonical metadata records : 6
✓ Metadata uniqueness        :

In [96]:
# ==================================================================================================
# SECTION 15 — SAVE GENERATION MANIFEST
# 30-RUN EXPERIMENTAL GENERATION MANIFEST
# ARTIFACT-DRIVEN + RAM-SAFE + RESTART-SAFE
# ==================================================================================================

print("=" * 100)
print("SECTION 15 — SAVE GENERATION MANIFEST")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import gc
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "PROJECT_ROOT",
    "DATASET_IDS",
    "BASELINE_METHODS",
    "REPETITIONS",
    "RUNTIME_DF",
    "SYNTHETIC_ARTIFACT_DF",
    "BASELINE_METADATA_DF",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Section 15 is missing required runtime objects: "
        f"{missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_BASELINES = [
    "independent_marginal",
    "gaussian_copula",
]

if list(BASELINE_METHODS) != EXPECTED_BASELINES:
    raise RuntimeError(
        "BASELINE_METHODS does not match the frozen Notebook 04 registry.\n"
        f"Expected: {EXPECTED_BASELINES}\n"
        f"Found   : {list(BASELINE_METHODS)}"
    )

if int(REPETITIONS) != 5:
    raise RuntimeError(
        f"Notebook 04 requires 5 repetitions, found {REPETITIONS}."
    )

EXPECTED_MANIFEST_RECORDS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)

EXPECTED_BASELINE_METADATA_RECORDS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)

if EXPECTED_MANIFEST_RECORDS != 30:
    raise RuntimeError(
        "Expected exactly 30 generation-manifest records. "
        f"Found {EXPECTED_MANIFEST_RECORDS}."
    )

if EXPECTED_BASELINE_METADATA_RECORDS != 6:
    raise RuntimeError(
        "Expected exactly 6 canonical metadata records. "
        f"Found {EXPECTED_BASELINE_METADATA_RECORDS}."
    )

print()
print(
    f"✓ Datasets                    : {len(DATASET_IDS)}"
)
print(
    f"✓ Baselines                   : {len(BASELINE_METHODS)}"
)
print(
    f"✓ Repetitions                 : {REPETITIONS}"
)
print(
    f"✓ Expected manifest records   : {EXPECTED_MANIFEST_RECORDS}"
)


# --------------------------------------------------------------------------------------------------
# 4. Authoritative Paths
# --------------------------------------------------------------------------------------------------

NB04_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_04"
)

BASELINE_ROOT = (
    NB04_ROOT
    / "baselines"
)

MANIFEST_ROOT = (
    BASELINE_ROOT
    / "manifests"
)

MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Section 13 Synthetic Artifact Registry
# --------------------------------------------------------------------------------------------------

REQUIRED_SYNTHETIC_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "dataset_id",
    "baseline",
    "artifact_type",
    "path",
    "relative_path",
    "rows",
    "columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_excluded",
    "identifiers_excluded",
    "file_size_bytes",
    "sha256",
    "status",
]

missing_synthetic_columns = [
    column
    for column in REQUIRED_SYNTHETIC_COLUMNS
    if column not in SYNTHETIC_ARTIFACT_DF.columns
]

if missing_synthetic_columns:
    raise RuntimeError(
        "SYNTHETIC_ARTIFACT_DF is missing required column(s): "
        f"{missing_synthetic_columns}"
    )

if len(SYNTHETIC_ARTIFACT_DF) != EXPECTED_MANIFEST_RECORDS:
    raise RuntimeError(
        "Synthetic artifact registry record count mismatch.\n"
        f"Expected: {EXPECTED_MANIFEST_RECORDS}\n"
        f"Found   : {len(SYNTHETIC_ARTIFACT_DF)}"
    )

print()
print(
    f"✓ Section 13 synthetic artifacts : "
    f"{len(SYNTHETIC_ARTIFACT_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 6. Validate Section 14 Metadata Registry
# --------------------------------------------------------------------------------------------------

REQUIRED_METADATA_COLUMNS = [
    "dataset_id",
    "baseline",
    "metadata_path",
    "metadata_sha256",
    "training_rows",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "fit_data",
    "validation_used",
    "test_used",
    "status",
]

missing_metadata_columns = [
    column
    for column in REQUIRED_METADATA_COLUMNS
    if column not in BASELINE_METADATA_DF.columns
]

if missing_metadata_columns:
    raise RuntimeError(
        "BASELINE_METADATA_DF is missing required column(s): "
        f"{missing_metadata_columns}"
    )

if len(BASELINE_METADATA_DF) != EXPECTED_BASELINE_METADATA_RECORDS:
    raise RuntimeError(
        "Canonical metadata registry record count mismatch.\n"
        f"Expected: {EXPECTED_BASELINE_METADATA_RECORDS}\n"
        f"Found   : {len(BASELINE_METADATA_DF)}"
    )

print(
    f"✓ Section 14 metadata records    : "
    f"{len(BASELINE_METADATA_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 7. Validate Section 12 Runtime Registry
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "dataset_id",
    "dataset_index",
    "baseline",
    "baseline_index",
    "training_rows",
    "rows_requested",
    "rows_generated",
    "fit_runtime_seconds",
    "generation_runtime_seconds",
    "total_runtime_seconds",
    "model_artifact_path",
    "model_artifact_sha256",
    "synthetic_artifact",
    "synthetic_sha256",
]

missing_runtime_columns = [
    column
    for column in REQUIRED_RUNTIME_COLUMNS
    if column not in RUNTIME_DF.columns
]

if missing_runtime_columns:
    raise RuntimeError(
        "RUNTIME_DF is missing required column(s): "
        f"{missing_runtime_columns}"
    )

if len(RUNTIME_DF) != EXPECTED_MANIFEST_RECORDS:
    raise RuntimeError(
        "Runtime registry record count mismatch.\n"
        f"Expected: {EXPECTED_MANIFEST_RECORDS}\n"
        f"Found   : {len(RUNTIME_DF)}"
    )

print(
    f"✓ Section 12 runtime records     : "
    f"{len(RUNTIME_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 8. Expected Experimental Units
# --------------------------------------------------------------------------------------------------

EXPECTED_UNITS = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


# --------------------------------------------------------------------------------------------------
# 9. Initialize Final Generation Manifest
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 10. Process 30 Experimental Runs
# --------------------------------------------------------------------------------------------------

for repetition in range(
    1,
    int(REPETITIONS) + 1,
):

    print()
    print("=" * 100)
    print(
        f"REPETITION {repetition} / {REPETITIONS}"
    )
    print("=" * 100)

    for dataset_index, dataset_id in enumerate(
        DATASET_IDS
    ):

        for baseline_index, baseline_name in enumerate(
            BASELINE_METHODS
        ):

            # --------------------------------------------------------------------------------------
            # 10.1 Synthetic Artifact Record
            # --------------------------------------------------------------------------------------

            synthetic_matches = SYNTHETIC_ARTIFACT_DF[
                (
                    SYNTHETIC_ARTIFACT_DF["repetition"]
                    .astype(int)
                    == repetition
                )
                &
                (
                    SYNTHETIC_ARTIFACT_DF["dataset_id"]
                    == dataset_id
                )
                &
                (
                    SYNTHETIC_ARTIFACT_DF["baseline"]
                    == baseline_name
                )
            ].copy()

            if len(synthetic_matches) != 1:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "expected exactly one synthetic artifact "
                    f"record, found {len(synthetic_matches)}."
                )

            synthetic_record = synthetic_matches.iloc[0]


            # --------------------------------------------------------------------------------------
            # 10.2 Runtime Record
            # --------------------------------------------------------------------------------------

            runtime_matches = RUNTIME_DF[
                (
                    RUNTIME_DF["repetition"]
                    .astype(int)
                    == repetition
                )
                &
                (
                    RUNTIME_DF["dataset_id"]
                    == dataset_id
                )
                &
                (
                    RUNTIME_DF["baseline"]
                    == baseline_name
                )
            ].copy()

            if len(runtime_matches) != 1:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "expected exactly one runtime record, "
                    f"found {len(runtime_matches)}."
                )

            runtime_record = runtime_matches.iloc[0]


            # --------------------------------------------------------------------------------------
            # 10.3 Canonical Metadata Record
            # --------------------------------------------------------------------------------------

            metadata_matches = BASELINE_METADATA_DF[
                (
                    BASELINE_METADATA_DF["dataset_id"]
                    == dataset_id
                )
                &
                (
                    BASELINE_METADATA_DF["baseline"]
                    == baseline_name
                )
            ].copy()

            if len(metadata_matches) != 1:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}: "
                    "expected exactly one canonical metadata "
                    f"record, found {len(metadata_matches)}."
                )

            metadata_record = metadata_matches.iloc[0]


            # --------------------------------------------------------------------------------------
            # 10.4 Seed Consistency
            # --------------------------------------------------------------------------------------

            synthetic_repetition_seed = int(
                synthetic_record["repetition_seed"]
            )

            runtime_repetition_seed = int(
                runtime_record["repetition_seed"]
            )

            if (
                synthetic_repetition_seed
                != runtime_repetition_seed
            ):
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "repetition-seed mismatch."
                )


            synthetic_experiment_seed = int(
                synthetic_record["experiment_seed"]
            )

            runtime_experiment_seed = int(
                runtime_record["experiment_seed"]
            )

            if (
                synthetic_experiment_seed
                != runtime_experiment_seed
            ):
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "experiment-seed mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 10.5 Row-Count Consistency
            # --------------------------------------------------------------------------------------

            training_rows = int(
                runtime_record["training_rows"]
            )

            rows_requested = int(
                runtime_record["rows_requested"]
            )

            rows_generated = int(
                runtime_record["rows_generated"]
            )

            synthetic_rows = int(
                synthetic_record["rows"]
            )

            if training_rows != rows_requested:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "training rows != requested rows."
                )

            if rows_requested != rows_generated:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "requested rows != generated rows."
                )

            if rows_generated != synthetic_rows:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "runtime rows != persisted synthetic rows."
                )


            # --------------------------------------------------------------------------------------
            # 10.6 Synthetic Artifact
            # --------------------------------------------------------------------------------------

            synthetic_path = Path(
                synthetic_record["path"]
            )

            if not synthetic_path.exists():
                raise FileNotFoundError(
                    f"Synthetic artifact does not exist:\n"
                    f"{synthetic_path}"
                )

            synthetic_file_size = int(
                synthetic_path.stat().st_size
            )

            if synthetic_file_size <= 0:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "synthetic artifact is empty."
                )


            # --------------------------------------------------------------------------------------
            # 10.7 Synthetic SHA-256
            # --------------------------------------------------------------------------------------

            calculated_synthetic_sha256 = (
                calculate_sha256(
                    synthetic_path
                )
            )

            recorded_synthetic_sha256 = str(
                synthetic_record["sha256"]
            )

            if (
                calculated_synthetic_sha256
                != recorded_synthetic_sha256
            ):
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "synthetic SHA-256 mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 10.8 Model Artifact
            # --------------------------------------------------------------------------------------

            model_path = Path(
                runtime_record["model_artifact_path"]
            )

            if not model_path.exists():
                raise FileNotFoundError(
                    f"Model artifact does not exist:\n"
                    f"{model_path}"
                )

            model_file_size = int(
                model_path.stat().st_size
            )

            if model_file_size <= 0:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "model artifact is empty."
                )

            calculated_model_sha256 = calculate_sha256(
                model_path
            )

            recorded_model_sha256 = str(
                runtime_record[
                    "model_artifact_sha256"
                ]
            )

            if (
                calculated_model_sha256
                != recorded_model_sha256
            ):
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "model SHA-256 mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 10.9 Metadata Artifact
            # --------------------------------------------------------------------------------------

            metadata_path = Path(
                metadata_record["metadata_path"]
            )

            if not metadata_path.exists():
                raise FileNotFoundError(
                    f"Metadata artifact does not exist:\n"
                    f"{metadata_path}"
                )

            metadata_file_size = int(
                metadata_path.stat().st_size
            )

            if metadata_file_size <= 0:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}: "
                    "metadata artifact is empty."
                )

            calculated_metadata_sha256 = (
                calculate_sha256(
                    metadata_path
                )
            )

            recorded_metadata_sha256 = str(
                metadata_record[
                    "metadata_sha256"
                ]
            )

            if (
                calculated_metadata_sha256
                != recorded_metadata_sha256
            ):
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}: "
                    "metadata SHA-256 mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 10.10 Construct Final Manifest Record
            # --------------------------------------------------------------------------------------

            GENERATION_MANIFEST_RECORDS.append(
                {
                    "notebook_id": (
                        NOTEBOOK_ID
                        if "NOTEBOOK_ID" in globals()
                        else "04"
                    ),

                    "notebook_version": (
                        NOTEBOOK_VERSION
                        if "NOTEBOOK_VERSION" in globals()
                        else "1.0"
                    ),

                    "repetition": repetition,

                    "repetition_seed": (
                        synthetic_repetition_seed
                    ),

                    "experiment_seed": (
                        synthetic_experiment_seed
                    ),

                    "dataset_id": dataset_id,

                    "dataset_index": dataset_index,

                    "baseline": baseline_name,

                    "baseline_index": baseline_index,

                    "fit_split": "train",

                    "validation_used_for_fit": False,

                    "test_used_for_fit": False,

                    "target_used_as_predictor": False,

                    "identifier_used": False,

                    "provenance_used": False,

                    "training_rows": training_rows,

                    "rows_requested": rows_requested,

                    "synthetic_rows": synthetic_rows,

                    "synthetic_columns": int(
                        synthetic_record["columns"]
                    ),

                    "synthetic_data_path": str(
                        synthetic_record["relative_path"]
                    ),

                    "synthetic_data_sha256": (
                        calculated_synthetic_sha256
                    ),

                    "synthetic_file_size_bytes": (
                        synthetic_file_size
                    ),

                    "model_path": str(
                        model_path.relative_to(
                            NB04_ROOT
                        )
                    ),

                    "model_sha256": (
                        calculated_model_sha256
                    ),

                    "model_file_size_bytes": (
                        model_file_size
                    ),

                    "metadata_path": str(
                        metadata_path.relative_to(
                            NB04_ROOT
                        )
                    ),

                    "metadata_sha256": (
                        calculated_metadata_sha256
                    ),

                    "fit_runtime_seconds": float(
                        runtime_record[
                            "fit_runtime_seconds"
                        ]
                    ),

                    "generation_runtime_seconds": float(
                        runtime_record[
                            "generation_runtime_seconds"
                        ]
                    ),

                    "total_runtime_seconds": float(
                        runtime_record[
                            "total_runtime_seconds"
                        ]
                    ),

                    "execution_status": str(
                        synthetic_record[
                            "execution_status"
                        ]
                    ),

                    "status": "PASS",

                    "created_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                }
            )

            print(
                f"✓ {dataset_id:<20} | "
                f"{baseline_name:<24} | "
                f"repetition={repetition} | "
                f"seed={synthetic_experiment_seed} | "
                f"PASS"
            )


# --------------------------------------------------------------------------------------------------
# 11. Create Manifest DataFrame
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_DF = pd.DataFrame(
    GENERATION_MANIFEST_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 12. Validate Record Count
# --------------------------------------------------------------------------------------------------

if len(
    GENERATION_MANIFEST_DF
) != EXPECTED_MANIFEST_RECORDS:

    raise RuntimeError(
        "Generation manifest record count mismatch.\n"
        f"Expected: {EXPECTED_MANIFEST_RECORDS}\n"
        f"Found   : {len(GENERATION_MANIFEST_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 13. Validate Experimental-Unit Uniqueness
# --------------------------------------------------------------------------------------------------

MANIFEST_KEYS = [
    "repetition",
    "dataset_id",
    "baseline",
    "experiment_seed",
]

if GENERATION_MANIFEST_DF.duplicated(
    subset=MANIFEST_KEYS
).any():

    raise RuntimeError(
        "Duplicate generation-manifest experimental units detected."
    )


# --------------------------------------------------------------------------------------------------
# 14. Validate Experimental-Unit Coverage
# --------------------------------------------------------------------------------------------------

actual_manifest_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_MANIFEST_DF.iterrows()
}

if actual_manifest_units != EXPECTED_UNITS:

    missing_units = sorted(
        EXPECTED_UNITS
        - actual_manifest_units
    )

    unexpected_units = sorted(
        actual_manifest_units
        - EXPECTED_UNITS
    )

    raise RuntimeError(
        "Generation manifest coverage mismatch.\n"
        f"Missing    : {missing_units}\n"
        f"Unexpected : {unexpected_units}"
    )


# --------------------------------------------------------------------------------------------------
# 15. Validate All Records PASS
# --------------------------------------------------------------------------------------------------

if not (
    GENERATION_MANIFEST_DF["status"]
    .astype(str)
    .eq("PASS")
).all():

    raise RuntimeError(
        "One or more generation manifest records failed."
    )


# --------------------------------------------------------------------------------------------------
# 16. Sort Manifest
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_DF = (
    GENERATION_MANIFEST_DF
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 17. Save Generation Manifest
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "generation_manifest.csv"
)

GENERATION_MANIFEST_DF.to_csv(
    GENERATION_MANIFEST_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 18. Validate Persistence
# --------------------------------------------------------------------------------------------------

if not GENERATION_MANIFEST_PATH.exists():

    raise RuntimeError(
        "Generation manifest file was not created."
    )

if GENERATION_MANIFEST_PATH.stat().st_size <= 0:

    raise RuntimeError(
        "Generation manifest file is empty."
    )


# --------------------------------------------------------------------------------------------------
# 19. Reload and Verify
# --------------------------------------------------------------------------------------------------

RELOADED_GENERATION_MANIFEST_DF = pd.read_csv(
    GENERATION_MANIFEST_PATH
)

if len(
    RELOADED_GENERATION_MANIFEST_DF
) != EXPECTED_MANIFEST_RECORDS:

    raise RuntimeError(
        "Persisted generation manifest record count mismatch."
    )


reloaded_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in RELOADED_GENERATION_MANIFEST_DF.iterrows()
}

if reloaded_units != EXPECTED_UNITS:

    raise RuntimeError(
        "Persisted generation manifest coverage validation failed."
    )


# --------------------------------------------------------------------------------------------------
# 20. Final Manifest SHA-256
# --------------------------------------------------------------------------------------------------

GENERATION_MANIFEST_SHA256 = calculate_sha256(
    GENERATION_MANIFEST_PATH
)

if not GENERATION_MANIFEST_SHA256:

    raise RuntimeError(
        "Generation manifest SHA-256 calculation failed."
    )


# --------------------------------------------------------------------------------------------------
# 21. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 15 — GENERATION MANIFEST SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets                       : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines per dataset          : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                    : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected manifest records      : "
    f"{EXPECTED_MANIFEST_RECORDS}"
)

print(
    f"✓ Generation manifest records    : "
    f"{len(GENERATION_MANIFEST_DF)}"
)

print(
    "✓ Section 13 synthetic linkage        : PASS"
)

print(
    "✓ Section 14 metadata linkage         : PASS"
)

print(
    "✓ Section 12 runtime linkage          : PASS"
)

print(
    "✓ Model artifact existence            : PASS"
)

print(
    "✓ Model artifact SHA-256              : PASS"
)

print(
    "✓ Synthetic artifact existence        : PASS"
)

print(
    "✓ Synthetic artifact SHA-256          : PASS"
)

print(
    "✓ Metadata artifact SHA-256           : PASS"
)

print(
    "✓ Training/synthetic row consistency  : PASS"
)

print(
    "✓ Requested/generated row consistency : PASS"
)

print(
    "✓ Target policy                       : PASS"
)

print(
    "✓ Identifier exclusion                : PASS"
)

print(
    "✓ Provenance exclusion                : PASS"
)

print(
    "✓ Repetition seed traceability        : PASS"
)

print(
    "✓ Experiment seed traceability        : PASS"
)

print(
    "✓ Manifest uniqueness                 : PASS"
)

print(
    "✓ 5-repetition coverage               : PASS"
)

print(
    "✓ 30-run experimental coverage        : PASS"
)

print(
    "✓ Manifest persistence                : PASS"
)

print(
    "✓ Manifest reload validation          : PASS"
)

print(
    f"✓ Generation manifest                 : "
    f"{GENERATION_MANIFEST_PATH}"
)

print(
    f"✓ Generation manifest SHA-256         : "
    f"{GENERATION_MANIFEST_SHA256}"
)

print()
print(
    "✓ SECTION 15 — GENERATION MANIFEST: PASS"
)


# --------------------------------------------------------------------------------------------------
# 22. RAM Cleanup
# --------------------------------------------------------------------------------------------------

del RELOADED_GENERATION_MANIFEST_DF

gc.collect()

print(
    "✓ Final RAM cleanup completed."
)

SECTION 15 — SAVE GENERATION MANIFEST

✓ Datasets                    : 3
✓ Baselines                   : 2
✓ Repetitions                 : 5
✓ Expected manifest records   : 30

✓ Section 13 synthetic artifacts : 30
✓ Section 14 metadata records    : 6
✓ Section 12 runtime records     : 30

REPETITION 1 / 5
✓ adult_income         | independent_marginal     | repetition=1 | seed=3136 | PASS
✓ adult_income         | gaussian_copula          | repetition=1 | seed=3146 | PASS
✓ bank_marketing       | independent_marginal     | repetition=1 | seed=3236 | PASS
✓ bank_marketing       | gaussian_copula          | repetition=1 | seed=3246 | PASS
✓ diabetes_130us       | independent_marginal     | repetition=1 | seed=3336 | PASS
✓ diabetes_130us       | gaussian_copula          | repetition=1 | seed=3346 | PASS

REPETITION 2 / 5
✓ adult_income         | independent_marginal     | repetition=2 | seed=3137 | PASS
✓ adult_income         | gaussian_copula          | repetition=2 | seed=3147 | PASS
✓ 

In [98]:
# ==================================================================================================
# SECTION 16 — VERIFY ARTIFACTS
# 30-RUN ARTIFACT INTEGRITY VERIFICATION
# RAM-SAFE + PERSISTED-ARTIFACT-DRIVEN
# ==================================================================================================

print("=" * 100)
print("SECTION 16 — VERIFY ARTIFACTS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import gc
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "PROJECT_ROOT",
    "NB04_ROOT",
    "DATASET_IDS",
    "BASELINE_METHODS",
    "REPETITIONS",
    "GENERATION_MANIFEST_DF",
    "TRAINING_DATA",
    "TRAINING_GENERATIVE_COLUMNS",
    "TRAINING_TARGET_COLUMNS",
    "TRAINING_PROVENANCE_COLUMNS",
    "TRAINING_IDENTIFIER_COLUMNS",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Section 16 is missing required runtime objects: "
        f"{missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Frozen Experimental Configuration
# --------------------------------------------------------------------------------------------------

EXPECTED_RUNS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)

if EXPECTED_RUNS != 30:
    raise RuntimeError(
        "Expected exactly 30 experimental runs.\n"
        f"Found: {EXPECTED_RUNS}"
    )


print()
print(
    f"✓ Datasets                 : {len(DATASET_IDS)}"
)

print(
    f"✓ Baselines                : {len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions              : {REPETITIONS}"
)

print(
    f"✓ Expected artifact runs   : {EXPECTED_RUNS}"
)


# --------------------------------------------------------------------------------------------------
# 4. Validate Section 15 Generation Manifest
# --------------------------------------------------------------------------------------------------

REQUIRED_MANIFEST_COLUMNS = [
    "repetition",
    "repetition_seed",
    "experiment_seed",
    "dataset_id",
    "baseline",
    "fit_split",
    "validation_used_for_fit",
    "test_used_for_fit",
    "target_used_as_predictor",
    "identifier_used",
    "provenance_used",
    "training_rows",
    "rows_requested",
    "synthetic_rows",
    "synthetic_columns",
    "synthetic_data_path",
    "synthetic_data_sha256",
    "model_path",
    "model_sha256",
    "metadata_path",
    "metadata_sha256",
    "status",
]

missing_manifest_columns = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in GENERATION_MANIFEST_DF.columns
]

if missing_manifest_columns:
    raise RuntimeError(
        "Generation manifest is missing required column(s): "
        f"{missing_manifest_columns}"
    )


if len(GENERATION_MANIFEST_DF) != EXPECTED_RUNS:
    raise RuntimeError(
        "Generation manifest count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {len(GENERATION_MANIFEST_DF)}"
    )


print(
    f"✓ Generation manifest records : "
    f"{len(GENERATION_MANIFEST_DF)}"
)


# --------------------------------------------------------------------------------------------------
# 5. Expected Experimental Units
# --------------------------------------------------------------------------------------------------

EXPECTED_EXPERIMENTAL_UNITS = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


# --------------------------------------------------------------------------------------------------
# 6. Validate Manifest Experimental-Unit Uniqueness
# --------------------------------------------------------------------------------------------------

MANIFEST_UNIT_COLUMNS = [
    "repetition",
    "dataset_id",
    "baseline",
    "experiment_seed",
]

if GENERATION_MANIFEST_DF.duplicated(
    subset=MANIFEST_UNIT_COLUMNS
).any():
    raise RuntimeError(
        "Duplicate experimental units detected in "
        "generation manifest."
    )


actual_manifest_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_MANIFEST_DF.iterrows()
}


if actual_manifest_units != EXPECTED_EXPERIMENTAL_UNITS:

    missing_units = sorted(
        EXPECTED_EXPERIMENTAL_UNITS
        - actual_manifest_units
    )

    unexpected_units = sorted(
        actual_manifest_units
        - EXPECTED_EXPERIMENTAL_UNITS
    )

    raise RuntimeError(
        "Generation manifest experimental-unit coverage mismatch.\n"
        f"Missing    : {missing_units}\n"
        f"Unexpected : {unexpected_units}"
    )


print(
    "✓ 30-run experimental coverage : PASS"
)

print(
    "✓ Manifest experimental-unit uniqueness : PASS"
)


# --------------------------------------------------------------------------------------------------
# 7. Verification Container
# --------------------------------------------------------------------------------------------------

ARTIFACT_VERIFICATION_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 8. Verify Every Repetition × Dataset × Baseline
# --------------------------------------------------------------------------------------------------

for repetition in range(
    1,
    int(REPETITIONS) + 1,
):

    print()
    print("=" * 100)
    print(
        f"REPETITION {repetition} / {REPETITIONS}"
    )
    print("=" * 100)

    for dataset_id in DATASET_IDS:

        for baseline_name in BASELINE_METHODS:

            # --------------------------------------------------------------------------------------
            # 8.1 Locate Exactly One Manifest Record
            # --------------------------------------------------------------------------------------

            matches = GENERATION_MANIFEST_DF[
                (
                    GENERATION_MANIFEST_DF[
                        "repetition"
                    ]
                    .astype(int)
                    == repetition
                )
                &
                (
                    GENERATION_MANIFEST_DF[
                        "dataset_id"
                    ]
                    == dataset_id
                )
                &
                (
                    GENERATION_MANIFEST_DF[
                        "baseline"
                    ]
                    == baseline_name
                )
            ].copy()


            if len(matches) != 1:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "expected exactly one manifest record, "
                    f"found {len(matches)}."
                )


            record = matches.iloc[0]


            # --------------------------------------------------------------------------------------
            # 8.2 Read Experiment Identifiers
            # --------------------------------------------------------------------------------------

            repetition_seed = int(
                record["repetition_seed"]
            )

            experiment_seed = int(
                record["experiment_seed"]
            )


            # --------------------------------------------------------------------------------------
            # 8.3 Resolve Artifact Paths
            # --------------------------------------------------------------------------------------

            synthetic_path = (
                NB04_ROOT
                / str(
                    record[
                        "synthetic_data_path"
                    ]
                )
            )

            model_path = (
                NB04_ROOT
                / str(
                    record[
                        "model_path"
                    ]
                )
            )

            metadata_path = (
                NB04_ROOT
                / str(
                    record[
                        "metadata_path"
                    ]
                )
            )


            # --------------------------------------------------------------------------------------
            # 8.4 Verify Artifact Existence
            # --------------------------------------------------------------------------------------

            artifact_paths = [
                (
                    "synthetic data",
                    synthetic_path,
                ),
                (
                    "model",
                    model_path,
                ),
                (
                    "metadata",
                    metadata_path,
                ),
            ]


            for label, path in artifact_paths:

                if not path.exists():
                    raise FileNotFoundError(
                        f"{dataset_id}/{baseline_name}/"
                        f"repetition {repetition}: "
                        f"{label} artifact missing:\n"
                        f"{path}"
                    )


                if path.stat().st_size <= 0:
                    raise RuntimeError(
                        f"{dataset_id}/{baseline_name}/"
                        f"repetition {repetition}: "
                        f"{label} artifact is empty:\n"
                        f"{path}"
                    )


            # --------------------------------------------------------------------------------------
            # 8.5 Verify Synthetic SHA-256
            # --------------------------------------------------------------------------------------

            actual_synthetic_hash = calculate_sha256(
                synthetic_path
            )

            expected_synthetic_hash = str(
                record[
                    "synthetic_data_sha256"
                ]
            )


            if actual_synthetic_hash != expected_synthetic_hash:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "synthetic-data SHA-256 mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 8.6 Verify Model SHA-256
            # --------------------------------------------------------------------------------------

            actual_model_hash = calculate_sha256(
                model_path
            )

            expected_model_hash = str(
                record[
                    "model_sha256"
                ]
            )


            if actual_model_hash != expected_model_hash:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "model SHA-256 mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 8.7 Verify Metadata SHA-256
            # --------------------------------------------------------------------------------------

            actual_metadata_hash = calculate_sha256(
                metadata_path
            )

            expected_metadata_hash = str(
                record[
                    "metadata_sha256"
                ]
            )


            if actual_metadata_hash != expected_metadata_hash:
                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "metadata SHA-256 mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 8.8 Reload Synthetic Artifact
            # --------------------------------------------------------------------------------------

            reloaded_df = pd.read_csv(
                synthetic_path,
                low_memory=False,
            )


            # --------------------------------------------------------------------------------------
            # 8.9 Validate Exact Generative Schema
            # --------------------------------------------------------------------------------------

            expected_columns = list(
                TRAINING_GENERATIVE_COLUMNS[
                    dataset_id
                ]
            )


            if list(
                reloaded_df.columns
            ) != expected_columns:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "persisted synthetic schema mismatch.\n"
                    f"Expected: {expected_columns}\n"
                    f"Found   : {list(reloaded_df.columns)}"
                )


            # --------------------------------------------------------------------------------------
            # 8.10 Validate Training-Row Count
            # --------------------------------------------------------------------------------------

            expected_rows = int(
                len(
                    TRAINING_DATA[
                        dataset_id
                    ]
                )
            )

            actual_rows = int(
                len(reloaded_df)
            )


            if actual_rows != expected_rows:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "persisted synthetic row count mismatch.\n"
                    f"Expected: {expected_rows:,}\n"
                    f"Found   : {actual_rows:,}"
                )


            # --------------------------------------------------------------------------------------
            # 8.11 Validate Manifest Row Counts
            # --------------------------------------------------------------------------------------

            if int(
                record[
                    "training_rows"
                ]
            ) != expected_rows:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "manifest training-row count mismatch."
                )


            if int(
                record[
                    "rows_requested"
                ]
            ) != actual_rows:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "manifest requested-row count mismatch."
                )


            if int(
                record[
                    "synthetic_rows"
                ]
            ) != actual_rows:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "manifest synthetic-row count mismatch."
                )


            if int(
                record[
                    "synthetic_columns"
                ]
            ) != len(
                reloaded_df.columns
            ):

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "manifest synthetic-column count mismatch."
                )


            # --------------------------------------------------------------------------------------
            # 8.12 Verify Target
            # --------------------------------------------------------------------------------------

            target = (
                TRAINING_TARGET_COLUMNS[
                    dataset_id
                ]
            )


            if target not in reloaded_df.columns:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    f"target column '{target}' missing."
                )


            # --------------------------------------------------------------------------------------
            # 8.13 Verify Provenance Exclusion
            # --------------------------------------------------------------------------------------

            provenance = (
                TRAINING_PROVENANCE_COLUMNS[
                    dataset_id
                ]
            )


            if provenance in reloaded_df.columns:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "persisted provenance leakage detected."
                )


            # --------------------------------------------------------------------------------------
            # 8.14 Verify Identifier Exclusion
            # --------------------------------------------------------------------------------------

            identifiers = list(
                TRAINING_IDENTIFIER_COLUMNS[
                    dataset_id
                ]
            )


            leaked_identifiers = (
                set(identifiers)
                .intersection(
                    reloaded_df.columns
                )
            )


            if leaked_identifiers:

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "persisted identifier leakage detected: "
                    f"{sorted(leaked_identifiers)}"
                )


            # --------------------------------------------------------------------------------------
            # 8.15 Verify Research-Control Flags
            # --------------------------------------------------------------------------------------

            if str(
                record[
                    "fit_split"
                ]
            ) != "train":

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "fit_split is not 'train'."
                )


            if bool(
                record[
                    "validation_used_for_fit"
                ]
            ):

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "validation data marked as used for fitting."
                )


            if bool(
                record[
                    "test_used_for_fit"
                ]
            ):

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "test data marked as used for fitting."
                )


            if bool(
                record[
                    "target_used_as_predictor"
                ]
            ):

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "target leakage flag is active."
                )


            if bool(
                record[
                    "identifier_used"
                ]
            ):

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "identifier usage flag is active."
                )


            if bool(
                record[
                    "provenance_used"
                ]
            ):

                raise RuntimeError(
                    f"{dataset_id}/{baseline_name}/"
                    f"repetition {repetition}: "
                    "provenance usage flag is active."
                )


            # --------------------------------------------------------------------------------------
            # 8.16 Record Verification
            # --------------------------------------------------------------------------------------

            ARTIFACT_VERIFICATION_RECORDS.append(
                {
                    "repetition": repetition,
                    "repetition_seed": repetition_seed,
                    "experiment_seed": experiment_seed,
                    "dataset_id": dataset_id,
                    "baseline": baseline_name,
                    "synthetic_data_verified": True,
                    "model_verified": True,
                    "metadata_verified": True,
                    "sha256_verified": True,
                    "schema_verified": True,
                    "sample_size_verified": True,
                    "target_verified": True,
                    "identifier_exclusion_verified": True,
                    "provenance_exclusion_verified": True,
                    "status": "PASS",
                }
            )


            print(
                f"✓ {dataset_id:<20} | "
                f"{baseline_name:<24} | "
                f"repetition={repetition} | "
                f"seed={experiment_seed} | "
                f"CSV PASS | MODEL PASS | METADATA PASS"
            )


            # --------------------------------------------------------------------------------------
            # 8.17 RAM Cleanup
            # --------------------------------------------------------------------------------------

            del reloaded_df

            gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. Create Verification DataFrame
# --------------------------------------------------------------------------------------------------

ARTIFACT_VERIFICATION_DF = pd.DataFrame(
    ARTIFACT_VERIFICATION_RECORDS
)


# --------------------------------------------------------------------------------------------------
# 10. Verify Record Count
# --------------------------------------------------------------------------------------------------

if len(
    ARTIFACT_VERIFICATION_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Artifact verification record count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {len(ARTIFACT_VERIFICATION_DF)}"
    )


# --------------------------------------------------------------------------------------------------
# 11. Verify Experimental-Unit Uniqueness
# --------------------------------------------------------------------------------------------------

VERIFICATION_KEYS = [
    "repetition",
    "dataset_id",
    "baseline",
    "experiment_seed",
]


if ARTIFACT_VERIFICATION_DF.duplicated(
    subset=VERIFICATION_KEYS
).any():

    raise RuntimeError(
        "Duplicate artifact verification records detected."
    )


# --------------------------------------------------------------------------------------------------
# 12. Verify Experimental-Unit Coverage
# --------------------------------------------------------------------------------------------------

actual_verification_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in ARTIFACT_VERIFICATION_DF.iterrows()
}


if actual_verification_units != EXPECTED_EXPERIMENTAL_UNITS:

    missing_units = sorted(
        EXPECTED_EXPERIMENTAL_UNITS
        - actual_verification_units
    )

    unexpected_units = sorted(
        actual_verification_units
        - EXPECTED_EXPERIMENTAL_UNITS
    )

    raise RuntimeError(
        "Artifact verification coverage mismatch.\n"
        f"Missing    : {missing_units}\n"
        f"Unexpected : {unexpected_units}"
    )


# --------------------------------------------------------------------------------------------------
# 13. Verify All Artifact Checks Passed
# --------------------------------------------------------------------------------------------------

VERIFICATION_COLUMNS = [
    "synthetic_data_verified",
    "model_verified",
    "metadata_verified",
    "sha256_verified",
    "schema_verified",
    "sample_size_verified",
    "target_verified",
    "identifier_exclusion_verified",
    "provenance_exclusion_verified",
]


for column in VERIFICATION_COLUMNS:

    if not ARTIFACT_VERIFICATION_DF[
        column
    ].astype(bool).all():

        raise RuntimeError(
            f"Artifact verification failed for column '{column}'."
        )


if not ARTIFACT_VERIFICATION_DF[
    "status"
].astype(str).eq("PASS").all():

    raise RuntimeError(
        "One or more artifact verification records "
        "do not have PASS status."
    )


# --------------------------------------------------------------------------------------------------
# 14. Save Artifact Verification Registry
# --------------------------------------------------------------------------------------------------

NB04_VALIDATION_ROOT = (
    NB04_ROOT
    / "validation"
)

NB04_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


ARTIFACT_VERIFICATION_REGISTRY_PATH = (
    NB04_VALIDATION_ROOT
    / "artifact_verification_registry.csv"
)


ARTIFACT_VERIFICATION_DF = (
    ARTIFACT_VERIFICATION_DF
    .sort_values(
        by=[
            "repetition",
            "dataset_id",
            "baseline",
        ]
    )
    .reset_index(
        drop=True
    )
)


ARTIFACT_VERIFICATION_DF.to_csv(
    ARTIFACT_VERIFICATION_REGISTRY_PATH,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 15. Create Artifact Validation Report
# --------------------------------------------------------------------------------------------------

ARTIFACT_VALIDATION_REPORT = {
    "notebook_id": (
        NOTEBOOK_ID
        if "NOTEBOOK_ID" in globals()
        else "04"
    ),

    "notebook_version": (
        NOTEBOOK_VERSION
        if "NOTEBOOK_VERSION" in globals()
        else "1.0"
    ),

    "verification_type": (
        "30_run_artifact_integrity_verification"
    ),

    "expected_runs": EXPECTED_RUNS,

    "verified_runs": len(
        ARTIFACT_VERIFICATION_DF
    ),

    "datasets": list(
        DATASET_IDS
    ),

    "baselines": list(
        BASELINE_METHODS
    ),

    "repetitions": int(
        REPETITIONS
    ),

    "all_synthetic_data_verified": True,

    "all_models_verified": True,

    "all_metadata_verified": True,

    "all_sha256_verified": True,

    "schema_verification": True,

    "sample_size_verification": True,

    "target_verification": True,

    "identifier_exclusion_verification": True,

    "provenance_exclusion_verification": True,

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


ARTIFACT_VALIDATION_PATH = (
    NB04_VALIDATION_ROOT
    / "artifact_validation.json"
)


with ARTIFACT_VALIDATION_PATH.open(
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        ARTIFACT_VALIDATION_REPORT,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 16. Reload Validation Report
# --------------------------------------------------------------------------------------------------

if not ARTIFACT_VALIDATION_PATH.exists():

    raise RuntimeError(
        "Artifact validation report was not created."
    )


with ARTIFACT_VALIDATION_PATH.open(
    "r",
    encoding="utf-8",
) as handle:

    reloaded_validation_report = json.load(
        handle
    )


if int(
    reloaded_validation_report[
        "verified_runs"
    ]
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Persisted artifact validation report "
        "contains an incorrect verified-run count."
    )


# --------------------------------------------------------------------------------------------------
# 17. Final Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 16 — ARTIFACT VERIFICATION SUMMARY")
print("=" * 100)

print(
    f"✓ Datasets                         : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines                        : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                      : "
    f"{REPETITIONS}"
)

print(
    f"✓ Expected artifact runs           : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Verified artifact runs           : "
    f"{len(ARTIFACT_VERIFICATION_DF)}"
)

print(
    "✓ Synthetic data verification      : PASS"
)

print(
    "✓ Model verification               : PASS"
)

print(
    "✓ Metadata verification            : PASS"
)

print(
    "✓ SHA-256 verification             : PASS"
)

print(
    "✓ Exact schema verification        : PASS"
)

print(
    "✓ Sample-size verification         : PASS"
)

print(
    "✓ Target verification              : PASS"
)

print(
    "✓ Identifier exclusion             : PASS"
)

print(
    "✓ Provenance exclusion             : PASS"
)

print(
    "✓ Repetition coverage              : PASS"
)

print(
    "✓ Experiment-unit uniqueness       : PASS"
)

print(
    "✓ 30-run experimental coverage     : PASS"
)

print(
    "✓ Artifact verification registry   : "
    f"{ARTIFACT_VERIFICATION_REGISTRY_PATH}"
)

print(
    "✓ Artifact validation report       : "
    f"{ARTIFACT_VALIDATION_PATH}"
)

print()
print(
    "✓ SECTION 16 — ARTIFACT VERIFICATION: PASS"
)


# --------------------------------------------------------------------------------------------------
# 18. Final RAM Cleanup
# --------------------------------------------------------------------------------------------------

del reloaded_validation_report

gc.collect()

print(
    "✓ Final RAM cleanup completed."
)

SECTION 16 — VERIFY ARTIFACTS

✓ Datasets                 : 3
✓ Baselines                : 2
✓ Repetitions              : 5
✓ Expected artifact runs   : 30
✓ Generation manifest records : 30
✓ 30-run experimental coverage : PASS
✓ Manifest experimental-unit uniqueness : PASS

REPETITION 1 / 5
✓ adult_income         | independent_marginal     | repetition=1 | seed=3136 | CSV PASS | MODEL PASS | METADATA PASS
✓ adult_income         | gaussian_copula          | repetition=1 | seed=3146 | CSV PASS | MODEL PASS | METADATA PASS
✓ bank_marketing       | independent_marginal     | repetition=1 | seed=3236 | CSV PASS | MODEL PASS | METADATA PASS
✓ bank_marketing       | gaussian_copula          | repetition=1 | seed=3246 | CSV PASS | MODEL PASS | METADATA PASS
✓ diabetes_130us       | independent_marginal     | repetition=1 | seed=3336 | CSV PASS | MODEL PASS | METADATA PASS
✓ diabetes_130us       | gaussian_copula          | repetition=1 | seed=3346 | CSV PASS | MODEL PASS | METADATA PASS

REP

In [100]:
# ==================================================================================================
# SECTION 17 — COMPLETION SUMMARY
# FINAL NOTEBOOK 04 COMPLETION AND INTEGRITY GATE
# ==================================================================================================

print("=" * 100)
print("SECTION 17 — COMPLETION SUMMARY")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Required Runtime Objects
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "NOTEBOOK_VERSION",
    "PROJECT_ROOT",
    "CANONICAL_NB02_ROOT",
    "NB04_ROOT",
    "DATASET_IDS",
    "BASELINE_METHODS",
    "REPETITIONS",
    "TRAINING_DATA",
    "BASELINE_DEFINITIONS",
    "SYNTHETIC_ARTIFACT_DF",
    "GENERATION_MANIFEST_DF",
    "BASELINE_METADATA_DF",
    "ARTIFACT_VERIFICATION_DF",
    "NB04_SYNTHETIC_ROOT",
    "NB04_MODEL_ROOT",
    "NB04_METADATA_ROOT",
    "NB04_MANIFEST_ROOT",
    "NB04_VALIDATION_ROOT",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Section 17 is missing required runtime objects: "
        f"{missing_runtime_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Authoritative Experimental Counts
# --------------------------------------------------------------------------------------------------

EXPECTED_RUNS = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
    * int(REPETITIONS)
)

EXPECTED_BASELINE_METADATA = (
    len(DATASET_IDS)
    * len(BASELINE_METHODS)
)


if EXPECTED_RUNS != 30:
    raise RuntimeError(
        "Expected exactly 30 experimental runs.\n"
        f"Found: {EXPECTED_RUNS}"
    )


if EXPECTED_BASELINE_METADATA != 6:
    raise RuntimeError(
        "Expected exactly 6 canonical baseline metadata records.\n"
        f"Found: {EXPECTED_BASELINE_METADATA}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Notebook Summary
# --------------------------------------------------------------------------------------------------

print()
print("NOTEBOOK 04 — STATISTICAL BASELINES")
print("=" * 100)

print(
    f"Notebook version       : {NOTEBOOK_VERSION}"
)

print(
    f"Project root           : {PROJECT_ROOT}"
)

print(
    f"Notebook 02 input      : {CANONICAL_NB02_ROOT}"
)

print(
    f"Notebook 04 output     : {NB04_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 4. Dataset Summary
# --------------------------------------------------------------------------------------------------

print()
print("DATASETS")
print("-" * 100)

for dataset_id in DATASET_IDS:

    if dataset_id not in TRAINING_DATA:
        raise RuntimeError(
            f"{dataset_id}: training data is missing."
        )

    training_rows = int(
        len(
            TRAINING_DATA[
                dataset_id
            ]
        )
    )

    if training_rows <= 0:
        raise RuntimeError(
            f"{dataset_id}: invalid training-row count."
        )

    print(
        f"✓ {dataset_id:<20} | "
        f"training rows={training_rows:,}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Baseline Summary
# --------------------------------------------------------------------------------------------------

print()
print("BASELINES")
print("-" * 100)

for baseline_name in BASELINE_METHODS:

    if baseline_name not in BASELINE_DEFINITIONS:
        raise RuntimeError(
            f"Missing definition for baseline '{baseline_name}'."
        )

    print(
        f"✓ {baseline_name:<24} | "
        f"{BASELINE_DEFINITIONS[baseline_name]['name']}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Experimental Design Summary
# --------------------------------------------------------------------------------------------------

print()
print("EXPERIMENTAL DESIGN")
print("-" * 100)

print(
    f"✓ Datasets                    : "
    f"{len(DATASET_IDS)}"
)

print(
    f"✓ Baselines                   : "
    f"{len(BASELINE_METHODS)}"
)

print(
    f"✓ Repetitions                 : "
    f"{REPETITIONS}"
)

print(
    f"✓ Experimental runs           : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Canonical metadata records  : "
    f"{EXPECTED_BASELINE_METADATA}"
)


# --------------------------------------------------------------------------------------------------
# 7. Experimental Integrity
# --------------------------------------------------------------------------------------------------

print()
print("EXPERIMENTAL INTEGRITY")
print("-" * 100)

print(
    "✓ TRAIN split only used for fitting"
)

print(
    "✓ VALIDATION split excluded from fitting"
)

print(
    "✓ TEST split excluded from fitting"
)

print(
    "✓ Notebook 02 preprocessing not refitted"
)

print(
    "✓ Raw datasets not reloaded"
)

print(
    "✓ Native generative schema used"
)

print(
    "✓ Target retained in synthetic data"
)

print(
    "✓ Target never used as predictor"
)

print(
    "✓ Provenance excluded"
)

print(
    "✓ Identifiers excluded"
)

print(
    "✓ Synthetic size equals training size"
)

print(
    "✓ Reproducible seed policy applied"
)

print(
    "✓ Five-repetition experimental design applied"
)


# --------------------------------------------------------------------------------------------------
# 8. Artifact Count Validation
# --------------------------------------------------------------------------------------------------

synthetic_artifact_count = int(
    len(
        SYNTHETIC_ARTIFACT_DF
    )
)

model_artifact_count = int(
    len(
        GENERATION_MANIFEST_DF
    )
)

metadata_artifact_count = int(
    len(
        BASELINE_METADATA_DF
    )
)

manifest_record_count = int(
    len(
        GENERATION_MANIFEST_DF
    )
)

verified_run_count = int(
    len(
        ARTIFACT_VERIFICATION_DF
    )
)


# Synthetic artifacts = 30

if synthetic_artifact_count != EXPECTED_RUNS:

    raise RuntimeError(
        "Synthetic artifact count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {synthetic_artifact_count}"
    )


# Model artifacts = 30

if model_artifact_count != EXPECTED_RUNS:

    raise RuntimeError(
        "Model artifact count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {model_artifact_count}"
    )


# Metadata artifacts = 6

if metadata_artifact_count != EXPECTED_BASELINE_METADATA:

    raise RuntimeError(
        "Baseline metadata count mismatch.\n"
        f"Expected: {EXPECTED_BASELINE_METADATA}\n"
        f"Found   : {metadata_artifact_count}"
    )


# Manifest records = 30

if manifest_record_count != EXPECTED_RUNS:

    raise RuntimeError(
        "Generation manifest count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {manifest_record_count}"
    )


# Verification records = 30

if verified_run_count != EXPECTED_RUNS:

    raise RuntimeError(
        "Artifact verification count mismatch.\n"
        f"Expected: {EXPECTED_RUNS}\n"
        f"Found   : {verified_run_count}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Verify Artifact Verification Status
# --------------------------------------------------------------------------------------------------

if not (
    ARTIFACT_VERIFICATION_DF[
        "status"
    ]
    .astype(str)
    .eq("PASS")
).all():

    raise RuntimeError(
        "One or more artifact verification records are not PASS."
    )


# --------------------------------------------------------------------------------------------------
# 10. Verify 30-Run Experimental Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_EXPERIMENTAL_UNITS = {
    (
        repetition,
        dataset_id,
        baseline_name,
    )
    for repetition in range(
        1,
        int(REPETITIONS) + 1,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


# Generation manifest coverage

manifest_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in GENERATION_MANIFEST_DF.iterrows()
}


if manifest_units != EXPECTED_EXPERIMENTAL_UNITS:

    raise RuntimeError(
        "Generation manifest experimental coverage mismatch."
    )


# Artifact verification coverage

verification_units = {
    (
        int(row["repetition"]),
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in ARTIFACT_VERIFICATION_DF.iterrows()
}


if verification_units != EXPECTED_EXPERIMENTAL_UNITS:

    raise RuntimeError(
        "Artifact verification experimental coverage mismatch."
    )


print()
print(
    "✓ 30-run generation coverage     : PASS"
)

print(
    "✓ 30-run verification coverage   : PASS"
)


# --------------------------------------------------------------------------------------------------
# 11. Verify Canonical Metadata Coverage
# --------------------------------------------------------------------------------------------------

EXPECTED_METADATA_UNITS = {
    (
        dataset_id,
        baseline_name,
    )
    for dataset_id in DATASET_IDS
    for baseline_name in BASELINE_METHODS
}


metadata_units = {
    (
        row["dataset_id"],
        row["baseline"],
    )
    for _, row in BASELINE_METADATA_DF.iterrows()
}


if metadata_units != EXPECTED_METADATA_UNITS:

    raise RuntimeError(
        "Canonical baseline metadata coverage mismatch."
    )


print(
    "✓ Six canonical metadata units    : PASS"
)


# --------------------------------------------------------------------------------------------------
# 12. Output Locations
# --------------------------------------------------------------------------------------------------

print()
print("OUTPUT LOCATIONS")
print("-" * 100)

print(
    f"Synthetic : {NB04_SYNTHETIC_ROOT}"
)

print(
    f"Models    : {NB04_MODEL_ROOT}"
)

print(
    f"Metadata  : {NB04_METADATA_ROOT}"
)

print(
    f"Manifests : {NB04_MANIFEST_ROOT}"
)

print(
    f"Validation: {NB04_VALIDATION_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 13. Final Completion Summary
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)

print(
    f"✓ Expected experimental runs : "
    f"{EXPECTED_RUNS}"
)

print(
    f"✓ Synthetic artifacts        : "
    f"{synthetic_artifact_count}"
)

print(
    f"✓ Model artifacts            : "
    f"{model_artifact_count}"
)

print(
    f"✓ Metadata artifacts         : "
    f"{metadata_artifact_count}"
)

print(
    f"✓ Manifest records           : "
    f"{manifest_record_count}"
)

print(
    f"✓ Artifact verification      : "
    f"{verified_run_count}/{EXPECTED_RUNS} PASS"
)

print(
    "✓ Experimental coverage      : 30/30 PASS"
)

print(
    "✓ Canonical metadata         : 6/6 PASS"
)


# --------------------------------------------------------------------------------------------------
# 14. Final Completion Gate
# --------------------------------------------------------------------------------------------------

FINAL_COMPLETION_CHECKS = {
    "synthetic_artifacts": (
        synthetic_artifact_count == EXPECTED_RUNS
    ),

    "model_artifacts": (
        model_artifact_count == EXPECTED_RUNS
    ),

    "metadata_artifacts": (
        metadata_artifact_count
        == EXPECTED_BASELINE_METADATA
    ),

    "manifest_records": (
        manifest_record_count == EXPECTED_RUNS
    ),

    "verified_runs": (
        verified_run_count == EXPECTED_RUNS
    ),

    "verification_status": (
        ARTIFACT_VERIFICATION_DF[
            "status"
        ]
        .astype(str)
        .eq("PASS")
        .all()
    ),

    "generation_coverage": (
        manifest_units
        == EXPECTED_EXPERIMENTAL_UNITS
    ),

    "verification_coverage": (
        verification_units
        == EXPECTED_EXPERIMENTAL_UNITS
    ),

    "metadata_coverage": (
        metadata_units
        == EXPECTED_METADATA_UNITS
    ),
}


failed_completion_checks = [
    check
    for check, passed
    in FINAL_COMPLETION_CHECKS.items()
    if not passed
]


if failed_completion_checks:

    raise RuntimeError(
        "Notebook 04 completion gate failed.\n"
        f"Failed checks: {failed_completion_checks}"
    )


# --------------------------------------------------------------------------------------------------
# 15. Final Status
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print(
    "✓ NOTEBOOK 04 — STATISTICAL BASELINES : PASS"
)
print("=" * 100)

print()
print(
    "✓ 3 datasets × 2 baselines × 5 repetitions = 30 runs"
)

print(
    "✓ 30 synthetic artifacts verified"
)

print(
    "✓ 30 model artifacts verified"
)

print(
    "✓ 6 canonical metadata artifacts verified"
)

print(
    "✓ 30 generation-manifest records verified"
)

print(
    "✓ 30 artifact-verification records verified"
)

print(
    "✓ All completion gates passed"
)


# --------------------------------------------------------------------------------------------------
# 16. RAM Cleanup
# --------------------------------------------------------------------------------------------------

gc.collect()

print()
print(
    "✓ RAM cleanup completed."
)

SECTION 17 — COMPLETION SUMMARY

NOTEBOOK 04 — STATISTICAL BASELINES
Notebook version       : 2.0
Project root           : /content/drive/MyDrive/SPP_GAN_Research
Notebook 02 input      : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
Notebook 04 output     : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_04

DATASETS
----------------------------------------------------------------------------------------------------
✓ adult_income         | training rows=34,189
✓ bank_marketing       | training rows=31,647
✓ diabetes_130us       | training rows=71,236

BASELINES
----------------------------------------------------------------------------------------------------
✓ independent_marginal     | Independent Marginal Sampling
✓ gaussian_copula          | Gaussian Copula

EXPERIMENTAL DESIGN
----------------------------------------------------------------------------------------------------
✓ Datasets                    : 3
✓ Baselines                   : 